In [5]:
# ============================================================
# IMPORTS STANDARDS
# ============================================================

from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag
from datetime import datetime, timezone

import re
import time
import shutil
import hashlib
import importlib
import subprocess
import sys
import math
import tempfile
import json


# ============================================================
# INSTALLATION AUTOMATIQUE DES PACKAGES EXTERNES
# ============================================================

def install_and_import(package_name, import_name=None):
    """
    Installe un package s'il n'est pas présent, puis l'importe.

    package_name : nom utilisé par pip
    import_name  : nom utilisé dans import Python
    """

    if import_name is None:
        import_name = package_name

    try:
        module = importlib.import_module(import_name)
        print(f"[OK] {package_name} déjà installé")

    except ImportError:
        print(f"[INSTALLATION] {package_name}")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])

        module = importlib.import_module(import_name)

        print(f"[OK] {package_name} installé")

    return module


# ============================================================
# IMPORTS EXTERNES ROBUSTES
# ============================================================

requests = install_and_import("requests")

pd = install_and_import("pandas")

BeautifulSoup = install_and_import(
    "beautifulsoup4",
    "bs4"
).BeautifulSoup

# API / librairie de recherche web DuckDuckGo
DDGS = install_and_import(
    "ddgs",
    "ddgs"
).DDGS

[OK] requests déjà installé
[OK] pandas déjà installé
[OK] beautifulsoup4 déjà installé
[OK] ddgs déjà installé


In [14]:
# ============================================================
# INSTALLATION / CHARGEMENT DES PACKAGES PDF — PHASE 2
# ============================================================

import sys
import subprocess
import importlib


def install_and_import(package_name, import_name=None):
    """
    Installe un package s'il est absent, puis l'importe.
    """
    if import_name is None:
        import_name = package_name

    try:
        module = importlib.import_module(import_name)
        print(f"{package_name} déjà installé.")
        return module

    except ImportError:
        print(f"{package_name} non trouvé. Installation en cours...")

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])

        module = importlib.import_module(import_name)
        print(f"{package_name} installé et chargé.")

        return module


fitz = install_and_import("pymupdf", "fitz")
pdfplumber = install_and_import("pdfplumber", "pdfplumber")

try:
    Image = install_and_import("pillow", "PIL.Image")
except Exception:
    Image = None
    print("Pillow indisponible. OCR image désactivé.")

try:
    pytesseract = install_and_import("pytesseract", "pytesseract")
except Exception:
    pytesseract = None
    print("pytesseract indisponible. OCR image désactivé.")


PYMUPDF_AVAILABLE = fitz is not None
PDFPLUMBER_AVAILABLE = pdfplumber is not None
PIL_AVAILABLE = Image is not None
PYTESSERACT_AVAILABLE = pytesseract is not None

print("Disponibilité des packages PDF :")
print("PyMuPDF :", PYMUPDF_AVAILABLE)
print("pdfplumber :", PDFPLUMBER_AVAILABLE)
print("Pillow :", PIL_AVAILABLE)
print("pytesseract :", PYTESSERACT_AVAILABLE)

pymupdf non trouvé. Installation en cours...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 95.6 MB/s  0:00:00m eta 0:00:01



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


pymupdf installé et chargé.
pdfplumber non trouvé. Installation en cours...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 92.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pdfplumber]3 [pdfminer.six]



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


pdfplumber installé et chargé.
pillow déjà installé.
pytesseract non trouvé. Installation en cours...
pytesseract installé et chargé.
Disponibilité des packages PDF :
PyMuPDF : True
pdfplumber : True
Pillow : True
pytesseract : True



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Pipeline robuste de collecte et d’ingestion documentaire ESG

## Objectif de cette première phase

Cette première étape du projet vise à construire un pipeline automatisé capable de :
- découvrir,
- collecter,
- valider,
- scorer,
- sélectionner,
- puis ingérer des documents ESG et réglementaires à partir de multiples sources documentaires.

L’objectif est de constituer une base documentaire ESG structurée, traçable et exploitable pour les étapes ultérieures :
- extraction d’informations ESG,
- construction d’indicateurs,
- scoring ESG,
- analyse de robustesse,
- et recherche quantitative en investissement durable.

---

## Fonctionnalités principales du pipeline

Le pipeline a été conçu pour être robuste, modulaire et extensible.

Il permet notamment :

### 1. Gestion de plusieurs modes d’entrée
Le système accepte différents types d’entrée :
- noms d’entreprises ;
- URLs PDF fournies manuellement ;
- fichiers PDF locaux uploadés.

---

### 2. Discovery documentaire automatisée
Le pipeline combine :
- exploration directe des sites corporate ;
- et recherche web complémentaire via DuckDuckGo Search (DDGS).

Cette approche améliore significativement la couverture documentaire ESG.

---

### 3. Validation et contrôle qualité
Chaque document détecté est :
- validé techniquement ;
- normalisé ;
- dédupliqué ;
- puis enregistré avec ses métadonnées et logs associés.

---

### 4. Scoring et sélection robuste
Les documents candidats sont évalués automatiquement selon :
- leur cohérence documentaire ;
- la présence de mots-clés ESG ;
- l’année fiscale ;
- le type de rapport attendu ;
- et plusieurs critères de pertinence.

Le pipeline distingue ensuite :
- les documents automatiquement sélectionnés ;
- les documents nécessitant une revue manuelle ;
- et les documents probablement absents.

---

### 5. Architecture exploitable pour les étapes futures
Le pipeline produit des sorties structurées et auditables permettant une intégration future avec :
- des outils d’extraction NLP ;
- des systèmes de scoring ESG ;
- une interface graphique ;
- ou des pipelines de recherche quantitative plus avancés.

## 1. Configuration de l’environnement

Cette cellule, configure les dossiers de travail et définit les paramètres globaux utilisés pour la collecte, le stockage et le suivi des documents ESG. On définit ici l’architecture locale du projet. Les documents téléchargés sont stockés dans `raw`, tandis que les fichiers de suivi, de logs et de métadonnées sont centralisés dans `metadata` et `logs`.

In [38]:
BASE_DIR = Path("esg_data")
RAW_DIR = BASE_DIR / "raw"
LOG_DIR = BASE_DIR / "logs"
META_DIR = BASE_DIR / "metadata"

for directory in [RAW_DIR, LOG_DIR, META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

METADATA_PATH = META_DIR / "documents_metadata.csv"
LOG_PATH = LOG_DIR / "ingestion_log.csv"
DISCOVERY_PATH = META_DIR / "discovery_results.csv"
PDF_INDEX_PATH = META_DIR / "company_pdf_index.csv"
SELECTION_PATH = META_DIR / "selected_documents.csv"


## 2. Paramètres de requête web

Ces paramètres encadrent les téléchargements : identification du script auprès des serveurs, délai maximal d’attente et taille maximale autorisée pour un fichier PDF.

In [39]:
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 ESG Research Bot; educational research use"
}

REQUEST_TIMEOUT = 25
MAX_DOWNLOAD_MB = 80


## 4. Schémas des fichiers de sortie

Cette cellule définit les colonnes attendues pour chaque fichier de suivi. Cela garantit une structure stable entre les différentes exécutions du pipeline.

In [40]:
# ============================================================
# 2. SCHÉMAS
# ============================================================

LOG_COLUMNS = [
    "input_mode", "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "source_url", "local_input_path",
    "status", "error_type", "error_message", "retrieval_date"
]

METADATA_COLUMNS = [
    "input_mode", "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "source_url", "local_path", "sha256",
    "file_size_bytes", "retrieval_date", "status"
]

PDF_INDEX_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "source_page", "candidate_pdf_url", "anchor_text",
    "valid_pdf", "retrieval_date"
]

DISCOVERY_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "expected_query", "source_page", "candidate_pdf_url",
    "anchor_text", "valid_pdf", "score", "confidence_level",
    "estimated_probability", "discovery_status", "selection_reason",
    "retrieval_date"
]

SELECTION_COLUMNS = [
    "company", "ticker", "isin", "jurisdiction",
    "strate", "doc_type", "doc_subtype", "fiscal_year",
    "period_type", "expected_query", "selected_url", "score",
    "confidence_level", "estimated_probability", "selection_status",
    "selection_reason", "retrieval_date"
]



## 5. Référentiel documentaire ESG

Le référentiel documentaire décrit les types de documents recherchés.  
Chaque document est associé à une strate de collecte, une catégorie, une périodicité et une requête-type utilisée pour la recherche automatique.

In [41]:
# ============================================================
# 3. RÉFÉRENTIEL DOCUMENTAIRE
# ============================================================

DOCUMENT_REGISTRY = [
    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "annual_report_urd", "period_type": "FY",
     "query_template": "{company} annual report universal registration document {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "sustainability_statement_csrd_esrs", "period_type": "FY",
     "query_template": "{company} sustainability statement ESRS CSRD {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "climate_report_tcfd_transition_plan", "period_type": "FY",
     "query_template": "{company} climate report TCFD transition plan {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "vigilance_plan", "period_type": "FY",
     "query_template": "{company} plan de vigilance {year} PDF"},

    {"strate": 1, "doc_type": "regulatory", "doc_subtype": "half_year_financial_report", "period_type": "H1",
     "query_template": "{company} half year financial report {year} PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "code_of_conduct", "period_type": "perpetual",
     "query_template": "{company} code of conduct PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "anti_corruption_policy", "period_type": "perpetual",
     "query_template": "{company} anti corruption policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "human_rights_policy", "period_type": "perpetual",
     "query_template": "{company} human rights policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "dei_policy", "period_type": "perpetual",
     "query_template": "{company} diversity equity inclusion policy PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "environmental_policy", "period_type": "perpetual",
     "query_template": "{company} environmental policy climate biodiversity water PDF"},

    {"strate": 2, "doc_type": "policy", "doc_subtype": "supplier_code_of_conduct", "period_type": "perpetual",
     "query_template": "{company} supplier code of conduct responsible purchasing PDF"},

    {"strate": 3, "doc_type": "corporate_communication", "doc_subtype": "investor_presentation", "period_type": "adhoc",
     "query_template": "{company} investor presentation ESG capital markets day {year} PDF"},

    {"strate": 3, "doc_type": "corporate_communication", "doc_subtype": "agm_minutes_resolutions", "period_type": "adhoc",
     "query_template": "{company} annual general meeting resolutions {year} PDF"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "cdp_response", "period_type": "FY",
     "query_template": "{company} CDP climate change response {year} PDF"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "sbti_validation", "period_type": "adhoc",
     "query_template": "{company} SBTi validated targets"},

    {"strate": 4, "doc_type": "external_source", "doc_subtype": "assurance_report", "period_type": "FY",
     "query_template": "{company} independent assurance report sustainability {year} PDF"}
]



## 6. Référentiel des entreprises

Cette cellule contient un référentiel initial d’émetteurs utilisé pour accélérer et fiabiliser la phase de discovery documentaire.

Pour les entreprises présentes dans ce référentiel, le pipeline dispose directement d’informations structurées telles que :
- le nom de l’entreprise,
- le ticker,
- l’ISIN,
- la juridiction,
- ainsi que les domaines web officiels à explorer.

Cela permet :
- de limiter les recherches à des sources corporate fiables,
- de réduire le bruit documentaire,
- d’améliorer la précision du scoring des documents candidats,
- et d’accélérer la collecte.

Cependant, le pipeline reste volontairement générique et extensible.

Lorsqu’une entreprise n’est pas présente dans le référentiel, une procédure de normalisation et d’inférence est appliquée automatiquement :
- génération heuristique de domaines probables à partir du nom de l’entreprise,
- exploration web complémentaire via moteurs de recherche,
- puis scoring des documents détectés selon leur pertinence documentaire.

Le référentiel agit donc comme une couche d’optimisation et non comme une dépendance structurelle du pipeline.

In [42]:
# ============================================================
# 4. RÉFÉRENTIEL ENTREPRISES
# ============================================================

EMITTERS = [
    {"company": "TotalEnergies", "ticker": "TTE", "isin": "FR0000120271", "jurisdiction": "France", "official_domains": ["totalenergies.com"]},
    {"company": "Schneider Electric", "ticker": "SU", "isin": "FR0000121972", "jurisdiction": "France", "official_domains": ["se.com", "schneider-electric.com"]},
    {"company": "LVMH", "ticker": "MC", "isin": "FR0000121014", "jurisdiction": "France", "official_domains": ["lvmh.com"]},
    {"company": "Air Liquide", "ticker": "AI", "isin": "FR0000120073", "jurisdiction": "France", "official_domains": ["airliquide.com"]},
    {"company": "Sanofi", "ticker": "SAN", "isin": "FR0000120578", "jurisdiction": "France", "official_domains": ["sanofi.com"]},
    {"company": "Airbus", "ticker": "AIR", "isin": "NL0000235190", "jurisdiction": "Netherlands", "official_domains": ["airbus.com"]},
    {"company": "Safran", "ticker": "SAF", "isin": "FR0000073272", "jurisdiction": "France", "official_domains": ["safran-group.com"]},
    {"company": "BNP Paribas", "ticker": "BNP", "isin": "FR0000131104", "jurisdiction": "France", "official_domains": ["bnpparibas.com"]},
    {"company": "L'Oreal", "ticker": "OR", "isin": "FR0000120321", "jurisdiction": "France", "official_domains": ["loreal.com"]},
    {"company": "AXA", "ticker": "CS", "isin": "FR0000120628", "jurisdiction": "France", "official_domains": ["axa.com"]},

    {"company": "Vinci", "ticker": "DG", "isin": "FR0000125486", "jurisdiction": "France", "official_domains": ["vinci.com"]},
    {"company": "EssilorLuxottica", "ticker": "EL", "isin": "FR0000121667", "jurisdiction": "France", "official_domains": ["essilorluxottica.com"]},
    {"company": "Hermes International", "ticker": "RMS", "isin": "FR0000052292", "jurisdiction": "France", "official_domains": ["hermes.com", "finance.hermes.com"]},
    {"company": "Engie", "ticker": "ENGI", "isin": "FR0010208488", "jurisdiction": "France", "official_domains": ["engie.com"]},
    {"company": "Danone", "ticker": "BN", "isin": "FR0000120644", "jurisdiction": "France", "official_domains": ["danone.com"]},
    {"company": "Societe Generale", "ticker": "GLE", "isin": "FR0000130809", "jurisdiction": "France", "official_domains": ["societegenerale.com"]},
    {"company": "Legrand", "ticker": "LR", "isin": "FR0010307819", "jurisdiction": "France", "official_domains": ["legrandgroup.com", "legrand.com"]},
    {"company": "Saint-Gobain", "ticker": "SGO", "isin": "FR0000125007", "jurisdiction": "France", "official_domains": ["saint-gobain.com"]},
    {"company": "Orange", "ticker": "ORA", "isin": "FR0000133308", "jurisdiction": "France", "official_domains": ["orange.com"]},
    {"company": "Thales", "ticker": "HO", "isin": "FR0000121329", "jurisdiction": "France", "official_domains": ["thalesgroup.com"]},

    {"company": "Veolia", "ticker": "VIE", "isin": "FR0000124141", "jurisdiction": "France", "official_domains": ["veolia.com"]},
    {"company": "Michelin", "ticker": "ML", "isin": "FR001400AJ45", "jurisdiction": "France", "official_domains": ["michelin.com"]},
    {"company": "Kering", "ticker": "KER", "isin": "FR0000121485", "jurisdiction": "France", "official_domains": ["kering.com"]},
    {"company": "ArcelorMittal", "ticker": "MT", "isin": "LU1598757687", "jurisdiction": "Luxembourg", "official_domains": ["arcelormittal.com"]},
    {"company": "STMicroelectronics", "ticker": "STMPA", "isin": "NL0000226223", "jurisdiction": "Netherlands", "official_domains": ["st.com"]},

    {"company": "Accor", "ticker": "AC", "isin": "FR0000120404", "jurisdiction": "France", "official_domains": ["group.accor.com", "accor.com"]},
    {"company": "Bouygues", "ticker": "EN", "isin": "FR0000120503", "jurisdiction": "France", "official_domains": ["bouygues.com"]},
    {"company": "Bureau Veritas", "ticker": "BVI", "isin": "FR0006174348", "jurisdiction": "France", "official_domains": ["bureauveritas.com"]},
    {"company": "Capgemini", "ticker": "CAP", "isin": "FR0000125338", "jurisdiction": "France", "official_domains": ["capgemini.com"]},
    {"company": "Carrefour", "ticker": "CA", "isin": "FR0000120172", "jurisdiction": "France", "official_domains": ["carrefour.com"]},
    {"company": "Credit Agricole", "ticker": "ACA", "isin": "FR0000045072", "jurisdiction": "France", "official_domains": ["credit-agricole.com"]},
    {"company": "Dassault Systemes", "ticker": "DSY", "isin": "FR0014003TT8", "jurisdiction": "France", "official_domains": ["3ds.com"]},
    {"company": "Edenred", "ticker": "EDEN", "isin": "FR0010908533", "jurisdiction": "France", "official_domains": ["edenred.com"]},
    {"company": "Euronext", "ticker": "ENX", "isin": "NL0006294274", "jurisdiction": "Netherlands", "official_domains": ["euronext.com"]},
    {"company": "Eurofins Scientific", "ticker": "ERF", "isin": "FR0014000MR3", "jurisdiction": "France", "official_domains": ["eurofins.com"]},
    {"company": "Pernod Ricard", "ticker": "RI", "isin": "FR0000120693", "jurisdiction": "France", "official_domains": ["pernod-ricard.com"]},
    {"company": "Publicis Groupe", "ticker": "PUB", "isin": "FR0000130577", "jurisdiction": "France", "official_domains": ["publicisgroupe.com"]},
    {"company": "Renault", "ticker": "RNO", "isin": "FR0000131906", "jurisdiction": "France", "official_domains": ["renaultgroup.com"]},
    {"company": "Stellantis", "ticker": "STLAP", "isin": "NL00150001Q9", "jurisdiction": "Netherlands", "official_domains": ["stellantis.com"]},
    {"company": "Unibail-Rodamco-Westfield", "ticker": "URW", "isin": "FR0013326246", "jurisdiction": "France", "official_domains": ["urw.com"]}
]


## 7. Fonctions utilitaires générales

Ces fonctions standardisent les noms de fichiers, génèrent les timestamps, normalisent les URLs et gèrent l’écriture incrémentale des fichiers CSV.

In [43]:
# ============================================================
# 5. OUTILS GÉNÉRAUX
# ============================================================

def safe_name(text):
    text = str(text).strip()
    text = re.sub(r"[^a-zA-Z0-9_-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def normalize_url(url):
    url = str(url).strip()
    url, _ = urldefrag(url)
    return url


def append_csv(row, path, columns):
    clean_row = {col: row.get(col, None) for col in columns}
    df = pd.DataFrame([clean_row], columns=columns)

    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)


def log_event(row):
    append_csv(row, LOG_PATH, LOG_COLUMNS)


def save_metadata(row):
    append_csv(row, METADATA_PATH, METADATA_COLUMNS)


def save_selection(row):
    append_csv(row, SELECTION_PATH, SELECTION_COLUMNS)


def compute_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(block)
    return sha256.hexdigest()


def short_hash(text, n=10):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]


def already_exists(sha256):
    if not METADATA_PATH.exists():
        return False

    metadata = pd.read_csv(METADATA_PATH)

    if "sha256" not in metadata.columns:
        return False

    return sha256 in metadata["sha256"].astype(str).values


def extract_years_from_text(text):
    years = re.findall(r"\b(20[0-3][0-9])\b", str(text))
    return sorted(set(int(y) for y in years))


def get_years_for_doc(doc_spec, years):
    if doc_spec["period_type"] == "perpetual":
        return ["current"]
    return years


def build_expected_query(doc_spec, company, year):
    template = doc_spec.get("query_template", "")
    return template.format(company=company, year=year)


def tokenize_query(query):
    tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9]{3,}", str(query).lower())
    stopwords = {
        "pdf", "the", "and", "for", "avec", "des", "les", "une",
        "document", "report", "rapport"
    }
    return [t for t in tokens if t not in stopwords]


## 8. Validation, normalisation et ingestion des documents PDF

Cet ensemble de fonctions constitue le cœur opérationnel du pipeline de collecte documentaire.

La première étape consiste à vérifier la validité technique des fichiers téléchargés afin d’éviter l’ingestion de documents corrompus, incomplets ou non conformes au format PDF attendu.

Le pipeline intègre également une couche de normalisation des entreprises permettant :
- d’harmoniser les informations corporate,
- d’associer automatiquement les métadonnées connues aux émetteurs,
- et de gérer les entreprises absentes du référentiel grâce à des mécanismes d’inférence de domaines web.

Une fois les documents identifiés, le système organise automatiquement leur stockage local selon :
- l’entreprise,
- l’année fiscale,
- la strate documentaire,
- et le type de document.

Le téléchargement des PDFs est réalisé de manière robuste avec :
- gestion des erreurs réseau,
- contrôle de taille,
- gestion des timeouts,
- et validation post-téléchargement.

Enfin, chaque document validé est enregistré dans une base de métadonnées centralisée contenant :
- les informations d’origine du document,
- son empreinte SHA-256,
- son emplacement local,
- ainsi que l’historique des événements de collecte.

Cette architecture permet de garantir :
- la traçabilité complète du pipeline,
- l’élimination des doublons,
- la reproductibilité de la collecte,
- et une structuration exploitable pour les étapes ultérieures d’extraction et de scoring ESG.

In [44]:
# ============================================================
# 6. VALIDATION PDF
# ============================================================

def validate_pdf(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        return False, "TYPE_2_FILE_MISSING", "File not found"

    size = file_path.stat().st_size

    if size == 0:
        return False, "TYPE_2_EMPTY_FILE", "File is empty"

    if size < 10_000:
        return False, "TYPE_2_TOO_SMALL", f"Suspicious file size: {size}"

    with open(file_path, "rb") as f:
        header = f.read(1024)

    if not header.startswith(b"%PDF-"):
        return False, "TYPE_2_NOT_PDF", "File is not a valid PDF"

    return True, None, None


# ============================================================
# 7. NORMALISATION DES ENTREPRISES
# ============================================================

def infer_domains_from_company_name(company):
    clean = company.lower()
    clean = clean.replace("&", "and")
    clean = re.sub(r"[^a-z0-9]+", "", clean)

    if not clean:
        return []

    return [f"{clean}.com"]


def normalize_emitters(companies, emitters_reference=EMITTERS, allow_domain_guess=True):
    emitter_map = {item["company"].lower(): item for item in emitters_reference}
    normalized = []

    for company in companies:
        key = str(company).lower().strip()

        if key in emitter_map:
            item = emitter_map[key].copy()
            item["reference_status"] = "FOUND_IN_REFERENCE"
            normalized.append(item)
        else:
            domains = infer_domains_from_company_name(company) if allow_domain_guess else []
            normalized.append({
                "company": company,
                "ticker": None,
                "isin": None,
                "jurisdiction": None,
                "official_domains": domains,
                "reference_status": "NOT_IN_REFERENCE_DOMAIN_GUESSED" if domains else "NOT_IN_REFERENCE_NO_DOMAIN"
            })

    return normalized


# ============================================================
# 8. ENREGISTREMENT COMMUN D'UN PDF
# ============================================================

def register_pdf(
    input_mode,
    file_path,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="unknown_document",
    fiscal_year="unknown",
    period_type=None,
    source_url=None,
    local_input_path=None
):
    file_path = Path(file_path)
    retrieval_date = now_utc()

    valid, error_type, error_message = validate_pdf(file_path)

    if not valid:
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": source_url,
            "local_input_path": local_input_path,
            "status": "FAILED",
            "error_type": error_type,
            "error_message": error_message,
            "retrieval_date": retrieval_date
        })
        return None

    sha256 = compute_sha256(file_path)

    if already_exists(sha256):
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": source_url,
            "local_input_path": local_input_path,
            "status": "DUPLICATE",
            "error_type": "TYPE_3_DUPLICATE",
            "error_message": "Document already exists",
            "retrieval_date": retrieval_date
        })
        return None

    metadata_row = {
        "input_mode": input_mode,
        "company": company,
        "ticker": ticker,
        "isin": isin,
        "jurisdiction": jurisdiction,
        "strate": strate,
        "doc_type": doc_type,
        "doc_subtype": doc_subtype,
        "fiscal_year": fiscal_year,
        "period_type": period_type,
        "source_url": source_url,
        "local_path": str(file_path),
        "sha256": sha256,
        "file_size_bytes": file_path.stat().st_size,
        "retrieval_date": retrieval_date,
        "status": "SUCCESS"
    }

    save_metadata(metadata_row)

    log_event({
        "input_mode": input_mode,
        "company": company,
        "ticker": ticker,
        "isin": isin,
        "jurisdiction": jurisdiction,
        "strate": strate,
        "doc_type": doc_type,
        "doc_subtype": doc_subtype,
        "fiscal_year": fiscal_year,
        "period_type": period_type,
        "source_url": source_url,
        "local_input_path": local_input_path,
        "status": "SUCCESS",
        "error_type": None,
        "error_message": None,
        "retrieval_date": retrieval_date
    })

    return metadata_row


# ============================================================
# 9. TÉLÉCHARGEMENT PDF
# ============================================================

def make_output_path(company, fiscal_year, input_mode, doc_subtype, source_url=None, original_filename=None, strate=None):
    if strate is not None:
        base_path = RAW_DIR / safe_name(company) / str(fiscal_year) / f"strate_{strate}"
    else:
        base_path = RAW_DIR / safe_name(company) / str(fiscal_year) / input_mode

    folder = base_path / safe_name(doc_subtype)
    folder.mkdir(parents=True, exist_ok=True)

    if original_filename:
        stem = safe_name(Path(original_filename).stem)
        unique = short_hash(str(original_filename))
    elif source_url:
        parsed_name = Path(urlparse(source_url).path).name
        stem = safe_name(Path(parsed_name).stem) if parsed_name.lower().endswith(".pdf") else "document"
        unique = short_hash(source_url)
    else:
        stem = "document"
        unique = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

    filename = f"{stem}_{unique}.pdf"
    return folder / filename


def download_pdf(url, output_path, timeout=REQUEST_TIMEOUT, max_mb=MAX_DOWNLOAD_MB):
    url = normalize_url(url)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    max_bytes = max_mb * 1024 * 1024

    try:
        with requests.get(
            url,
            timeout=timeout,
            headers=REQUEST_HEADERS,
            stream=True,
            allow_redirects=True
        ) as response:

            if response.status_code != 200:
                return False, "TYPE_1_HTTP_ERROR", f"Status code: {response.status_code}"

            content_length = response.headers.get("Content-Length")

            if content_length and int(content_length) > max_bytes:
                return False, "TYPE_1_FILE_TOO_LARGE", f"File exceeds {max_mb} MB"

            total = 0

            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
                tmp_path = Path(tmp.name)

                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue

                    total += len(chunk)

                    if total > max_bytes:
                        tmp_path.unlink(missing_ok=True)
                        return False, "TYPE_1_FILE_TOO_LARGE", f"File exceeds {max_mb} MB"

                    tmp.write(chunk)

        shutil.move(str(tmp_path), str(output_path))
        return True, None, None

    except requests.exceptions.Timeout:
        return False, "TYPE_1_TIMEOUT", "Request timeout"

    except requests.exceptions.SSLError:
        return False, "TYPE_1_SSL_ERROR", "SSL error"

    except requests.exceptions.RequestException as e:
        return False, "TYPE_1_REQUEST_ERROR", str(e)

    except Exception as e:
        return False, "TYPE_1_UNKNOWN_DOWNLOAD_ERROR", str(e)


def ingest_from_url(
    url,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="user_url_pdf",
    fiscal_year="unknown",
    period_type=None,
    input_mode="url"
):
    output_path = make_output_path(
        company=company,
        fiscal_year=fiscal_year,
        input_mode=input_mode,
        doc_subtype=doc_subtype,
        source_url=url,
        strate=strate
    )

    success, error_type, error_message = download_pdf(url, output_path)

    if not success:
        log_event({
            "input_mode": input_mode,
            "company": company,
            "ticker": ticker,
            "isin": isin,
            "jurisdiction": jurisdiction,
            "strate": strate,
            "doc_type": doc_type,
            "doc_subtype": doc_subtype,
            "fiscal_year": fiscal_year,
            "period_type": period_type,
            "source_url": url,
            "local_input_path": None,
            "status": "FAILED",
            "error_type": error_type,
            "error_message": error_message,
            "retrieval_date": now_utc()
        })
        return None

    return register_pdf(
        input_mode=input_mode,
        file_path=output_path,
        company=company,
        ticker=ticker,
        isin=isin,
        jurisdiction=jurisdiction,
        strate=strate,
        doc_type=doc_type,
        doc_subtype=doc_subtype,
        fiscal_year=fiscal_year,
        period_type=period_type,
        source_url=url
    )


def ingest_from_local_pdf(
    file_path,
    company="unknown_company",
    ticker=None,
    isin=None,
    jurisdiction=None,
    strate=None,
    doc_type=None,
    doc_subtype="uploaded_pdf",
    fiscal_year="unknown",
    period_type=None
):
    file_path = Path(file_path)

    output_path = make_output_path(
        company=company,
        fiscal_year=fiscal_year,
        input_mode="local_pdf",
        doc_subtype=doc_subtype,
        original_filename=file_path.name,
        strate=strate
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file_path, output_path)

    return register_pdf(
        input_mode="local_pdf",
        file_path=output_path,
        company=company,
        ticker=ticker,
        isin=isin,
        jurisdiction=jurisdiction,
        strate=strate,
        doc_type=doc_type,
        doc_subtype=doc_subtype,
        fiscal_year=fiscal_year,
        period_type=period_type,
        source_url=None,
        local_input_path=str(file_path)
    )


## 9. Recherche web via DuckDuckGo (DDGS)

Cette fonction implémente une phase de discovery documentaire complémentaire à l’exploration directe des sites corporate.

L’objectif est d’améliorer la couverture documentaire du pipeline lorsqu’un document :
- n’est pas facilement accessible depuis les pages investisseurs ou ESG officielles,
- possède une architecture web difficile à crawler,
- ou est référencé sur des sources externes spécialisées.

À partir du type de document recherché (`doc_spec`), une requête documentaire standardisée est construite automatiquement en intégrant :
- le nom de l’entreprise,
- l’année fiscale,
- ainsi que des mots-clés métiers associés au document attendu.

Une recherche web est ensuite effectuée via DuckDuckGo Search (DDGS).

Le pipeline conserve prioritairement :
- les URLs PDF explicites,
- les pages fortement documentaires,
- ainsi que les résultats contenant des termes compatibles avec le reporting ESG ou financier.

Les résultats retournés sont normalisés dans un format homogène avec le reste du pipeline afin d’être intégrés directement dans les étapes suivantes :
- validation PDF,
- scoring documentaire,
- sélection robuste,
- puis ingestion finale.

Cette approche permet de rendre la collecte plus robuste et moins dépendante de l’architecture spécifique des sites corporate.

In [45]:
def search_pdf_candidates_with_ddgs(company, doc_spec, year, max_results=8, sleep_seconds=1.5):
    """
    Recherche web complémentaire via DDGS.
    Retourne des URLs candidates au même format que l'index corporate.
    """
    query = doc_spec["query_template"].format(company=company, year=year)

    # On force la recherche PDF sans dépendre uniquement de DuckDuckGo
    if "pdf" not in query.lower():
        query = query + " PDF"

    results = []

    try:
        with DDGS() as ddgs:
            search_results = ddgs.text(
                query,
                max_results=max_results
            )

            for item in search_results:
                href = item.get("href")
                title = item.get("title", "")
                body = item.get("body", "")

                if not href:
                    continue

                combined = f"{href} {title} {body}".lower()

                # On garde prioritairement les PDFs ou pages très documentaires
                if ".pdf" in combined or "annual report" in combined or "sustainability" in combined:
                    results.append({
                        "source_page": "DDGS_SEARCH",
                        "candidate_pdf_url": normalize_url(href),
                        "anchor_text": f"{title} {body}".strip()
                    })

        time.sleep(sleep_seconds)

    except Exception as e:
        print(f"[DDGS] Search failed for {company} / {doc_spec['doc_subtype']} / {year}: {e}")

    return results


## 10. Discovery documentaire et construction d’un index PDF intermédiaire

Cette étape constitue le cœur du processus de discovery documentaire du pipeline ESG.

L’objectif est d’identifier automatiquement un ensemble de documents candidats potentiellement pertinents pour l’analyse ESG et financière des entreprises étudiées, avant toute phase de scoring ou d’ingestion.

Le pipeline repose sur une approche hybride combinant deux sources complémentaires de discovery :

### 1. Exploration directe des sites corporate

Une première phase consiste à explorer automatiquement les domaines officiels des entreprises.

À partir des sites corporate connus, le pipeline génère un ensemble de pages candidates typiquement utilisées pour la publication des documents réglementaires et ESG :
- espaces investisseurs,
- publications financières,
- pages ESG et sustainability,
- rapports réglementés,
- documents corporate,
- rapports annuels,
- pages CSR / responsabilité.

Chaque page est ensuite analysée afin d’extraire les liens pointant vers des fichiers PDF potentiels.

Cette approche permet de privilégier :
- des sources corporate fiables,
- une meilleure qualité documentaire,
- et une réduction du bruit informationnel.

---

### 2. Recherche web complémentaire via DuckDuckGo Search (DDGS)

L’exploration directe des sites corporate peut néanmoins être insuffisante dans certains cas :
- documents difficilement accessibles,
- architecture web complexe,
- PDFs non reliés depuis les pages principales,
- sous-domaines non explorés,
- ou documents référencés sur des sources externes spécialisées.

Pour améliorer la couverture documentaire, le pipeline intègre donc une seconde phase de discovery via DuckDuckGo Search (DDGS).

Des requêtes documentaires standardisées sont générées automatiquement à partir :
- du nom de l’entreprise,
- du type de document recherché,
- de l’année fiscale,
- et de mots-clés ESG ou réglementaires.

Les résultats retournés sont ensuite filtrés afin de conserver prioritairement :
- les URLs PDF explicites,
- les pages fortement documentaires,
- ainsi que les contenus compatibles avec le reporting ESG ou financier.

Cette approche permet d’augmenter significativement la robustesse du pipeline et de réduire sa dépendance à la structure spécifique des sites corporate.

---

### 3. Validation et normalisation des PDFs détectés

Les URLs collectées sont ensuite :
- normalisées,
- dédupliquées,
- puis validées techniquement.

Le pipeline vérifie notamment :
- la disponibilité des URLs,
- la cohérence du type MIME,
- ainsi que la signature PDF du fichier.

Un mécanisme de cache est utilisé afin d’éviter des validations réseau répétées sur les mêmes URLs et d’améliorer les performances globales de la collecte.

---

### 4. Construction d’un index PDF intermédiaire

Tous les documents détectés sont centralisés dans un index documentaire intermédiaire (`pdf_index`).

Cet index joue un rôle central dans l’architecture du pipeline :
- centralisation des candidats documentaires,
- suppression des doublons,
- traçabilité des sources,
- séparation entre discovery et scoring,
- réduction des appels réseau inutiles,
- et possibilité de rejouer les étapes de scoring sans refaire la collecte web.

Chaque entrée de l’index contient notamment :
- l’entreprise associée,
- l’URL candidate,
- la page source,
- le texte d’ancrage,
- le statut de validation PDF,
- ainsi que les métadonnées temporelles de collecte.

Cet index constitue ensuite la base de travail pour les étapes suivantes :
- scoring documentaire,
- sélection robuste des meilleurs documents,
- téléchargement,
- puis ingestion finale dans la base documentaire ESG.

In [46]:
# ============================================================
# 10. DISCOVERY
# ============================================================

def generate_candidate_pages(official_domains):
    pages = []

    paths = [
        "",
        "/investors",
        "/investors/publications",
        "/investors/regulated-information",
        "/finance",
        "/finance/publications",
        "/sustainability",
        "/sustainability/publications",
        "/group/sustainability",
        "/publications",
        "/documents",
        "/reports",
        "/responsibility",
        "/csr",
        "/esg",
        "/en",
        "/en/investors",
        "/en/sustainability",
        "/fr",
        "/fr/investors",
        "/fr/developpement-durable",
    ]

    for domain in official_domains:
        if not domain:
            continue

        domain = str(domain).strip()

        bases = []
        if domain.startswith("http"):
            bases.append(domain.rstrip("/"))
        else:
            bases.append(f"https://www.{domain}".rstrip("/"))
            bases.append(f"https://{domain}".rstrip("/"))

        for base in bases:
            for path in paths:
                pages.append(f"{base}{path}")

    return list(dict.fromkeys(pages))


def extract_pdf_links_from_page(page_url, timeout=12):
    try:
        response = requests.get(
            page_url,
            timeout=timeout,
            headers=REQUEST_HEADERS,
            allow_redirects=True
        )

        if response.status_code != 200:
            return []

        soup = BeautifulSoup(response.text, "html.parser")
        links = []

        for tag in soup.find_all("a", href=True):
            href = tag["href"]
            anchor_text = tag.get_text(" ", strip=True)

            combined = f"{href} {anchor_text}".lower()

            if ".pdf" in combined:
                candidate_url = normalize_url(urljoin(page_url, href))

                links.append({
                    "source_page": page_url,
                    "candidate_pdf_url": candidate_url,
                    "anchor_text": anchor_text
                })

        return links

    except requests.exceptions.RequestException:
        return []


PDF_VALIDATION_CACHE = {}


def is_valid_pdf_url(url, timeout=10):
    url = normalize_url(url)

    if url in PDF_VALIDATION_CACHE:
        return PDF_VALIDATION_CACHE[url]

    try:
        with requests.get(
            url,
            timeout=timeout,
            stream=True,
            headers=REQUEST_HEADERS,
            allow_redirects=True
        ) as response:

            if response.status_code != 200:
                PDF_VALIDATION_CACHE[url] = False
                return False

            content_type = response.headers.get("Content-Type", "").lower()

            if "pdf" in content_type:
                PDF_VALIDATION_CACHE[url] = True
                return True

            first_bytes = next(response.iter_content(chunk_size=5), b"")
            is_pdf = first_bytes == b"%PDF-"

            PDF_VALIDATION_CACHE[url] = is_pdf
            return is_pdf

    except requests.exceptions.RequestException:
        PDF_VALIDATION_CACHE[url] = False
        return False


def build_company_pdf_index(company, emitter, years=None, document_registry=None, use_ddgs=True):
    pages = generate_candidate_pages(emitter.get("official_domains", []))
    raw_links = []

    # 1. Discovery corporate classique
    for page_url in pages:
        raw_links.extend(extract_pdf_links_from_page(page_url))
        time.sleep(0.1)

    # 2. Discovery complémentaire via DDGS
    if use_ddgs and years is not None and document_registry is not None:
        for doc_spec in document_registry:
            for year in get_years_for_doc(doc_spec, years):
                ddgs_links = search_pdf_candidates_with_ddgs(
                    company=company,
                    doc_spec=doc_spec,
                    year=year,
                    max_results=5,
                    sleep_seconds=1.5
                )
                raw_links.extend(ddgs_links)

    if not raw_links:
        return pd.DataFrame(columns=PDF_INDEX_COLUMNS)

    pdf_index = pd.DataFrame(raw_links)
    pdf_index = pdf_index.drop_duplicates(subset=["candidate_pdf_url"]).reset_index(drop=True)

    pdf_index["company"] = company
    pdf_index["ticker"] = emitter.get("ticker")
    pdf_index["isin"] = emitter.get("isin")
    pdf_index["jurisdiction"] = emitter.get("jurisdiction")
    pdf_index["valid_pdf"] = pdf_index["candidate_pdf_url"].apply(is_valid_pdf_url)
    pdf_index["retrieval_date"] = now_utc()

    pdf_index = pdf_index[PDF_INDEX_COLUMNS]

    pdf_index.to_csv(
        PDF_INDEX_PATH,
        mode="a" if PDF_INDEX_PATH.exists() else "w",
        header=not PDF_INDEX_PATH.exists(),
        index=False
    )

    return pdf_index



## 11. Scoring documentaire et évaluation de pertinence des PDFs

Après la phase de discovery, le pipeline dispose généralement d’un grand nombre de documents candidats potentiels.  
Cette étape a pour objectif d’évaluer automatiquement la pertinence de chaque document détecté afin d’identifier les PDFs les plus cohérents avec le document ESG réellement recherché.

Le système de scoring repose sur une logique heuristique multicritère combinant :
- correspondance avec le nom de l’entreprise,
- cohérence de l’année fiscale,
- présence de mots-clés documentaires positifs,
- détection de mots-clés négatifs,
- proximité avec la requête documentaire attendue,
- ainsi que certains indices structurels comme l’extension PDF.

---

### 1. Référentiel de mots-clés documentaires

Chaque type de document ESG ou réglementaire possède son propre ensemble de mots-clés métiers.

Par exemple :
- rapports annuels,
- sustainability statements,
- rapports TCFD,
- plans de vigilance,
- politiques ESG,
- réponses CDP,
- validations SBTi,
- rapports d’assurance.

Ces référentiels permettent d’orienter le scoring vers les documents les plus susceptibles de correspondre à la catégorie recherchée.

Le pipeline intègre également des mots-clés négatifs afin de pénaliser certains faux positifs fréquents :
- communiqués de presse,
- droits de vote,
- présentations investisseurs,
- rapports semestriels,
- documents non pertinents.

---

### 2. Scoring heuristique des documents candidats

Chaque document candidat est transformé en une représentation textuelle combinant :
- l’URL du document,
- le texte d’ancrage,
- ainsi que certains éléments contextuels.

Le pipeline attribue ensuite un score de pertinence selon plusieurs critères :
- présence du nom de l’entreprise,
- exactitude de l’année fiscale,
- présence des mots-clés attendus,
- similarité avec la requête documentaire cible,
- et cohérence globale du document.

Les incohérences détectées peuvent entraîner des pénalités de score :
- mauvaise année,
- absence d’année,
- présence de termes incompatibles avec le document recherché.

---

### 3. Conversion du score en niveau de confiance

Le score brut obtenu est ensuite converti en :
- pseudo-probabilité estimée de pertinence,
- ainsi qu’en niveau de confiance interprétable :
  - HIGH_CONFIDENCE,
  - MEDIUM_CONFIDENCE,
  - LOW_CONFIDENCE,
  - ou REJECTED.

Cette étape permet de rendre le pipeline plus robuste et plus facilement exploitable pour les phases de sélection automatique.

---

### 4. Construction d’une base de résultats scorés

L’ensemble des documents candidats scorés est ensuite regroupé dans une table de discovery enrichie contenant :
- les informations de l’entreprise,
- le type de document recherché,
- l’URL candidate,
- le score obtenu,
- les raisons du scoring,
- ainsi que les métadonnées de collecte.

Même lorsqu’aucun document pertinent n’est trouvé, le pipeline génère des lignes explicites documentant l’absence de résultat.  
Cela garantit :
- une meilleure traçabilité,
- une couverture documentaire mesurable,
- ainsi qu’une auditabilité complète du pipeline.

Cette étape constitue donc le mécanisme central de filtrage intelligent entre la discovery documentaire brute et la sélection finale des documents ESG à ingérer.

In [47]:
# ============================================================
# 11. SCORING
# ============================================================

KEYWORDS_BY_DOC = {
    "annual_report_urd": [
        "annual report", "universal registration document", "registration document", "urd",
        "document d'enregistrement universel", "rapport annuel"
    ],
    "sustainability_statement_csrd_esrs": [
        "sustainability statement", "csrd", "esrs", "sustainability report",
        "rapport de durabilité", "non-financial statement", "dpef",
        "declaration de performance extra-financiere"
    ],
    "climate_report_tcfd_transition_plan": [
        "climate", "tcfd", "transition plan", "climat", "climate report"
    ],
    "vigilance_plan": [
        "vigilance", "devoir de vigilance", "plan de vigilance"
    ],
    "half_year_financial_report": [
        "half-year", "half year", "interim", "semestriel", "half-yearly"
    ],
    "code_of_conduct": [
        "code of conduct", "ethics", "ethique", "code of ethics"
    ],
    "anti_corruption_policy": [
        "anti corruption", "anti-corruption", "bribery", "corruption"
    ],
    "human_rights_policy": [
        "human rights", "droits humains"
    ],
    "dei_policy": [
        "diversity", "equity", "inclusion", "dei", "diversité", "equal opportunities"
    ],
    "environmental_policy": [
        "environment", "environmental", "climate", "biodiversity", "water", "environnement"
    ],
    "supplier_code_of_conduct": [
        "supplier", "purchasing", "achats", "responsible purchasing", "supplier code"
    ],
    "investor_presentation": [
        "investor presentation", "capital markets", "roadshow", "presentation"
    ],
    "agm_minutes_resolutions": [
        "agm", "general meeting", "resolution", "assemblée générale", "shareholders meeting"
    ],
    "cdp_response": [
        "cdp", "climate change"
    ],
    "sbti_validation": [
        "sbti", "science based", "science-based"
    ],
    "assurance_report": [
        "assurance", "limited assurance", "reasonable assurance", "audit", "verification statement"
    ]
}

NEGATIVE_KEYWORDS_BY_DOC = {
    "annual_report_urd": [
        "voting rights", "voting-rights", "press release", "half-year", "presentation"
    ],
    "half_year_financial_report": [
        "annual report", "universal registration document"
    ],
    "investor_presentation": [
        "voting rights", "press release"
    ],
    "agm_minutes_resolutions": [
        "annual report", "sustainability report"
    ]
}


def score_to_probability(score):
    return round(1 / (1 + math.exp(-(score - 10) / 5)), 3)


def confidence_from_score(score, year_match=True):
    if score is None:
        return "NO_CANDIDATE"
    if score >= 18 and year_match:
        return "HIGH_CONFIDENCE"
    if score >= 10:
        return "MEDIUM_CONFIDENCE"
    if score >= 5:
        return "LOW_CONFIDENCE"
    return "REJECTED"


def score_pdf_candidate(pdf_url, anchor_text, company, doc_subtype, year, expected_query=None):
    text = f"{pdf_url} {anchor_text}".lower()
    score = 0
    reasons = []

    company_tokens = re.findall(r"[a-zA-Z0-9]{3,}", company.lower())

    for token in company_tokens:
        if token in text:
            score += 2
            reasons.append(f"company_token:{token}")

    year_match = True
    detected_years = extract_years_from_text(text)

    if year != "current":
        year = int(year)

        if str(year) in text:
            score += 8
            reasons.append("exact_year_match")
            year_match = True
        elif detected_years:
            nearest_gap = min(abs(year - y) for y in detected_years)

            if nearest_gap == 1:
                score -= 3
                reasons.append("near_year_mismatch")
            else:
                score -= 6
                reasons.append("far_year_mismatch")

            year_match = False
        else:
            score -= 4
            reasons.append("year_missing")
            year_match = False

    for keyword in KEYWORDS_BY_DOC.get(doc_subtype, []):
        if keyword in text:
            score += 5
            reasons.append(f"keyword:{keyword}")

    for keyword in NEGATIVE_KEYWORDS_BY_DOC.get(doc_subtype, []):
        if keyword in text:
            score -= 6
            reasons.append(f"negative_keyword:{keyword}")

    if expected_query:
        query_tokens = tokenize_query(expected_query)
        matched_tokens = [token for token in query_tokens if token in text]

        if matched_tokens:
            query_bonus = min(6, len(set(matched_tokens)))
            score += query_bonus
            reasons.append(f"query_token_overlap:{query_bonus}")

    if ".pdf" in text:
        score += 2
        reasons.append("pdf_extension")

    probability = score_to_probability(score)
    confidence = confidence_from_score(score, year_match=year_match)

    return {
        "score": score,
        "confidence_level": confidence,
        "estimated_probability": probability,
        "year_match": year_match,
        "reasons": "; ".join(reasons)
    }


def missing_discovery_row(company, emitter, doc_spec, year, expected_query, status, reason):
    return {
        "company": company,
        "ticker": emitter.get("ticker"),
        "isin": emitter.get("isin"),
        "jurisdiction": emitter.get("jurisdiction"),
        "strate": doc_spec["strate"],
        "doc_type": doc_spec["doc_type"],
        "doc_subtype": doc_spec["doc_subtype"],
        "fiscal_year": year,
        "period_type": doc_spec["period_type"],
        "expected_query": expected_query,
        "source_page": None,
        "candidate_pdf_url": None,
        "anchor_text": None,
        "valid_pdf": False,
        "score": None,
        "confidence_level": "NO_CANDIDATE",
        "estimated_probability": None,
        "discovery_status": status,
        "selection_reason": reason,
        "retrieval_date": now_utc()
    }


def score_pdf_index_against_registry(company, emitter, pdf_index, years, document_registry):
    rows = []

    valid_index = pdf_index[pdf_index["valid_pdf"] == True].copy() if not pdf_index.empty else pd.DataFrame()

    for doc_spec in document_registry:
        for year in get_years_for_doc(doc_spec, years):
            expected_query = build_expected_query(doc_spec, company, year)

            if valid_index.empty:
                rows.append(
                    missing_discovery_row(
                        company=company,
                        emitter=emitter,
                        doc_spec=doc_spec,
                        year=year,
                        expected_query=expected_query,
                        status="NO_VALID_PDF_IN_INDEX",
                        reason="No valid PDF was available in the explored corporate pages."
                    )
                )
                continue

            for _, pdf_row in valid_index.iterrows():
                scoring = score_pdf_candidate(
                    pdf_url=pdf_row["candidate_pdf_url"],
                    anchor_text=pdf_row["anchor_text"],
                    company=company,
                    doc_subtype=doc_spec["doc_subtype"],
                    year=year,
                    expected_query=expected_query
                )

                rows.append({
                    "company": company,
                    "ticker": emitter.get("ticker"),
                    "isin": emitter.get("isin"),
                    "jurisdiction": emitter.get("jurisdiction"),
                    "strate": doc_spec["strate"],
                    "doc_type": doc_spec["doc_type"],
                    "doc_subtype": doc_spec["doc_subtype"],
                    "fiscal_year": year,
                    "period_type": doc_spec["period_type"],
                    "expected_query": expected_query,
                    "source_page": pdf_row["source_page"],
                    "candidate_pdf_url": pdf_row["candidate_pdf_url"],
                    "anchor_text": pdf_row["anchor_text"],
                    "valid_pdf": True,
                    "score": scoring["score"],
                    "confidence_level": scoring["confidence_level"],
                    "estimated_probability": scoring["estimated_probability"],
                    "discovery_status": "SCORED_CANDIDATE",
                    "selection_reason": scoring["reasons"],
                    "retrieval_date": now_utc()
                })

    discovery = pd.DataFrame(rows, columns=DISCOVERY_COLUMNS)

    if not discovery.empty:
        discovery.to_csv(
            DISCOVERY_PATH,
            mode="a" if DISCOVERY_PATH.exists() else "w",
            header=not DISCOVERY_PATH.exists(),
            index=False
        )

    return discovery


## 12. Sélection robuste, orchestration du pipeline et modes d’ingestion

Cette dernière couche du pipeline transforme les résultats de discovery et de scoring en une collecte documentaire exploitable et structurée.

L’objectif est de passer d’un ensemble de documents candidats potentiels à une sélection robuste de documents ESG réellement pertinents, tout en garantissant :
- la traçabilité des décisions,
- la reproductibilité du pipeline,
- et la robustesse de l’ingestion finale.

---

## 1. Sélection robuste des meilleurs documents

Après la phase de scoring, plusieurs documents candidats peuvent exister pour un même :
- émetteur,
- exercice fiscal,
- et type documentaire.

Le pipeline applique alors une logique de sélection robuste visant à conserver uniquement le meilleur candidat documentaire par catégorie.

La sélection repose sur des seuils de confiance configurables :
- un seuil de téléchargement automatique,
- ainsi qu’un seuil intermédiaire nécessitant une revue manuelle.

Trois cas principaux sont alors distingués :
- document automatiquement sélectionné pour ingestion,
- document nécessitant une validation humaine complémentaire,
- ou absence de document suffisamment pertinent.

Cette approche permet :
- de limiter les faux positifs,
- d’éviter les ingestions erronées,
- et de conserver une logique de contrôle qualité explicite.

Toutes les décisions de sélection sont enregistrées dans une table dédiée afin de garantir l’auditabilité complète du pipeline.

---

## 2. Orchestration complète du pipeline documentaire

Le pipeline implémente ensuite une logique d’orchestration unifiée reliant automatiquement :
- la normalisation des entreprises,
- la discovery documentaire,
- le scoring,
- la sélection,
- puis l’ingestion finale des documents retenus.

Cette orchestration permet de lancer une collecte ESG complète à partir d’une simple liste d’entreprises et d’années fiscales.

Le pipeline :
1. normalise les émetteurs,
2. construit un index documentaire par entreprise,
3. score les documents candidats,
4. sélectionne les meilleurs documents,
5. puis télécharge et enregistre automatiquement les PDFs validés.

Cette architecture modulaire facilite :
- la maintenance,
- l’extensibilité,
- la reproductibilité,
- ainsi que l’industrialisation potentielle du pipeline.

---

## 3. Gestion de plusieurs modes d’entrée utilisateur

Le pipeline ne dépend pas exclusivement de la discovery automatique.

En complément de la collecte web, il permet également :
- l’ingestion directe d’URLs fournies par l’utilisateur,
- ainsi que l’import manuel de fichiers PDF locaux.

Cette flexibilité permet :
- d’intégrer des documents obtenus manuellement,
- de compléter certains cas de collecte complexes,
- ou d’enrichir ponctuellement la base documentaire ESG.

Tous les documents importés suivent ensuite exactement les mêmes étapes :
- validation,
- normalisation,
- déduplication,
- enregistrement des métadonnées,
- et traçabilité des événements de collecte.

Le pipeline conserve ainsi une logique unifiée quel que soit le mode d’entrée utilisé.

In [48]:
# ============================================================
# 12. SÉLECTION ROBUSTE
# ============================================================

def select_best_documents(discovery_results, min_score_download=10, min_score_review=5):
    if discovery_results.empty:
        return pd.DataFrame(columns=SELECTION_COLUMNS)

    selected_rows = []
    group_cols = ["company", "fiscal_year", "doc_subtype"]

    for _, group in discovery_results.groupby(group_cols, dropna=False):
        scored = group[group["score"].notna()].copy()

        if scored.empty:
            best = group.iloc[0]

            row = {
                "company": best["company"],
                "ticker": best["ticker"],
                "isin": best["isin"],
                "jurisdiction": best["jurisdiction"],
                "strate": best["strate"],
                "doc_type": best["doc_type"],
                "doc_subtype": best["doc_subtype"],
                "fiscal_year": best["fiscal_year"],
                "period_type": best["period_type"],
                "expected_query": best["expected_query"],
                "selected_url": None,
                "score": None,
                "confidence_level": "NO_CANDIDATE",
                "estimated_probability": None,
                "selection_status": best["discovery_status"],
                "selection_reason": best["selection_reason"],
                "retrieval_date": now_utc()
            }

            selected_rows.append(row)
            save_selection(row)
            continue

        scored = scored.sort_values("score", ascending=False)
        best = scored.iloc[0]

        if best["score"] >= min_score_download:
            status = "SELECTED_FOR_DOWNLOAD"
            reason = "Best candidate passed automatic download threshold."
            selected_url = best["candidate_pdf_url"]

        elif best["score"] >= min_score_review:
            status = "REVIEW_REQUIRED"
            reason = "Candidate exists but confidence is not sufficient for automatic ingestion."
            selected_url = None

        else:
            status = "NO_RELEVANT_DOCUMENT_FOUND"
            reason = "No candidate reached the minimum relevance threshold."
            selected_url = None

        row = {
            "company": best["company"],
            "ticker": best["ticker"],
            "isin": best["isin"],
            "jurisdiction": best["jurisdiction"],
            "strate": best["strate"],
            "doc_type": best["doc_type"],
            "doc_subtype": best["doc_subtype"],
            "fiscal_year": best["fiscal_year"],
            "period_type": best["period_type"],
            "expected_query": best["expected_query"],
            "selected_url": selected_url,
            "score": best["score"],
            "confidence_level": best["confidence_level"],
            "estimated_probability": best["estimated_probability"],
            "selection_status": status,
            "selection_reason": reason,
            "retrieval_date": now_utc()
        }

        selected_rows.append(row)
        save_selection(row)

    return pd.DataFrame(selected_rows, columns=SELECTION_COLUMNS)


# ============================================================
# 13. ORCHESTRATION DISCOVERY → SÉLECTION → INGESTION
# ============================================================

def run_document_discovery_optimized(
    companies,
    years,
    emitters_reference=EMITTERS,
    document_registry=DOCUMENT_REGISTRY,
    allow_domain_guess=True
):
    normalized_emitters = normalize_emitters(
        companies=companies,
        emitters_reference=emitters_reference,
        allow_domain_guess=allow_domain_guess
    )

    all_discovery = []

    for emitter in normalized_emitters:
        company = emitter["company"]

        print(f"\n[DISCOVERY] Company: {company}")
        print(f"[DISCOVERY] Reference status: {emitter.get('reference_status')}")
        print(f"[DISCOVERY] Explored domains: {emitter.get('official_domains', [])}")

        pdf_index = build_company_pdf_index(
            company=company,
            emitter=emitter,
            years=years,
            document_registry=document_registry,
            use_ddgs=True
        )
        
        print(f"[DISCOVERY] Unique PDF candidates found: {len(pdf_index)}")

        discovery = score_pdf_index_against_registry(
            company=company,
            emitter=emitter,
            pdf_index=pdf_index,
            years=years,
            document_registry=document_registry
        )

        all_discovery.append(discovery)

    if not all_discovery:
        return pd.DataFrame(columns=DISCOVERY_COLUMNS)

    return pd.concat(all_discovery, ignore_index=True)


def ingest_company_documents_from_selection(selection_df, emitters_reference=EMITTERS):
    if selection_df.empty:
        return []

    emitter_map = {item["company"]: item for item in emitters_reference}
    results = []

    selected = selection_df[selection_df["selection_status"] == "SELECTED_FOR_DOWNLOAD"].copy()

    for _, row in selected.iterrows():
        company = row["company"]
        emitter = emitter_map.get(company, {})

        result = ingest_from_url(
            url=row["selected_url"],
            company=company,
            ticker=emitter.get("ticker", row.get("ticker")),
            isin=emitter.get("isin", row.get("isin")),
            jurisdiction=emitter.get("jurisdiction", row.get("jurisdiction")),
            strate=row["strate"],
            doc_type=row["doc_type"],
            doc_subtype=row["doc_subtype"],
            fiscal_year=row["fiscal_year"],
            period_type=row["period_type"],
            input_mode="company_name"
        )

        results.append(result)
        time.sleep(0.3)

    return results


def ingest_from_company_names(
    companies,
    years,
    min_score_download=10,
    min_score_review=5,
    allow_domain_guess=True
):
    discovery_results = run_document_discovery_optimized(
        companies=companies,
        years=years,
        allow_domain_guess=allow_domain_guess
    )

    selection_df = select_best_documents(
        discovery_results=discovery_results,
        min_score_download=min_score_download,
        min_score_review=min_score_review
    )

    ingestion_results = ingest_company_documents_from_selection(selection_df)

    return {
        "discovery_results": discovery_results,
        "selection_results": selection_df,
        "ingestion_results": ingestion_results
    }


# ============================================================
# 14. MODES D'ENTRÉE UTILISATEUR
# ============================================================

def ingest_user_urls(urls, company="user_provided", doc_subtype="user_url_pdf", fiscal_year="unknown"):
    results = []

    for url in urls:
        result = ingest_from_url(
            url=url,
            company=company,
            doc_subtype=doc_subtype,
            fiscal_year=fiscal_year,
            input_mode="user_url"
        )
        results.append(result)

    return results


def ingest_user_local_pdfs(file_paths, company="user_provided", doc_subtype="uploaded_pdf", fiscal_year="unknown"):
    results = []

    for file_path in file_paths:
        result = ingest_from_local_pdf(
            file_path=file_path,
            company=company,
            doc_subtype=doc_subtype,
            fiscal_year=fiscal_year
        )
        results.append(result)
        
    return results
    

## 13. Réinitialisation du pipeline de collecte

Cette cellule remet le pipeline dans un état propre avant une nouvelle exécution.

Elle supprime :
- les fichiers de logs et de métadonnées,
- les résultats de discovery et de sélection,
- ainsi que les PDFs précédemment téléchargés dans `raw/`.

Les dossiers devenus vides sont ensuite automatiquement nettoyés.

Cette étape permet de relancer une collecte complète sans interférence avec les exécutions précédentes et garantit une meilleure reproductibilité du pipeline.

In [49]:
# Supprimer les fichiers CSV
for file in [LOG_PATH, METADATA_PATH, DISCOVERY_PATH, PDF_INDEX_PATH, SELECTION_PATH]:
    if file.exists():
        file.unlink()
        print(f"Supprimé : {file}")

# Supprimer tous les PDF téléchargés dans raw/
if RAW_DIR.exists():
    for pdf_file in RAW_DIR.rglob("*.pdf"):
        pdf_file.unlink()
        print(f"PDF supprimé : {pdf_file}")

# Optionnel : supprimer les dossiers vides dans raw/
for folder in sorted(RAW_DIR.rglob("*"), reverse=True):
    if folder.is_dir() and not any(folder.iterdir()):
        folder.rmdir()
        print(f"Dossier vide supprimé : {folder}")

## 14. Exécution principale sur l’univers CAC 40

Cette cellule lance la collecte documentaire principale sur l’ensemble des entreprises du CAC 40 définies dans le référentiel `EMITTERS`.

Le pipeline distingue deux niveaux de documents :
- les documents prioritaires, utilisés pour la collecte principale ;
- les documents secondaires, conservés pour d’éventuels compléments ultérieurs.

Le run principal se concentre ici sur les documents ESG et réglementaires les plus importants :
- rapports annuels / URD,
- sustainability statements,
- rapports climat ou TCFD,
- plans de vigilance,
- rapports d’assurance.

Pour chaque entreprise et chaque année étudiée, le pipeline exécute successivement :
- la discovery documentaire,
- le scoring des PDFs candidats,
- la sélection robuste des meilleurs documents,
- puis l’ingestion automatique des documents suffisamment fiables.

Les résultats imprimés permettent de contrôler rapidement :
- le volume de documents détectés,
- le nombre de documents sélectionnés,
- le nombre de PDFs effectivement ingérés,
- ainsi que la répartition des statuts de sélection.

In [50]:
# ============================================================
# 0. Entreprises CAC 40
# ============================================================

CAC40_COMPANIES = [emitter["company"] for emitter in EMITTERS]


# ============================================================
# 1. Registre principal : documents ESG prioritaires
# ============================================================

CORE_REPORTS_REGISTRY = [
    doc for doc in DOCUMENT_REGISTRY
    if doc["doc_subtype"] in [
        "annual_report_urd",
        "sustainability_statement_csrd_esrs",
        "climate_report_tcfd_transition_plan",
        "vigilance_plan",
        "assurance_report"
    ]
]


# ============================================================
# 2. Registre secondaire : documents complémentaires
# ============================================================

SECONDARY_REPORTS_REGISTRY = [
    doc for doc in DOCUMENT_REGISTRY
    if doc["doc_subtype"] in [
        "code_of_conduct",
        "anti_corruption_policy",
        "human_rights_policy",
        "dei_policy",
        "environmental_policy",
        "supplier_code_of_conduct",
        "investor_presentation",
        "agm_minutes_resolutions",
        "cdp_response",
        "sbti_validation",
        "half_year_financial_report"
    ]
]

# ============================================================
# RUN PRINCIPAL CORRIGÉ — documents importants uniquement
# ============================================================

discovery_cac40_core = run_document_discovery_optimized(
    companies=CAC40_COMPANIES,
    years=[2023, 2024],
    emitters_reference=EMITTERS,
    document_registry=CORE_REPORTS_REGISTRY,
    allow_domain_guess=False
)

selection_cac40_core = select_best_documents(
    discovery_results=discovery_cac40_core,
    min_score_download=12,
    min_score_review=7
)

ingestion_cac40_core = ingest_company_documents_from_selection(
    selection_df=selection_cac40_core,
    emitters_reference=EMITTERS
)

print("===== RUN PRINCIPAL — DOCUMENTS IMPORTANTS =====")
print("Discovery:", discovery_cac40_core.shape)
print("Selection:", selection_cac40_core.shape)
print("Documents ingérés:", len([x for x in ingestion_cac40_core if x is not None]))
print(selection_cac40_core["selection_status"].value_counts(dropna=False))


[DISCOVERY] Company: TotalEnergies
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['totalenergies.com']
[DISCOVERY] Unique PDF candidates found: 45

[DISCOVERY] Company: Schneider Electric
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['se.com', 'schneider-electric.com']
[DISCOVERY] Unique PDF candidates found: 45

[DISCOVERY] Company: LVMH
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['lvmh.com']
[DISCOVERY] Unique PDF candidates found: 35

[DISCOVERY] Company: Air Liquide
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['airliquide.com']
[DISCOVERY] Unique PDF candidates found: 112

[DISCOVERY] Company: Sanofi
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['sanofi.com']
[DDGS] Search failed for Sanofi / assurance_report / 2023: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.c

## 15. Analyse de couverture documentaire et collecte secondaire ciblée

### 1. Évaluation de la couverture documentaire prioritaire

Cette cellule mesure, pour chaque couple entreprise-année, la qualité de la collecte principale sur les documents ESG prioritaires.

On calcule notamment :
- le nombre de documents prioritaires sélectionnés automatiquement ;
- le nombre de documents nécessitant une revue manuelle ;
- le score moyen et maximal ;
- un ratio de couverture documentaire.

Ce diagnostic permet d’identifier les entreprises pour lesquelles la collecte principale est suffisante et celles qui nécessitent une collecte complémentaire.

In [51]:
# ============================================================
# 1. Score de couverture des documents importants
# ============================================================

CORE_REQUIRED_DOCS = [
    "annual_report_urd",
    "sustainability_statement_csrd_esrs",
    "climate_report_tcfd_transition_plan",
    "vigilance_plan",
    "assurance_report"
]


def build_core_coverage_score(selection_core, required_docs=CORE_REQUIRED_DOCS):
    df = selection_core.copy()

    df["is_selected"] = df["selection_status"].eq("SELECTED_FOR_DOWNLOAD")
    df["is_review"] = df["selection_status"].eq("REVIEW_REQUIRED")

    coverage = (
        df.groupby(["company", "fiscal_year"], dropna=False)
        .agg(
            n_core_docs=("doc_subtype", "count"),
            n_selected_core=("is_selected", "sum"),
            n_review_core=("is_review", "sum"),
            avg_core_score=("score", "mean"),
            max_core_score=("score", "max")
        )
        .reset_index()
    )

    coverage["core_coverage_ratio"] = (
        coverage["n_selected_core"] / len(required_docs)
    )

    def classify(row):
        if row["core_coverage_ratio"] >= 0.6 and row["avg_core_score"] >= 12:
            return "GOOD_CORE_COVERAGE"
        elif row["core_coverage_ratio"] >= 0.4:
            return "MEDIUM_CORE_COVERAGE"
        else:
            return "LOW_CORE_COVERAGE"

    coverage["core_coverage_status"] = coverage.apply(classify, axis=1)

    return coverage


core_coverage = build_core_coverage_score(selection_cac40_core)

display(
    core_coverage.sort_values(
        ["core_coverage_status", "core_coverage_ratio"],
        ascending=[True, False]
    )
)



,company,fiscal_year,n_core_docs,n_selected_core,n_review_core,avg_core_score,max_core_score,core_coverage_ratio,core_coverage_status
0,AXA,2023,5,5,0,29.6,37,1.0,GOOD_CORE_COVERAGE
1,AXA,2024,5,5,0,24.6,32,1.0,GOOD_CORE_COVERAGE
2,Accor,2023,5,5,0,27.6,35,1.0,GOOD_CORE_COVERAGE
3,Accor,2024,5,5,0,24.8,32,1.0,GOOD_CORE_COVERAGE
4,Air Liquide,2023,5,5,0,29.4,33,1.0,GOOD_CORE_COVERAGE
...,...,...,...,...,...,...,...,...,...
17,Capgemini,2024,5,4,1,20.6,29,0.8,GOOD_CORE_COVERAGE
43,LVMH,2024,5,4,1,21.8,29,0.8,GOOD_CORE_COVERAGE
57,STMicroelectronics,2024,5,4,1,21.8,29,0.8,GOOD_CORE_COVERAGE
50,Pernod Ricard,2023,5,3,2,20.2,29,0.6,GOOD_CORE_COVERAGE


### 2. Identification des besoins de collecte 

Cette cellule applique une règle de décision simple : la collecte secondaire est activée uniquement pour les couples entreprise-année dont la couverture documentaire principale est insuffisante ou moyenne.

L’objectif est d’éviter une collecte secondaire systématique sur tout l’univers CAC 40, afin de réduire :
- le temps de calcul ;
- les requêtes web inutiles ;
- le bruit documentaire ;
- les risques de doublons.

La collecte secondaire devient ainsi ciblée, conditionnelle et justifiée par un diagnostic de couverture.

In [52]:
def choose_secondary_scope(core_coverage):
    target_statuses = [
        "LOW_CORE_COVERAGE",
        "MEDIUM_CORE_COVERAGE"
    ]

    secondary_targets = (
        core_coverage[
            core_coverage["core_coverage_status"].isin(target_statuses)
        ]
        .copy()
        .sort_values(["core_coverage_status", "core_coverage_ratio"])
    )

    return secondary_targets


secondary_targets = choose_secondary_scope(core_coverage)

SECONDARY_TARGET_COMPANIES = sorted(secondary_targets["company"].dropna().unique())
SECONDARY_TARGET_YEARS = sorted(secondary_targets["fiscal_year"].dropna().unique())

print("Nombre de couples entreprise-année ciblés :", len(secondary_targets))
print("Nombre d'entreprises ciblées :", len(SECONDARY_TARGET_COMPANIES))
print("Années ciblées :", SECONDARY_TARGET_YEARS)

display(secondary_targets)

Nombre de couples entreprise-année ciblés : 1
Nombre d'entreprises ciblées : 1
Années ciblées : [np.int64(2023)]


,company,fiscal_year,n_core_docs,n_selected_core,n_review_core,avg_core_score,max_core_score,core_coverage_ratio,core_coverage_status
74,Unibail-Rodamco-Westfield,2023,5,1,4,14.6,29,0.2,LOW_CORE_COVERAGE


### 3. Collecte secondaire ciblée

Cette cellule lance une collecte complémentaire uniquement sur les entreprises et années identifiées comme insuffisamment couvertes lors du run principal.

Le registre secondaire contient des documents utiles mais moins prioritaires que les rapports principaux :
- politiques ESG ;
- codes de conduite ;
- documents fournisseurs ;
- présentations investisseurs ;
- réponses CDP ;
- validations SBTi ;
- rapports semestriels.

Cette étape permet d’enrichir la base documentaire sans alourdir inutilement la collecte globale.

In [53]:
if len(secondary_targets) == 0:
    print("Aucune collecte secondaire nécessaire : la couverture principale est suffisante.")

    discovery_cac40_secondary_targeted = pd.DataFrame(columns=DISCOVERY_COLUMNS)
    selection_cac40_secondary_targeted = pd.DataFrame(columns=SELECTION_COLUMNS)

else:
    discovery_cac40_secondary_targeted = run_document_discovery_optimized(
        companies=SECONDARY_TARGET_COMPANIES,
        years=SECONDARY_TARGET_YEARS,
        emitters_reference=EMITTERS,
        document_registry=SECONDARY_REPORTS_REGISTRY,
        allow_domain_guess=False
    )

    selection_cac40_secondary_targeted = select_best_documents(
        discovery_results=discovery_cac40_secondary_targeted,
        min_score_download=16,
        min_score_review=9
    )

    print("===== RUN SECONDAIRE CIBLÉ =====")
    print("Discovery:", discovery_cac40_secondary_targeted.shape)
    print("Selection:", selection_cac40_secondary_targeted.shape)

    display(
        selection_cac40_secondary_targeted[
            "selection_status"
        ].value_counts(dropna=False)
    )

    display(selection_cac40_secondary_targeted.head(50))


[DISCOVERY] Company: Unibail-Rodamco-Westfield
[DISCOVERY] Reference status: FOUND_IN_REFERENCE
[DISCOVERY] Explored domains: ['urw.com']
[DDGS] Search failed for Unibail-Rodamco-Westfield / environmental_policy / current: ('error sending request for url (https://html.duckduckgo.com/html/)', 'https://html.duckduckgo.com/html/')
[DISCOVERY] Unique PDF candidates found: 7
===== RUN SECONDAIRE CIBLÉ =====
Discovery: (55, 20)
Selection: (11, 17)


selection_status
REVIEW_REQUIRED          6
SELECTED_FOR_DOWNLOAD    5
Name: count, dtype: int64

,company,ticker,isin,jurisdiction,strate,doc_type,doc_subtype,fiscal_year,period_type,expected_query,selected_url,score,confidence_level,estimated_probability,selection_status,selection_reason,retrieval_date
0,Unibail-Rodamco-Westfield,URW,FR0013326246,France,3,corporate_communication,agm_minutes_resolutions,2023,adhoc,Unibail-Rodamco-Westfield annual general meeti...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.687988+00:00
1,Unibail-Rodamco-Westfield,URW,FR0013326246,France,4,external_source,cdp_response,2023,FY,Unibail-Rodamco-Westfield CDP climate change r...,https://www.bnains.org/archives/communiques/Un...,21,HIGH_CONFIDENCE,0.900,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.689835+00:00
2,Unibail-Rodamco-Westfield,URW,FR0013326246,France,1,regulatory,half_year_financial_report,2023,H1,Unibail-Rodamco-Westfield half year financial ...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.691363+00:00
3,Unibail-Rodamco-Westfield,URW,FR0013326246,France,3,corporate_communication,investor_presentation,2023,adhoc,Unibail-Rodamco-Westfield investor presentatio...,https://www.bnains.org/archives/communiques/Un...,20,HIGH_CONFIDENCE,0.881,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.693034+00:00
4,Unibail-Rodamco-Westfield,URW,FR0013326246,France,4,external_source,sbti_validation,2023,adhoc,Unibail-Rodamco-Westfield SBTi validated targets,https://www.bnains.org/archives/communiques/Un...,19,HIGH_CONFIDENCE,0.858,SELECTED_FOR_DOWNLOAD,Best candidate passed automatic download thres...,2026-05-10T08:40:05.694599+00:00
5,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,anti_corruption_policy,current,perpetual,Unibail-Rodamco-Westfield anti corruption poli...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.696044+00:00
6,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,code_of_conduct,current,perpetual,Unibail-Rodamco-Westfield code of conduct PDF,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.700451+00:00
7,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,dei_policy,current,perpetual,Unibail-Rodamco-Westfield diversity equity inc...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.703143+00:00
8,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,environmental_policy,current,perpetual,Unibail-Rodamco-Westfield environmental policy...,None,11,MEDIUM_CONFIDENCE,0.550,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.704863+00:00
9,Unibail-Rodamco-Westfield,URW,FR0013326246,France,2,policy,human_rights_policy,current,perpetual,Unibail-Rodamco-Westfield human rights policy PDF,None,12,MEDIUM_CONFIDENCE,0.599,REVIEW_REQUIRED,Candidate exists but confidence is not suffici...,2026-05-10T08:40:05.706535+00:00


### 4. Ingestion des documents secondaires sélectionnés

Cette cellule télécharge et enregistre uniquement les documents secondaires qui ont été retenus automatiquement après la phase de sélection.

Elle prolonge donc le run secondaire ciblé en appliquant la même logique d’ingestion que pour les documents prioritaires :
- téléchargement du PDF ;
- validation technique ;
- détection des doublons ;
- enregistrement des métadonnées ;
- mise à jour des logs de collecte.

Les documents nécessitant une revue manuelle ne sont pas ingérés automatiquement.

In [54]:
if selection_cac40_secondary_targeted.empty:
    print("Aucun document secondaire sélectionné : aucune ingestion secondaire lancée.")

    ingestion_cac40_secondary_selected = []

else:
    secondary_to_ingest = selection_cac40_secondary_targeted[
        selection_cac40_secondary_targeted["selection_status"].eq("SELECTED_FOR_DOWNLOAD")
    ].copy()

    ingestion_cac40_secondary_selected = ingest_company_documents_from_selection(
        selection_df=secondary_to_ingest,
        emitters_reference=EMITTERS
    )

    print("===== INGESTION SECONDAIRE SÉLECTIVE =====")
    print(
        "Documents secondaires effectivement ingérés :",
        len([x for x in ingestion_cac40_secondary_selected if x is not None])
    )

===== INGESTION SECONDAIRE SÉLECTIVE =====
Documents secondaires effectivement ingérés : 1


# PHASE 2 — Extraction intelligente et structuration des données ESG

Objectif de la phase

Après la phase 1 consacrée à la collecte et à la sélection des documents ESG pertinents, cette phase vise à transformer les PDFs retenus en données ESG structurées, exploitables et auditables.

Le pipeline devra permettre d’extraire automatiquement des informations ESG à partir :

- du texte ;
- des tableaux ;
- des images et graphiques ;
- des annexes documentaires.

L’objectif final est de construire un panel ESG harmonisé pouvant être utilisé pour :

- le scoring ESG ;
- l’analyse des risques ;
- les modèles quantitatifs ;
- la construction et l’évaluation de portefeuilles durables.

Cette phase repose sur plusieurs principes :

- traçabilité des données extraites ;
- robustesse face à l’hétérogénéité des rapports ESG ;
- standardisation des métriques ;
- automatisation progressive avec contrôle qualité.

## 2. Configuration héritée de la phase 1 et nouveaux répertoires

La phase 2 s’appuie directement sur les sorties validées de la phase 1.  
L’objectif n’est plus de découvrir ou télécharger les documents, mais de transformer les PDFs retenus en données ESG structurées, traçables et auditables.

La configuration reprend donc l’arborescence existante :

- `esg_data/raw/` : PDFs collectés ;
- `esg_data/metadata/documents_metadata.csv` : métadonnées d’ingestion ;
- `esg_data/metadata/selected_documents.csv` : décisions de sélection documentaire.

La phase 2 ajoute des répertoires dédiés aux étapes d’extraction :

- `parsed/pages/` : texte extrait page par page ;
- `parsed/blocks/` : blocs textuels ou visuels structurés ;
- `parsed/tables/` : tableaux extraits ;
- `parsed/images/` : images ou pages rendues pour OCR/revue ;
- `extraction/candidates/` : candidats métriques détectés ;
- `extraction/validated/` : métriques ESG validées automatiquement ou manuellement ;
- `review/` : file de revue humaine ;
- `panel/` : panel ESG final prêt pour l’analyse.

In [3]:
# ============================================================
# CONFIGURATION — PHASE 2
# Héritage de la phase 1 + sorties d'extraction ESG
# ============================================================

BASE_DIR = Path("esg_data")

# Répertoires hérités de la phase 1
RAW_DIR = BASE_DIR / "raw"
META_DIR = BASE_DIR / "metadata"
LOG_DIR = BASE_DIR / "logs"

METADATA_PATH = META_DIR / "documents_metadata.csv"
SELECTION_PATH = META_DIR / "selected_documents.csv"

# Nouveaux répertoires phase 2
PHASE2_DIR = BASE_DIR / "phase2_extraction"

PARSED_DIR = PHASE2_DIR / "parsed"
PAGES_DIR = PARSED_DIR / "pages"
BLOCKS_DIR = PARSED_DIR / "blocks"
TABLES_DIR = PARSED_DIR / "tables"
IMAGES_DIR = PARSED_DIR / "images"

EXTRACTION_DIR = PHASE2_DIR / "extraction"
CANDIDATES_DIR = EXTRACTION_DIR / "candidates"
VALIDATED_DIR = EXTRACTION_DIR / "validated"

REVIEW_DIR = PHASE2_DIR / "review"
PANEL_DIR = PHASE2_DIR / "panel"

PHASE2_LOG_DIR = PHASE2_DIR / "logs"

# Fichiers de sortie principaux
PARSED_PAGES_PATH = PAGES_DIR / "parsed_pages.csv"
PARSED_BLOCKS_PATH = BLOCKS_DIR / "parsed_blocks.csv"
EXTRACTED_TABLES_PATH = TABLES_DIR / "extracted_tables.csv"
METRIC_CANDIDATES_PATH = CANDIDATES_DIR / "metric_candidates.csv"
VALIDATED_METRICS_PATH = VALIDATED_DIR / "validated_esg_metrics.csv"
REVIEW_QUEUE_PATH = REVIEW_DIR / "review_queue.csv"
FINAL_PANEL_PATH = PANEL_DIR / "esg_panel_final.csv"
PHASE2_LOG_PATH = PHASE2_LOG_DIR / "phase2_extraction_log.csv"

# Création des répertoires
for directory in [
    PARSED_DIR,
    PAGES_DIR,
    BLOCKS_DIR,
    TABLES_DIR,
    IMAGES_DIR,
    EXTRACTION_DIR,
    CANDIDATES_DIR,
    VALIDATED_DIR,
    REVIEW_DIR,
    PANEL_DIR,
    PHASE2_LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Vérification des dépendances de la phase 1
required_phase1_files = {
    "documents_metadata": METADATA_PATH,
    "selected_documents": SELECTION_PATH,
}

missing_files = [
    name for name, path in required_phase1_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Fichiers hérités de la phase 1 manquants : "
        + ", ".join(missing_files)
    )

# Chargement des entrées phase 1
documents_metadata = pd.read_csv(METADATA_PATH)
selected_documents = pd.read_csv(SELECTION_PATH)

print("Configuration Phase 2 initialisée.")
print(f"PDF raw directory: {RAW_DIR}")
print(f"Nombre de documents en métadonnées: {len(documents_metadata)}")
print(f"Nombre de documents sélectionnés: {len(selected_documents)}")

Configuration Phase 2 initialisée.
PDF raw directory: esg_data/raw
Nombre de documents en métadonnées: 118
Nombre de documents sélectionnés: 411


## 3. Schémas explicites des sorties

Comme dans la phase 1, les schémas des sorties sont déclarés explicitement afin de garantir une structure stable entre les différents runs du pipeline.

Cette étape est importante car la phase 2 ne vise pas seulement à extraire des valeurs ESG, mais à produire des données traçables, contrôlables et exploitables.

La conception suit une logique de preuve : une métrique ESG n’est jamais seulement une valeur. Elle doit être associée à :

- une entreprise ;
- une année fiscale ;
- un document source ;
- une page ;
- un bloc textuel ou tabulaire ;
- une méthode d’extraction ;
- un score de confiance.

Ces schémas permettent donc de préparer les sorties intermédiaires et finales du pipeline : pages parsées, blocs extraits, tableaux, candidats métriques, métriques validées, file de revue et panel ESG final.

In [4]:
# ============================================================
# 3. SCHÉMAS DES SORTIES PHASE 2
# ============================================================

EXTRACTION_LOG_COLUMNS = [
    "run_id", "stage", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "status", "error_type", "error_message", "created_at"
]

PARSED_PAGE_COLUMNS = [
    "run_id", "document_id", "company", "ticker", "isin", "jurisdiction", "fiscal_year",
    "doc_subtype", "source_pdf_path", "sha256", "page_number", "text", "text_length",
    "has_tables", "has_images", "language", "section_labels", "section_scores",
    "extraction_quality_score", "created_at"
]

DOCUMENT_BLOCK_COLUMNS = [
    "run_id", "block_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "block_type", "block_order", "content", "bbox",
    "section_labels", "confidence", "created_at"
]

EXTRACTED_TABLE_COLUMNS = [
    "run_id", "table_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "table_order", "n_rows", "n_cols", "table_json",
    "extraction_method", "created_at"
]

EXTRACTED_IMAGE_COLUMNS = [
    "run_id", "image_id", "document_id", "company", "fiscal_year", "doc_subtype",
    "source_pdf_path", "page_number", "image_order", "image_path", "width", "height",
    "ocr_text", "ocr_status", "created_at"
]

METRIC_CANDIDATE_COLUMNS = [
    "run_id", "metric_id", "document_id", "company", "ticker", "isin", "jurisdiction",
    "fiscal_year", "doc_subtype", "metric_name", "metric_category", "raw_value", "raw_unit",
    "normalized_value", "normalized_unit", "source_page", "source_block_id",
    "source_text_excerpt", "extraction_method", "confidence_score", "validation_status",
    "quality_flags", "created_at"
]

REVIEW_QUEUE_COLUMNS = [
    "run_id", "review_id", "metric_id", "document_id", "company", "fiscal_year",
    "doc_subtype", "metric_name", "metric_category", "raw_value", "raw_unit",
    "normalized_value", "normalized_unit", "source_page", "source_block_id",
    "source_text_excerpt", "confidence_score", "review_reason", "review_priority",
    "review_status", "created_at"
]

VALIDATED_METRIC_COLUMNS = [
    "run_id", "metric_id", "document_id", "company", "ticker", "isin", "jurisdiction",
    "fiscal_year", "doc_subtype", "metric_name", "metric_category", "metric_value",
    "metric_unit", "source_page", "source_block_id", "source_text_excerpt",
    "extraction_method", "confidence_score", "validation_status", "quality_flags",
    "created_at"
]

COMPANY_YEAR_PANEL_COLUMNS = [
    "run_id", "company", "ticker", "isin", "jurisdiction", "fiscal_year", "metric_name",
    "metric_category", "metric_value", "metric_unit", "best_confidence_score",
    "validation_status", "source_document_id", "source_doc_subtype", "source_page",
    "source_text_excerpt", "quality_flags", "created_at"
]

QUALITY_REPORT_COLUMNS = [
    "run_id", "level", "key", "n_documents", "n_pages", "n_tables", "n_images",
    "n_metric_candidates", "n_validated_metrics", "n_review_required", "coverage_ratio",
    "created_at"
]

PHASE2_SCHEMAS = {
    "extraction_log": EXTRACTION_LOG_COLUMNS,
    "parsed_pages": PARSED_PAGE_COLUMNS,
    "document_blocks": DOCUMENT_BLOCK_COLUMNS,
    "extracted_tables": EXTRACTED_TABLE_COLUMNS,
    "extracted_images": EXTRACTED_IMAGE_COLUMNS,
    "metric_candidates": METRIC_CANDIDATE_COLUMNS,
    "review_queue": REVIEW_QUEUE_COLUMNS,
    "validated_metrics": VALIDATED_METRIC_COLUMNS,
    "company_year_panel": COMPANY_YEAR_PANEL_COLUMNS,
    "quality_report": QUALITY_REPORT_COLUMNS,
}

print("Schémas Phase 2 déclarés.")
print(f"Nombre de tables de sortie définies : {len(PHASE2_SCHEMAS)}")

Schémas Phase 2 déclarés.
Nombre de tables de sortie définies : 10


## 4. Fonctions utilitaires : identifiants, logs, I/O et normalisation texte

Cette section définit des fonctions transversales utilisées dans toute la phase 2.

Elles permettent de standardiser :
- la création d’un identifiant de run ;
- la génération d’identifiants stables pour les documents, pages, blocs, tables, images et métriques ;
- l’écriture de logs structurés ;
- l’écriture des sorties en Parquet, avec fallback automatique en CSV si `pyarrow` n’est pas disponible ;
- le nettoyage du texte extrait des PDFs ;
- la sérialisation JSON sécurisée pour stocker des objets complexes dans des fichiers tabulaires.

Ces fonctions assurent la robustesse, la traçabilité et la reproductibilité du pipeline.

In [6]:
# ============================================================
# 4. FONCTIONS UTILITAIRES — PHASE 2
# Identifiants, logs, I/O et normalisation texte
# ============================================================

def now_utc_iso():
    """
    Retourne l'horodatage courant au format ISO.
    """
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def create_run_id(prefix="phase2"):
    """
    Crée un identifiant unique pour un run du pipeline.
    """
    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    return f"{prefix}_{timestamp}"


def compute_file_sha256(file_path, chunk_size=1024 * 1024):
    """
    Calcule le hash SHA256 d'un fichier.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {file_path}")

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


def make_stable_id(*parts, prefix=None, length=16):
    """
    Crée un identifiant stable à partir d'une liste d'éléments.
    """
    raw = "||".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:length]

    if prefix:
        return f"{prefix}_{digest}"

    return digest


def safe_json_dumps(obj):
    """
    Convertit un objet Python en chaîne JSON sûre.
    """
    try:
        return json.dumps(obj, ensure_ascii=False)
    except TypeError:
        return json.dumps(str(obj), ensure_ascii=False)


def safe_json_loads(value, default=None):
    """
    Recharge une chaîne JSON de manière sécurisée.
    """
    if default is None:
        default = {}

    if pd.isna(value):
        return default

    try:
        return json.loads(value)
    except Exception:
        return default


def clean_extracted_text(text):
    """
    Nettoie un texte extrait d'un PDF.
    """
    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Normalisation des espaces insécables et caractères invisibles fréquents
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # Suppression des espaces multiples
    text = re.sub(r"[ \t]+", " ", text)

    # Normalisation des retours à la ligne excessifs
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Suppression des espaces en début/fin de lignes
    text = "\n".join(line.strip() for line in text.splitlines())

    return text.strip()


def normalize_text_for_matching(text):
    """
    Normalise un texte pour les recherches par mots-clés ou regex.
    """
    text = clean_extracted_text(text)
    text = text.lower()

    # Harmonisation simple des apostrophes et tirets
    text = text.replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")

    return text


def truncate_text(text, max_chars=500):
    """
    Tronque un texte pour les extraits de preuve.
    """
    text = clean_extracted_text(text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars].rstrip() + "..."


def append_log(log_path, log_row, columns=None):
    """
    Ajoute une ligne de log structurée dans un fichier CSV.
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    if columns is not None:
        row = {col: log_row.get(col, None) for col in columns}
    else:
        row = log_row

    df_log = pd.DataFrame([row])

    if log_path.exists():
        df_log.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df_log.to_csv(log_path, mode="w", header=True, index=False)


def write_dataframe(df, output_path, index=False):
    """
    Écrit un DataFrame en Parquet si possible, sinon en CSV.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.suffix.lower() == ".parquet":
        try:
            df.to_parquet(output_path, index=index)
            return output_path
        except ImportError:
            csv_path = output_path.with_suffix(".csv")
            df.to_csv(csv_path, index=index)
            return csv_path
        except Exception as e:
            csv_path = output_path.with_suffix(".csv")
            df.to_csv(csv_path, index=index)
            print(f"Parquet indisponible ou erreur d'écriture ({type(e).__name__}). Fallback CSV : {csv_path}")
            return csv_path

    elif output_path.suffix.lower() == ".csv":
        df.to_csv(output_path, index=index)
        return output_path

    else:
        csv_path = output_path.with_suffix(".csv")
        df.to_csv(csv_path, index=index)
        return csv_path


def ensure_dataframe_schema(df, columns):
    """
    Force un DataFrame à respecter un schéma de colonnes donné.
    """
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            df[col] = None

    return df[columns]


def initialize_empty_output(path, columns):
    """
    Initialise un fichier de sortie vide avec le schéma attendu.
    """
    empty_df = pd.DataFrame(columns=columns)
    return write_dataframe(empty_df, path)


RUN_ID = create_run_id()

print("Fonctions utilitaires Phase 2 chargées.")
print(f"RUN_ID courant : {RUN_ID}")

Fonctions utilitaires Phase 2 chargées.
RUN_ID courant : phase2_20260510_130745


/tmp/ipykernel_139103/965948883.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


## 5. Chargement des documents de la phase 1

Cette cellule constitue le point d’ancrage entre la phase 1 et la phase 2.

Elle charge les documents effectivement téléchargés en phase 1, vérifie leur présence locale, conserve les métadonnées utiles à l’extraction ESG, puis crée un identifiant documentaire stable.

Lorsque `selected_documents.csv` est disponible, le filtrage est effectué sur les documents sélectionnés en phase 1, en reliant `source_url` à `selected_url`.

In [9]:
# ============================================================
# 5. CHARGEMENT DES DOCUMENTS DE LA PHASE 1
# ============================================================

def normalize_join_key(series):
    """
    Normalise une colonne utilisée comme clé de jointure.
    """
    return (
        series
        .astype(str)
        .str.strip()
        .replace({"nan": None, "None": None})
    )


def load_phase1_documents(
    metadata_path=METADATA_PATH,
    selection_path=SELECTION_PATH,
    raw_dir=RAW_DIR,
    use_selected_documents=True
):
    """
    Charge les documents issus de la phase 1 et prépare l'entrée de la phase 2.
    """

    metadata_path = Path(metadata_path)
    selection_path = Path(selection_path)
    raw_dir = Path(raw_dir)

    if not metadata_path.exists():
        raise FileNotFoundError(f"Fichier metadata introuvable : {metadata_path}")

    documents = pd.read_csv(metadata_path)
    documents.columns = [col.strip() for col in documents.columns]

    # On ne garde que les documents téléchargés avec succès si la colonne existe
    if "status" in documents.columns:
        documents = documents[
            documents["status"].astype(str).str.upper().eq("SUCCESS")
        ].copy()

    # Harmonisation des clés côté metadata
    metadata_join_cols = [
        "company", "ticker", "isin", "jurisdiction", "strate",
        "doc_type", "doc_subtype", "fiscal_year", "period_type", "source_url"
    ]

    for col in metadata_join_cols:
        if col in documents.columns:
            documents[col] = normalize_join_key(documents[col])

    # Filtrage sur les documents sélectionnés en phase 1
    if use_selected_documents and selection_path.exists():

        selected_documents = pd.read_csv(selection_path)
        selected_documents.columns = [col.strip() for col in selected_documents.columns]

        # On garde uniquement les documents sélectionnés pour téléchargement
        if "selection_status" in selected_documents.columns:
            selected_documents = selected_documents[
                selected_documents["selection_status"]
                .astype(str)
                .str.upper()
                .isin(["SELECTED_FOR_DOWNLOAD", "SELECTED", "KEEP"])
            ].copy()

        # selected_url correspond à source_url côté metadata
        if "selected_url" in selected_documents.columns:
            selected_documents["source_url"] = selected_documents["selected_url"]

        selection_join_cols = [
            "company", "ticker", "isin", "jurisdiction", "strate",
            "doc_type", "doc_subtype", "fiscal_year", "period_type", "source_url"
        ]

        for col in selection_join_cols:
            if col in selected_documents.columns:
                selected_documents[col] = normalize_join_key(selected_documents[col])

        merge_keys = [
            col for col in selection_join_cols
            if col in documents.columns and col in selected_documents.columns
        ]

        if merge_keys:
            selected_subset = (
                selected_documents[merge_keys + [
                    col for col in [
                        "score", "confidence_level", "estimated_probability",
                        "selection_status", "selection_reason"
                    ]
                    if col in selected_documents.columns
                ]]
                .drop_duplicates(subset=merge_keys)
            )

            documents = documents.merge(
                selected_subset,
                on=merge_keys,
                how="inner"
            )

    # Dans ton fichier metadata, le chemin local est stocké dans local_path
    path_col = "local_path"

    if path_col not in documents.columns:
        raise ValueError(
            "Colonne local_path absente de documents_metadata.csv. "
            f"Colonnes disponibles : {documents.columns.tolist()}"
        )

    def resolve_pdf_path(path_value):
        if pd.isna(path_value):
            return None

        path_value = str(path_value).strip()
        path = Path(path_value)

        if path.exists():
            return str(path)

        candidate = raw_dir / path.name
        if candidate.exists():
            return str(candidate)

        return str(path)

    documents["source_pdf_path"] = documents[path_col].apply(resolve_pdf_path)

    # On conserve uniquement les PDFs réellement présents localement
    documents["pdf_exists"] = documents["source_pdf_path"].apply(
        lambda p: Path(p).exists() if p else False
    )

    documents = documents[documents["pdf_exists"]].copy()

    # Hash documentaire : on utilise celui de la phase 1 si présent, sinon on le recalcule
    if "sha256" not in documents.columns:
        documents["sha256"] = documents["source_pdf_path"].apply(compute_file_sha256)
    else:
        missing_hash = documents["sha256"].isna() | documents["sha256"].astype(str).str.strip().eq("")
        documents.loc[missing_hash, "sha256"] = documents.loc[
            missing_hash, "source_pdf_path"
        ].apply(compute_file_sha256)

    # Création d'un identifiant documentaire stable
    documents["document_id"] = documents["sha256"].apply(
        lambda x: make_stable_id(x, prefix="doc")
    )

    # Colonnes attendues
    default_columns = {
        "company": None,
        "ticker": None,
        "isin": None,
        "jurisdiction": None,
        "strate": None,
        "doc_type": None,
        "doc_subtype": None,
        "fiscal_year": None,
        "period_type": None,
        "source_url": None,
        "local_path": None,
        "file_size_bytes": None,
        "retrieval_date": None,
        "selection_status": None,
        "confidence_level": None,
        "estimated_probability": None,
        "score": None,
        "selection_reason": None,
    }

    for col, default_value in default_columns.items():
        if col not in documents.columns:
            documents[col] = default_value

    # Déduplication documentaire
    documents = documents.drop_duplicates(subset=["document_id"]).copy()

    # Réorganisation des colonnes principales
    priority_columns = [
        "document_id", "company", "ticker", "isin", "jurisdiction",
        "strate", "doc_type", "doc_subtype", "fiscal_year", "period_type",
        "source_pdf_path", "local_path", "sha256", "file_size_bytes",
        "source_url", "selection_status", "confidence_level",
        "estimated_probability", "score", "selection_reason"
    ]

    remaining_columns = [
        col for col in documents.columns
        if col not in priority_columns
    ]

    documents = documents[priority_columns + remaining_columns]

    return documents


phase2_documents = load_phase1_documents()

print("Documents Phase 2 chargés.")
print(f"Nombre de PDFs disponibles pour extraction : {len(phase2_documents)}")

display(phase2_documents.head())

Documents Phase 2 chargés.
Nombre de PDFs disponibles pour extraction : 118


,document_id,company,ticker,isin,jurisdiction,strate,doc_type,doc_subtype,fiscal_year,period_type,...,source_url,selection_status,confidence_level,estimated_probability,score,selection_reason,input_mode,retrieval_date,status,pdf_exists
0,doc_44af52056b84b450,AXA,CS,FR0000120628,France,1,regulatory,annual_report_urd,2023,FY,...,https://www-axa-com.cdn.axa-contento-118412.eu...,SELECTED_FOR_DOWNLOAD,HIGH_CONFIDENCE,0.996,37,Best candidate passed automatic download thres...,company_name,2026-05-10T08:32:39.305963+00:00,SUCCESS,True
1,doc_038e6d3128b3014b,AXA,CS,FR0000120628,France,4,external_source,assurance_report,2023,FY,...,https://www.bankmandiri.co.id/documents/382688...,SELECTED_FOR_DOWNLOAD,HIGH_CONFIDENCE,0.858,19,Best candidate passed automatic download thres...,company_name,2026-05-10T08:32:40.425759+00:00,SUCCESS,True
2,doc_409d340cd4f6b397,AXA,CS,FR0000120628,France,1,regulatory,climate_report_tcfd_transition_plan,2023,FY,...,https://library.mizuhogroup.com/m/3712bf45ac9d...,SELECTED_FOR_DOWNLOAD,HIGH_CONFIDENCE,0.978,29,Best candidate passed automatic download thres...,company_name,2026-05-10T08:32:40.876141+00:00,SUCCESS,True
3,doc_4caf34e4800c704d,AXA,CS,FR0000120628,France,1,regulatory,sustainability_statement_csrd_esrs,2023,FY,...,https://www.tweuus.nl/wp-content/uploads/2023/...,SELECTED_FOR_DOWNLOAD,HIGH_CONFIDENCE,0.993,35,Best candidate passed automatic download thres...,company_name,2026-05-10T08:32:41.440579+00:00,SUCCESS,True
4,doc_22d50f3ba3d319fe,AXA,CS,FR0000120628,France,1,regulatory,vigilance_plan,2023,FY,...,https://assets.kpmg.com/content/dam/kpmgsites/...,SELECTED_FOR_DOWNLOAD,HIGH_CONFIDENCE,0.973,28,Best candidate passed automatic download thres...,company_name,2026-05-10T08:32:42.840086+00:00,SUCCESS,True


## 6. Reset des sorties de la phase 2

Cette cellule permet de réinitialiser complètement les sorties générées par la phase 2.

Elle supprime :
- les fichiers tabulaires produits par le parsing et l’extraction ;
- les logs d’exécution ;
- les images extraites des PDFs ;
- les dossiers devenus vides après suppression.

Ce reset est utile lorsque :
- le pipeline doit être relancé proprement ;
- une erreur de parsing a été corrigée ;
- la structure des sorties a évolué ;
- un nouveau `RUN_ID` doit être généré sans conserver les anciens artefacts.

Les documents collectés en phase 1 (`raw/`, métadonnées et sélection documentaire) ne sont pas supprimés.

In [ ]:
# ============================================================
# RESET COMPLET — PHASE 2 EXTRACTION
# Supprime toutes les sorties générées en phase 2
# ============================================================

PHASE2_OUTPUT_FILES = [
    PARSED_PAGES_PATH,
    PARSED_BLOCKS_PATH,
    EXTRACTED_TABLES_PATH,
    METRIC_CANDIDATES_PATH,
    VALIDATED_METRICS_PATH,
    REVIEW_QUEUE_PATH,
    FINAL_PANEL_PATH,
    PHASE2_LOG_PATH,
]

# ============================================================
# 1. Suppression des fichiers tabulaires Phase 2
# ============================================================

for file_path in PHASE2_OUTPUT_FILES:

    file_path = Path(file_path)

    # Suppression version parquet
    if file_path.with_suffix(".parquet").exists():
        file_path.with_suffix(".parquet").unlink()
        print(f"Parquet supprimé : {file_path.with_suffix('.parquet')}")

    # Suppression version csv
    if file_path.with_suffix(".csv").exists():
        file_path.with_suffix(".csv").unlink()
        print(f"CSV supprimé : {file_path.with_suffix('.csv')}")


# ============================================================
# 2. Suppression des images extraites
# ============================================================

if IMAGES_DIR.exists():

    for image_file in IMAGES_DIR.rglob("*"):

        if image_file.is_file():
            image_file.unlink()
            print(f"Image supprimée : {image_file}")


# ============================================================
# 3. Suppression des dossiers vides Phase 2
# ============================================================

PHASE2_DIRS = [
    IMAGES_DIR,
    TABLES_DIR,
    BLOCKS_DIR,
    PAGES_DIR,
    CANDIDATES_DIR,
    VALIDATED_DIR,
    REVIEW_DIR,
    PANEL_DIR,
    PHASE2_LOG_DIR,
]

for base_dir in PHASE2_DIRS:

    if base_dir.exists():

        for folder in sorted(base_dir.rglob("*"), reverse=True):

            if folder.is_dir() and not any(folder.iterdir()):
                folder.rmdir()
                print(f"Dossier vide supprimé : {folder}")


print("Reset Phase 2 terminé.")

## 7. Registre d'indicateurs ESG

La collecte de la phase 1 était pilotée par un registre documentaire.  
La phase 2 suit la même logique : l’extraction est pilotée par un registre d’indicateurs ESG.

Ce registre versionné constitue la référence centrale du pipeline d’extraction. Il définit :
- les métriques ESG à rechercher ;
- leurs alias multilingues ;
- les unités attendues ;
- les types de valeurs ;
- les bornes de validation et règles de cohérence.

Il sert notamment à :
- détecter les métriques ESG dans les textes et tableaux ;
- normaliser les noms d’indicateurs ;
- harmoniser les unités ;
- filtrer les valeurs aberrantes ;
- structurer le panel final entreprise-année.

Plusieurs fonctions accompagnent le registre :
- création automatique d’un registre par défaut si aucun catalogue n’existe encore ;
- sauvegarde versionnée du registre ;
- chargement et contrôle du schéma ;
- préparation des alias et unités pour les recherches textuelles automatiques.

Cette approche permet de rendre l’extraction ESG pilotable, extensible et reproductible, dans la continuité de l’architecture mise en place en phase 1.

In [ ]:
# ============================================================
# 7. REGISTRE D'INDICATEURS ESG
# ============================================================

REGISTRY_DIR = BASE_DIR / "data"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

ESG_METRIC_REGISTRY_PATH = REGISTRY_DIR / "esg_metric_registry_catalog.csv"

ESG_METRIC_REGISTRY_COLUMNS = [
    "metric_name",
    "metric_category",
    "esg_pillar",
    "description",
    "aliases",
    "expected_units",
    "normalized_unit",
    "value_type",
    "min_value",
    "max_value",
    "higher_is_better",
    "priority",
    "active",
    "version"
]


DEFAULT_ESG_METRIC_REGISTRY = [
    {
        "metric_name": "scope_1_ghg_emissions",
        "metric_category": "climate",
        "esg_pillar": "E",
        "description": "Direct greenhouse gas emissions from owned or controlled sources.",
        "aliases": [
            "scope 1", "scope 1 emissions", "scope 1 ghg emissions",
            "émissions scope 1", "émissions directes"
        ],
        "expected_units": ["tCO2e", "ktCO2e", "MtCO2e"],
        "normalized_unit": "tCO2e",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 1,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "scope_2_ghg_emissions",
        "metric_category": "climate",
        "esg_pillar": "E",
        "description": "Indirect greenhouse gas emissions from purchased energy.",
        "aliases": [
            "scope 2", "scope 2 emissions", "scope 2 ghg emissions",
            "émissions scope 2", "émissions indirectes énergie"
        ],
        "expected_units": ["tCO2e", "ktCO2e", "MtCO2e"],
        "normalized_unit": "tCO2e",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 1,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "scope_3_ghg_emissions",
        "metric_category": "climate",
        "esg_pillar": "E",
        "description": "Other indirect greenhouse gas emissions across the value chain.",
        "aliases": [
            "scope 3", "scope 3 emissions", "scope 3 ghg emissions",
            "émissions scope 3", "chaîne de valeur"
        ],
        "expected_units": ["tCO2e", "ktCO2e", "MtCO2e"],
        "normalized_unit": "tCO2e",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 1,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "total_ghg_emissions",
        "metric_category": "climate",
        "esg_pillar": "E",
        "description": "Total greenhouse gas emissions reported by the company.",
        "aliases": [
            "total ghg emissions", "total emissions", "greenhouse gas emissions",
            "émissions totales", "émissions de gaz à effet de serre", "ges"
        ],
        "expected_units": ["tCO2e", "ktCO2e", "MtCO2e"],
        "normalized_unit": "tCO2e",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 1,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "energy_consumption",
        "metric_category": "energy",
        "esg_pillar": "E",
        "description": "Total energy consumption.",
        "aliases": [
            "energy consumption", "total energy consumption",
            "consommation d'énergie", "consommation énergétique"
        ],
        "expected_units": ["MWh", "GWh", "TWh", "TJ"],
        "normalized_unit": "MWh",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 2,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "renewable_energy_share",
        "metric_category": "energy",
        "esg_pillar": "E",
        "description": "Share of renewable energy in total energy consumption.",
        "aliases": [
            "renewable energy share", "renewable electricity", "renewable energy",
            "part d'énergie renouvelable", "électricité renouvelable"
        ],
        "expected_units": ["%", "percent"],
        "normalized_unit": "%",
        "value_type": "percentage",
        "min_value": 0,
        "max_value": 100,
        "higher_is_better": True,
        "priority": 2,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "water_withdrawal",
        "metric_category": "water",
        "esg_pillar": "E",
        "description": "Total water withdrawal.",
        "aliases": [
            "water withdrawal", "water withdrawals", "water consumption",
            "prélèvements d'eau", "consommation d'eau"
        ],
        "expected_units": ["m3", "thousand m3", "million m3", "ML"],
        "normalized_unit": "m3",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 2,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "women_board_share",
        "metric_category": "diversity",
        "esg_pillar": "S",
        "description": "Share of women on the board of directors.",
        "aliases": [
            "women on the board", "female board members", "board gender diversity",
            "femmes au conseil", "part des femmes au conseil", "mixité du conseil"
        ],
        "expected_units": ["%", "percent"],
        "normalized_unit": "%",
        "value_type": "percentage",
        "min_value": 0,
        "max_value": 100,
        "higher_is_better": True,
        "priority": 2,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "employee_turnover_rate",
        "metric_category": "human_capital",
        "esg_pillar": "S",
        "description": "Employee turnover rate.",
        "aliases": [
            "employee turnover", "turnover rate", "staff turnover",
            "taux de rotation", "rotation du personnel"
        ],
        "expected_units": ["%", "percent"],
        "normalized_unit": "%",
        "value_type": "percentage",
        "min_value": 0,
        "max_value": 100,
        "higher_is_better": False,
        "priority": 3,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "lost_time_injury_frequency_rate",
        "metric_category": "health_safety",
        "esg_pillar": "S",
        "description": "Lost time injury frequency rate.",
        "aliases": [
            "ltifr", "lost time injury frequency rate",
            "lost-time injury frequency rate",
            "taux de fréquence des accidents", "accidents avec arrêt"
        ],
        "expected_units": ["rate", "per million hours"],
        "normalized_unit": "rate",
        "value_type": "numeric",
        "min_value": 0,
        "max_value": None,
        "higher_is_better": False,
        "priority": 2,
        "active": True,
        "version": "v1"
    },
    {
        "metric_name": "board_independence_share",
        "metric_category": "governance",
        "esg_pillar": "G",
        "description": "Share of independent directors on the board.",
        "aliases": [
            "board independence", "independent directors", "independent board members",
            "administrateurs indépendants", "indépendance du conseil"
        ],
        "expected_units": ["%", "percent"],
        "normalized_unit": "%",
        "value_type": "percentage",
        "min_value": 0,
        "max_value": 100,
        "higher_is_better": True,
        "priority": 2,
        "active": True,
        "version": "v1"
    }
]


def build_default_esg_metric_registry():
    """
    Construit le registre ESG par défaut sous forme de DataFrame.
    """

    registry = pd.DataFrame(DEFAULT_ESG_METRIC_REGISTRY)

    registry["aliases"] = registry["aliases"].apply(safe_json_dumps)
    registry["expected_units"] = registry["expected_units"].apply(safe_json_dumps)

    registry = ensure_dataframe_schema(registry, ESG_METRIC_REGISTRY_COLUMNS)

    return registry


def save_esg_metric_registry(registry, registry_path=ESG_METRIC_REGISTRY_PATH):
    """
    Sauvegarde le registre ESG dans un fichier CSV versionné.
    """

    registry_path = Path(registry_path)
    registry_path.parent.mkdir(parents=True, exist_ok=True)

    registry = ensure_dataframe_schema(registry, ESG_METRIC_REGISTRY_COLUMNS)
    registry.to_csv(registry_path, index=False)

    return registry_path


def load_esg_metric_registry(registry_path=ESG_METRIC_REGISTRY_PATH):
    """
    Charge le registre ESG depuis le disque.
    """

    registry_path = Path(registry_path)

    if not registry_path.exists():
        raise FileNotFoundError(
            f"Registre ESG introuvable : {registry_path}. "
            "Exécuter d'abord la cellule de création du registre."
        )

    registry = pd.read_csv(registry_path)
    registry.columns = [col.strip() for col in registry.columns]
    registry = ensure_dataframe_schema(registry, ESG_METRIC_REGISTRY_COLUMNS)

    registry["active"] = (
        registry["active"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    registry["higher_is_better"] = (
        registry["higher_is_better"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )

    registry["priority"] = pd.to_numeric(registry["priority"], errors="coerce")
    registry["min_value"] = pd.to_numeric(registry["min_value"], errors="coerce")
    registry["max_value"] = pd.to_numeric(registry["max_value"], errors="coerce")

    registry = registry[registry["active"]].copy()

    return registry


def prepare_metric_registry_for_matching(registry):
    """
    Prépare le registre pour la détection textuelle des métriques ESG.
    """

    registry = registry.copy()

    registry["aliases_list"] = registry["aliases"].apply(
        lambda x: safe_json_loads(x, default=[])
    )

    registry["expected_units_list"] = registry["expected_units"].apply(
        lambda x: safe_json_loads(x, default=[])
    )

    registry["normalized_aliases"] = registry["aliases_list"].apply(
        lambda aliases: [
            normalize_text_for_matching(alias)
            for alias in aliases
        ]
    )

    registry["normalized_metric_name"] = registry["metric_name"].apply(
        normalize_text_for_matching
    )

    registry["n_aliases"] = registry["normalized_aliases"].apply(len)

    return registry


# Création du registre si inexistant, comme dans la logique de la phase 1
if not ESG_METRIC_REGISTRY_PATH.exists():
    esg_metric_registry_raw = build_default_esg_metric_registry()
    save_esg_metric_registry(esg_metric_registry_raw, ESG_METRIC_REGISTRY_PATH)

    print("Registre ESG créé.")
    print(f"Chemin : {ESG_METRIC_REGISTRY_PATH}")
else:
    print("Registre ESG déjà existant.")

# Chargement du registre depuis le fichier de référence
esg_metric_registry = load_esg_metric_registry(ESG_METRIC_REGISTRY_PATH)

# Préparation pour matching textuel
esg_metric_registry = prepare_metric_registry_for_matching(esg_metric_registry)

print("Registre d'indicateurs ESG chargé et préparé.")
print(f"Nombre de métriques actives : {len(esg_metric_registry)}")

display(esg_metric_registry.head())

Registre ESG créé.
Chemin : esg_data/data/esg_metric_registry_catalog.csv
Registre d'indicateurs ESG chargé et préparé.
Nombre de métriques actives : 11


,metric_name,metric_category,esg_pillar,description,aliases,expected_units,normalized_unit,value_type,min_value,max_value,higher_is_better,priority,active,version,aliases_list,expected_units_list,normalized_aliases,normalized_metric_name,n_aliases
0,scope_1_ghg_emissions,climate,E,Direct greenhouse gas emissions from owned or ...,"[""scope 1"", ""scope 1 emissions"", ""scope 1 ghg ...","[""tCO2e"", ""ktCO2e"", ""MtCO2e""]",tCO2e,numeric,0,NaN,False,1,True,v1,"[scope 1, scope 1 emissions, scope 1 ghg emiss...","[tCO2e, ktCO2e, MtCO2e]","[scope 1, scope 1 emissions, scope 1 ghg emiss...",scope_1_ghg_emissions,5
1,scope_2_ghg_emissions,climate,E,Indirect greenhouse gas emissions from purchas...,"[""scope 2"", ""scope 2 emissions"", ""scope 2 ghg ...","[""tCO2e"", ""ktCO2e"", ""MtCO2e""]",tCO2e,numeric,0,NaN,False,1,True,v1,"[scope 2, scope 2 emissions, scope 2 ghg emiss...","[tCO2e, ktCO2e, MtCO2e]","[scope 2, scope 2 emissions, scope 2 ghg emiss...",scope_2_ghg_emissions,5
2,scope_3_ghg_emissions,climate,E,Other indirect greenhouse gas emissions across...,"[""scope 3"", ""scope 3 emissions"", ""scope 3 ghg ...","[""tCO2e"", ""ktCO2e"", ""MtCO2e""]",tCO2e,numeric,0,NaN,False,1,True,v1,"[scope 3, scope 3 emissions, scope 3 ghg emiss...","[tCO2e, ktCO2e, MtCO2e]","[scope 3, scope 3 emissions, scope 3 ghg emiss...",scope_3_ghg_emissions,5
3,total_ghg_emissions,climate,E,Total greenhouse gas emissions reported by the...,"[""total ghg emissions"", ""total emissions"", ""gr...","[""tCO2e"", ""ktCO2e"", ""MtCO2e""]",tCO2e,numeric,0,NaN,False,1,True,v1,"[total ghg emissions, total emissions, greenho...","[tCO2e, ktCO2e, MtCO2e]","[total ghg emissions, total emissions, greenho...",total_ghg_emissions,6
4,energy_consumption,energy,E,Total energy consumption.,"[""energy consumption"", ""total energy consumpti...","[""MWh"", ""GWh"", ""TWh"", ""TJ""]",MWh,numeric,0,NaN,False,2,True,v1,"[energy consumption, total energy consumption,...","[MWh, GWh, TWh, TJ]","[energy consumption, total energy consumption,...",energy_consumption,4


## 8. Routing ESG des pages et blocs

Avant d’extraire des métriques ESG, le pipeline commence par identifier les pages et blocs documentaires potentiellement pertinents.

Cette étape de routing permet de :
- réduire le bruit documentaire ;
- éviter de parser inutilement des sections hors ESG ;
- accélérer les étapes d’extraction ;
- documenter pourquoi une page ou un bloc a été considéré comme pertinent.

Le routing repose sur une approche explicable et contrôlable :
- dictionnaires de mots-clés ESG par thème ;
- scores de pertinence par catégorie ;
- attribution de labels multi-thématiques ;
- agrégation simple des signaux textuels.

Cette étape prépare ainsi les futures couches d’extraction en orientant le pipeline vers les zones documentaires les plus informatives.

In [ ]:
# ============================================================
# 8. ROUTING ESG DES PAGES ET BLOCS
# ============================================================

ESG_ROUTING_KEYWORDS = {
    "climate": [
        "climate", "carbon", "co2", "co2e", "ghg", "greenhouse gas",
        "scope 1", "scope 2", "scope 3", "emissions",
        "net zero", "decarbonization", "transition plan",
        "climat", "carbone", "émissions", "gaz à effet de serre",
        "neutralité carbone", "transition climatique"
    ],
    "energy": [
        "energy", "electricity", "renewable", "fuel", "power consumption",
        "énergie", "électricité", "renouvelable", "consommation énergétique"
    ],
    "water": [
        "water", "withdrawal", "discharge", "wastewater",
        "eau", "prélèvement", "consommation d'eau", "rejet d'eau"
    ],
    "waste": [
        "waste", "recycling", "hazardous waste", "circular economy",
        "déchets", "recyclage", "économie circulaire"
    ],
    "biodiversity": [
        "biodiversity", "ecosystem", "nature", "land use",
        "biodiversité", "écosystème", "nature", "usage des sols"
    ],
    "human_capital": [
        "employees", "workforce", "training", "turnover", "headcount",
        "salariés", "effectifs", "formation", "rotation du personnel"
    ],
    "health_safety": [
        "health and safety", "safety", "injury", "accident", "ltifr",
        "santé sécurité", "sécurité", "accident", "taux de fréquence"
    ],
    "diversity": [
        "diversity", "inclusion", "women", "gender", "equal opportunity",
        "diversité", "inclusion", "femmes", "égalité professionnelle", "mixité"
    ],
    "governance": [
        "governance", "board", "independent directors", "ethics",
        "compliance", "anti-corruption", "audit committee",
        "gouvernance", "conseil d'administration", "administrateurs indépendants",
        "éthique", "conformité", "anticorruption"
    ],
    "sustainable_investment": [
        "sustainable investment", "responsible investment", "esg integration",
        "taxonomy", "sfdr", "green bond",
        "investissement durable", "investissement responsable",
        "intégration esg", "taxonomie", "obligation verte"
    ]
}


def build_routing_dictionary(metric_registry=None):
    """
    Construit le dictionnaire de routing ESG à partir :
    - de mots-clés thématiques fixes ;
    - des alias présents dans le registre d'indicateurs ESG.
    """

    routing_dict = {
        category: set(
            normalize_text_for_matching(keyword)
            for keyword in keywords
        )
        for category, keywords in ESG_ROUTING_KEYWORDS.items()
    }

    if metric_registry is not None and len(metric_registry) > 0:
        for _, row in metric_registry.iterrows():
            category = row.get("metric_category", None)

            if pd.isna(category):
                continue

            category = str(category).strip()

            if category not in routing_dict:
                routing_dict[category] = set()

            aliases = row.get("normalized_aliases", [])

            if not isinstance(aliases, list):
                aliases = safe_json_loads(aliases, default=[])

            for alias in aliases:
                alias = normalize_text_for_matching(alias)
                if alias:
                    routing_dict[category].add(alias)

    routing_dict = {
        category: sorted(list(keywords))
        for category, keywords in routing_dict.items()
    }

    return routing_dict


def score_text_against_keywords(text, keywords):
    """
    Calcule un score simple de présence de mots-clés dans un texte.
    """

    normalized_text = normalize_text_for_matching(text)

    if not normalized_text:
        return 0, []

    matched_keywords = []

    for keyword in keywords:
        keyword_norm = normalize_text_for_matching(keyword)

        if not keyword_norm:
            continue

        pattern = r"\b" + re.escape(keyword_norm) + r"\b"

        if re.search(pattern, normalized_text):
            matched_keywords.append(keyword)

    score = len(set(matched_keywords))

    return score, sorted(list(set(matched_keywords)))


def route_text_to_esg_sections(
    text,
    routing_dict,
    min_score=1,
    max_labels=None
):
    """
    Attribue un ou plusieurs labels ESG à un texte.
    """

    section_scores = {}
    section_matches = {}

    for section, keywords in routing_dict.items():
        score, matches = score_text_against_keywords(text, keywords)

        if score >= min_score:
            section_scores[section] = score
            section_matches[section] = matches

    sorted_sections = sorted(
        section_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    if max_labels is not None:
        sorted_sections = sorted_sections[:max_labels]

    section_labels = [section for section, _ in sorted_sections]
    section_scores = {section: score for section, score in sorted_sections}

    return {
        "section_labels": section_labels,
        "section_scores": section_scores,
        "section_matches": section_matches,
        "is_esg_relevant": len(section_labels) > 0,
    }


def compute_esg_relevance_score(section_scores):
    """
    Agrège les scores thématiques en un score global de pertinence ESG.
    """

    if not section_scores:
        return 0.0

    values = list(section_scores.values())

    raw_score = sum(values)

    normalized_score = min(1.0, raw_score / 10)

    return round(normalized_score, 4)


def route_parsed_pages(
    parsed_pages,
    routing_dict,
    min_score=1,
    max_labels=4
):
    """
    Applique le routing ESG à une table de pages parsées.
    """

    routed_pages = parsed_pages.copy()

    routing_outputs = routed_pages["text"].apply(
        lambda text: route_text_to_esg_sections(
            text=text,
            routing_dict=routing_dict,
            min_score=min_score,
            max_labels=max_labels
        )
    )

    routed_pages["section_labels"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_labels"])
    )

    routed_pages["section_scores"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_scores"])
    )

    routed_pages["section_matches"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_matches"])
    )

    routed_pages["is_esg_relevant"] = routing_outputs.apply(
        lambda x: x["is_esg_relevant"]
    )

    routed_pages["esg_relevance_score"] = routing_outputs.apply(
        lambda x: compute_esg_relevance_score(x["section_scores"])
    )

    return routed_pages


def route_document_blocks(
    document_blocks,
    routing_dict,
    min_score=1,
    max_labels=4
):
    """
    Applique le routing ESG à une table de blocs documentaires.
    """

    routed_blocks = document_blocks.copy()

    routing_outputs = routed_blocks["content"].apply(
        lambda text: route_text_to_esg_sections(
            text=text,
            routing_dict=routing_dict,
            min_score=min_score,
            max_labels=max_labels
        )
    )

    routed_blocks["section_labels"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_labels"])
    )

    routed_blocks["section_scores"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_scores"])
    )

    routed_blocks["section_matches"] = routing_outputs.apply(
        lambda x: safe_json_dumps(x["section_matches"])
    )

    routed_blocks["is_esg_relevant"] = routing_outputs.apply(
        lambda x: x["is_esg_relevant"]
    )

    routed_blocks["esg_relevance_score"] = routing_outputs.apply(
        lambda x: compute_esg_relevance_score(x["section_scores"])
    )

    return routed_blocks


routing_dict = build_routing_dictionary(esg_metric_registry)

print("Dictionnaire de routing ESG construit.")
print(f"Nombre de catégories ESG : {len(routing_dict)}")

for category, keywords in routing_dict.items():
    print(f"- {category}: {len(keywords)} mots-clés / alias")

Dictionnaire de routing ESG construit.
Nombre de catégories ESG : 10
- climate: 37 mots-clés / alias
- energy: 17 mots-clés / alias
- water: 12 mots-clés / alias
- waste: 7 mots-clés / alias
- biodiversity: 7 mots-clés / alias
- human_capital: 13 mots-clés / alias
- health_safety: 12 mots-clés / alias
- diversity: 15 mots-clés / alias
- governance: 16 mots-clés / alias
- sustainable_investment: 11 mots-clés / alias


## 9. Parsing multimodal des PDFs : texte, blocs, tables et images

Cette cellule constitue le cœur de la phase 2.

À partir des documents chargés depuis la phase 1, le pipeline extrait plusieurs niveaux d'information :
- le texte page par page ;
- les blocs textuels avec coordonnées ;
- les tableaux détectés dans les pages ;
- les images embarquées dans les PDFs ;
- un OCR est appliqué sur les images lorsque l’environnement le permet.

Le parsing est conçu pour être robuste : une erreur sur un PDF, une page, une table ou une image est enregistrée dans les logs sans interrompre l’ensemble du pipeline.

Les sorties produites alimenteront ensuite le routing ESG, l’extraction de métriques candidates et la construction du panel final.

In [ ]:
# ============================================================
# 9. PARSING MULTIMODAL DES PDFS
# Texte, blocs, tables, images, OCR 
# ============================================================

import traceback

try:
    import fitz  # PyMuPDF
    PYMUPDF_AVAILABLE = True
except ImportError:
    PYMUPDF_AVAILABLE = False

try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
except ImportError:
    PYTESSERACT_AVAILABLE = False


def extract_ocr_from_image(image_path):
    """
    Applique un OCR simple sur une image si pytesseract et PIL sont disponibles.
    """

    if not (PYTESSERACT_AVAILABLE and PIL_AVAILABLE):
        return "", "OCR_NOT_AVAILABLE"

    try:
        image = Image.open(image_path)
        text = pytesseract.image_to_string(image)
        return clean_extracted_text(text), "SUCCESS"
    except Exception as e:
        return "", f"OCR_ERROR_{type(e).__name__}"


def parse_pdf_pages_and_blocks(document_row, run_id=RUN_ID):
    """
    Extrait le texte page par page et les blocs textuels d'un PDF avec PyMuPDF.
    """

    pages = []
    blocks = []

    source_pdf_path = document_row["source_pdf_path"]

    if not PYMUPDF_AVAILABLE:
        raise ImportError("PyMuPDF n'est pas disponible. Installer le package pymupdf.")

    pdf_doc = fitz.open(source_pdf_path)

    for page_index in range(len(pdf_doc)):
        page = pdf_doc[page_index]
        page_number = page_index + 1

        raw_text = page.get_text("text")
        text = clean_extracted_text(raw_text)

        page_dict = page.get_text("dict")
        page_blocks = page_dict.get("blocks", [])

        has_images = any(block.get("type") == 1 for block in page_blocks)
        has_text_blocks = any(block.get("type") == 0 for block in page_blocks)

        page_id = make_stable_id(
            document_row["document_id"],
            page_number,
            prefix="page"
        )

        pages.append({
            "run_id": run_id,
            "document_id": document_row["document_id"],
            "company": document_row.get("company"),
            "ticker": document_row.get("ticker"),
            "isin": document_row.get("isin"),
            "jurisdiction": document_row.get("jurisdiction"),
            "fiscal_year": document_row.get("fiscal_year"),
            "doc_subtype": document_row.get("doc_subtype"),
            "source_pdf_path": source_pdf_path,
            "sha256": document_row.get("sha256"),
            "page_number": page_number,
            "text": text,
            "text_length": len(text),
            "has_tables": None,
            "has_images": has_images,
            "language": document_row.get("language"),
            "section_labels": safe_json_dumps([]),
            "section_scores": safe_json_dumps({}),
            "extraction_quality_score": None,
            "created_at": now_utc_iso()
        })

        block_order = 0

        for block in page_blocks:
            block_type_code = block.get("type")

            if block_type_code != 0:
                continue

            block_text_parts = []

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    block_text_parts.append(span.get("text", ""))

            block_text = clean_extracted_text(" ".join(block_text_parts))

            if not block_text:
                continue

            block_order += 1

            block_id = make_stable_id(
                document_row["document_id"],
                page_number,
                block_order,
                block_text[:100],
                prefix="block"
            )

            blocks.append({
                "run_id": run_id,
                "block_id": block_id,
                "document_id": document_row["document_id"],
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": source_pdf_path,
                "page_number": page_number,
                "block_type": "text",
                "block_order": block_order,
                "content": block_text,
                "bbox": safe_json_dumps(block.get("bbox")),
                "section_labels": safe_json_dumps([]),
                "confidence": 1.0,
                "created_at": now_utc_iso()
            })

    pdf_doc.close()

    return pages, blocks


def parse_pdf_tables(document_row, run_id=RUN_ID):
    """
    Extrait les tableaux d'un PDF avec pdfplumber.
    """

    tables = []

    source_pdf_path = document_row["source_pdf_path"]

    if not PDFPLUMBER_AVAILABLE:
        return tables

    with pdfplumber.open(source_pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages):
            page_number = page_index + 1

            try:
                extracted_tables = page.extract_tables() or []
            except Exception:
                extracted_tables = []

            for table_order, table in enumerate(extracted_tables, start=1):
                if not table:
                    continue

                n_rows = len(table)
                n_cols = max([len(row) for row in table if row] or [0])

                table_id = make_stable_id(
                    document_row["document_id"],
                    page_number,
                    table_order,
                    prefix="table"
                )

                tables.append({
                    "run_id": run_id,
                    "table_id": table_id,
                    "document_id": document_row["document_id"],
                    "company": document_row.get("company"),
                    "fiscal_year": document_row.get("fiscal_year"),
                    "doc_subtype": document_row.get("doc_subtype"),
                    "source_pdf_path": source_pdf_path,
                    "page_number": page_number,
                    "table_order": table_order,
                    "n_rows": n_rows,
                    "n_cols": n_cols,
                    "table_json": safe_json_dumps(table),
                    "extraction_method": "pdfplumber_extract_tables",
                    "created_at": now_utc_iso()
                })

    return tables


def extract_pdf_images(document_row, images_dir=IMAGES_DIR, run_id=RUN_ID, apply_ocr=False):
    """
    Extrait les images embarquées dans un PDF avec PyMuPDF.

    Correction robuste :
    - gère les images JPEG avec canal alpha ;
    - force un fallback PNG si le format source pose problème ;
    - logge les erreurs sans interrompre le parsing global.
    """

    images = []

    source_pdf_path = document_row["source_pdf_path"]

    if not PYMUPDF_AVAILABLE:
        return images

    images_dir = Path(images_dir)
    document_image_dir = images_dir / str(document_row["document_id"])
    document_image_dir.mkdir(parents=True, exist_ok=True)

    pdf_doc = fitz.open(source_pdf_path)

    for page_index in range(len(pdf_doc)):
        page = pdf_doc[page_index]
        page_number = page_index + 1

        image_list = page.get_images(full=True)

        for image_order, image_info in enumerate(image_list, start=1):
            try:
                xref = image_info[0]
                base_image = pdf_doc.extract_image(xref)

                image_bytes = base_image.get("image")
                image_ext = str(base_image.get("ext", "png")).lower()
                width = base_image.get("width")
                height = base_image.get("height")

                image_id = make_stable_id(
                    document_row["document_id"],
                    page_number,
                    image_order,
                    xref,
                    prefix="image"
                )

                # Sécurisation du format
                if image_ext in ["jpeg", "jpg"]:
                    image_ext = "jpg"
                elif image_ext not in ["png", "jpg", "jpeg", "webp"]:
                    image_ext = "png"

                image_path = document_image_dir / f"{image_id}.{image_ext}"

                try:
                    with open(image_path, "wb") as f:
                        f.write(image_bytes)

                    # Vérification que l'image est bien lisible
                    if PIL_AVAILABLE:
                        with Image.open(image_path) as img:
                            img.verify()

                except Exception:
                    # Fallback robuste : reconstruction via Pixmap puis sauvegarde PNG
                    fallback_path = document_image_dir / f"{image_id}.png"

                    pix = fitz.Pixmap(pdf_doc, xref)

                    if pix.alpha:
                        pix_no_alpha = fitz.Pixmap(fitz.csRGB, pix)
                        pix_no_alpha.save(str(fallback_path))
                        pix_no_alpha = None
                    else:
                        pix.save(str(fallback_path))

                    pix = None

                    image_path = fallback_path
                    image_ext = "png"


                if apply_ocr:
                    ocr_text, ocr_status = extract_ocr_from_image(image_path)
                else:
                    ocr_text, ocr_status = "", "OCR_SKIPPED"

                images.append({
                    "run_id": run_id,
                    "image_id": image_id,
                    "document_id": document_row["document_id"],
                    "company": document_row.get("company"),
                    "fiscal_year": document_row.get("fiscal_year"),
                    "doc_subtype": document_row.get("doc_subtype"),
                    "source_pdf_path": source_pdf_path,
                    "page_number": page_number,
                    "image_order": image_order,
                    "image_path": str(image_path),
                    "width": width,
                    "height": height,
                    "ocr_text": ocr_text,
                    "ocr_status": ocr_status,
                    "created_at": now_utc_iso()
                })

            except Exception as e:
                append_log(
                    PHASE2_LOG_PATH,
                    {
                        "run_id": run_id,
                        "stage": "image_extraction",
                        "document_id": document_row.get("document_id"),
                        "company": document_row.get("company"),
                        "fiscal_year": document_row.get("fiscal_year"),
                        "doc_subtype": document_row.get("doc_subtype"),
                        "source_pdf_path": source_pdf_path,
                        "status": "ERROR",
                        "error_type": type(e).__name__,
                        "error_message": str(e),
                        "created_at": now_utc_iso()
                    },
                    columns=EXTRACTION_LOG_COLUMNS
                )

    pdf_doc.close()

    return images


def parse_single_pdf_multimodal(document_row, run_id=RUN_ID, apply_ocr=False):
    """
    Parse un PDF unique : pages, blocs, tables, images.
    """

    parsed_pages = []
    document_blocks = []
    extracted_tables = []
    extracted_images = []

    try:
        pages, blocks = parse_pdf_pages_and_blocks(document_row, run_id=run_id)
        parsed_pages.extend(pages)
        document_blocks.extend(blocks)

        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "pages_blocks_parsing",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "SUCCESS",
                "error_type": None,
                "error_message": None,
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    except Exception as e:
        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "pages_blocks_parsing",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "ERROR",
                "error_type": type(e).__name__,
                "error_message": str(e),
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    try:
        tables = parse_pdf_tables(document_row, run_id=run_id)
        extracted_tables.extend(tables)

        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "table_extraction",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "SUCCESS" if PDFPLUMBER_AVAILABLE else "SKIPPED",
                "error_type": None if PDFPLUMBER_AVAILABLE else "PDFPLUMBER_NOT_AVAILABLE",
                "error_message": None if PDFPLUMBER_AVAILABLE else "pdfplumber non disponible",
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    except Exception as e:
        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "table_extraction",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "ERROR",
                "error_type": type(e).__name__,
                "error_message": str(e),
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    try:
        images = extract_pdf_images(
            document_row,
            images_dir=IMAGES_DIR,
            run_id=run_id,
            apply_ocr=apply_ocr
        )
        extracted_images.extend(images)

        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "image_extraction",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "SUCCESS" if PYMUPDF_AVAILABLE else "SKIPPED",
                "error_type": None if PYMUPDF_AVAILABLE else "PYMUPDF_NOT_AVAILABLE",
                "error_message": None if PYMUPDF_AVAILABLE else "PyMuPDF non disponible",
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    except Exception as e:
        append_log(
            PHASE2_LOG_PATH,
            {
                "run_id": run_id,
                "stage": "image_extraction",
                "document_id": document_row.get("document_id"),
                "company": document_row.get("company"),
                "fiscal_year": document_row.get("fiscal_year"),
                "doc_subtype": document_row.get("doc_subtype"),
                "source_pdf_path": document_row.get("source_pdf_path"),
                "status": "ERROR",
                "error_type": type(e).__name__,
                "error_message": str(e),
                "created_at": now_utc_iso()
            },
            columns=EXTRACTION_LOG_COLUMNS
        )

    return parsed_pages, document_blocks, extracted_tables, extracted_images


def parse_all_phase2_documents(
    phase2_documents,
    run_id=RUN_ID,
    apply_ocr=False, 
    max_documents=None
):
    """
    Parse tous les PDFs disponibles pour la phase 2.
    """

    all_pages = []
    all_blocks = []
    all_tables = []
    all_images = []

    documents_to_parse = phase2_documents.copy()

    if max_documents is not None:
        documents_to_parse = documents_to_parse.head(max_documents).copy()

    print(f"Nombre de documents à parser : {len(documents_to_parse)}")

    for idx, document_row in documents_to_parse.iterrows():
        print(
            f"[{idx + 1}/{len(documents_to_parse)}] "
            f"{document_row.get('company')} | "
            f"{document_row.get('fiscal_year')} | "
            f"{document_row.get('doc_subtype')}"
        )

        pages, blocks, tables, images = parse_single_pdf_multimodal(
            document_row=document_row,
            run_id=run_id,
            apply_ocr=apply_ocr
        )

        all_pages.extend(pages)
        all_blocks.extend(blocks)
        all_tables.extend(tables)
        all_images.extend(images)

    parsed_pages_df = ensure_dataframe_schema(
        pd.DataFrame(all_pages),
        PARSED_PAGE_COLUMNS
    )

    document_blocks_df = ensure_dataframe_schema(
        pd.DataFrame(all_blocks),
        DOCUMENT_BLOCK_COLUMNS
    )

    extracted_tables_df = ensure_dataframe_schema(
        pd.DataFrame(all_tables),
        EXTRACTED_TABLE_COLUMNS
    )

    extracted_images_df = ensure_dataframe_schema(
        pd.DataFrame(all_images),
        EXTRACTED_IMAGE_COLUMNS
    )

    return parsed_pages_df, document_blocks_df, extracted_tables_df, extracted_images_df


parsed_pages_df, document_blocks_df, extracted_tables_df, extracted_images_df = parse_all_phase2_documents(
    phase2_documents=phase2_documents,
    run_id=RUN_ID,
    apply_ocr=False,
    max_documents=None
)

write_dataframe(parsed_pages_df, PAGES_DIR / "parsed_pages.parquet")
write_dataframe(document_blocks_df, BLOCKS_DIR / "document_blocks.parquet")
write_dataframe(extracted_tables_df, TABLES_DIR / "extracted_tables.parquet")
write_dataframe(extracted_images_df, IMAGES_DIR / "extracted_images.parquet")

print("Parsing multimodal terminé.")
print(f"Pages extraites : {len(parsed_pages_df)}")
print(f"Blocs extraits : {len(document_blocks_df)}")
print(f"Tables extraites : {len(extracted_tables_df)}")
print(f"Images extraites : {len(extracted_images_df)}")

Nombre de documents à parser : 118
[1/118] AXA | 2023 | annual_report_urd


/tmp/ipykernel_139103/965948883.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


[2/118] AXA | 2023 | assurance_report
[3/118] AXA | 2023 | climate_report_tcfd_transition_plan
[4/118] AXA | 2023 | sustainability_statement_csrd_esrs
[5/118] AXA | 2023 | vigilance_plan
[6/118] AXA | 2024 | annual_report_urd
[7/118] AXA | 2024 | assurance_report
[8/118] AXA | 2024 | climate_report_tcfd_transition_plan
[9/118] AXA | 2024 | vigilance_plan
[10/118] Accor | 2023 | annual_report_urd
[11/118] Accor | 2023 | vigilance_plan
[12/118] Accor | 2024 | annual_report_urd
[13/118] Accor | 2024 | climate_report_tcfd_transition_plan
[14/118] Accor | 2024 | vigilance_plan
[15/118] Air Liquide | 2023 | annual_report_urd
[16/118] Air Liquide | 2023 | assurance_report
[17/118] Air Liquide | 2023 | sustainability_statement_csrd_esrs
[18/118] Air Liquide | 2023 | vigilance_plan
[19/118] Air Liquide | 2024 | climate_report_tcfd_transition_plan
[20/118] Air Liquide | 2024 | sustainability_statement_csrd_esrs
[21/118] Air Liquide | 2024 | vigilance_plan
[22/118] Airbus | 2023 | sustainability_

Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot 

[77/118] L'Oreal | 2024 | assurance_report
[78/118] L'Oreal | 2024 | sustainability_statement_csrd_esrs
[79/118] L'Oreal | 2024 | vigilance_plan
[80/118] LVMH | 2023 | annual_report_urd
[81/118] Legrand | 2024 | assurance_report
[82/118] Legrand | 2024 | climate_report_tcfd_transition_plan
[83/118] Michelin | 2024 | assurance_report
[84/118] Michelin | 2024 | vigilance_plan
[85/118] Orange | 2024 | assurance_report
[86/118] Pernod Ricard | 2024 | annual_report_urd
[87/118] Pernod Ricard | 2024 | vigilance_plan
[88/118] Publicis Groupe | 2023 | annual_report_urd
[89/118] Publicis Groupe | 2023 | assurance_report
[90/118] Renault | 2023 | assurance_report
[91/118] Renault | 2023 | vigilance_plan
[92/118] Renault | 2024 | assurance_report
[93/118] Renault | 2024 | climate_report_tcfd_transition_plan
[94/118] STMicroelectronics | 2023 | climate_report_tcfd_transition_plan
[95/118] Safran | 2023 | vigilance_plan
[96/118] Saint-Gobain | 2024 | assurance_report
[97/118] Sanofi | 2024 | assura

Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMY

[101/118] Societe Generale | 2023 | climate_report_tcfd_transition_plan
[102/118] Societe Generale | 2023 | vigilance_plan
[103/118] Societe Generale | 2024 | annual_report_urd
[104/118] Societe Generale | 2024 | assurance_report
[105/118] Stellantis | 2023 | annual_report_urd
[106/118] Stellantis | 2023 | sustainability_statement_csrd_esrs
[107/118] Thales | 2023 | assurance_report
[108/118] Thales | 2023 | climate_report_tcfd_transition_plan
[109/118] TotalEnergies | 2023 | annual_report_urd
[110/118] TotalEnergies | 2023 | assurance_report
[111/118] TotalEnergies | 2024 | assurance_report
[112/118] TotalEnergies | 2024 | climate_report_tcfd_transition_plan
[113/118] Unibail-Rodamco-Westfield | 2024 | assurance_report
[114/118] Veolia | 2024 | annual_report_urd
[115/118] Vinci | 2024 | assurance_report
[116/118] Vinci | 2024 | sustainability_statement_csrd_esrs
[117/118] Vinci | 2024 | vigilance_plan
[118/118] Unibail-Rodamco-Westfield | 2023 | agm_minutes_resolutions
Parsing multimo

## 10. Normalisation des valeurs et unités

Avant de valider les métriques ESG extraites, les valeurs brutes doivent être standardisées.

Cette cellule définit les fonctions de normalisation utilisées par le moteur d’extraction :
- conversion des nombres au format français ou anglais ;
- gestion des pourcentages ;
- harmonisation des unités carbone : `tCO2e`, `ktCO2e`, `MtCO2e` ;
- harmonisation des unités énergie : `kWh`, `MWh`, `GWh`, `TWh`, `TJ` ;
- détection des années de référence ou années cibles ;
- contrôle simple des valeurs numériques.

Cette étape permet de comparer les données entre entreprises, documents et années, même lorsque les rapports utilisent des formats différents.

In [16]:
# ============================================================
# 10. NORMALISATION DES VALEURS ET UNITÉS
# ============================================================

UNIT_ALIASES = {
    "tCO2e": [
        "tco2e", "t co2e", "tonnes co2e", "tonnes of co2e",
        "tons co2e", "t eq co2", "teqco2", "t éq. co2", "t éq co2"
    ],
    "ktCO2e": [
        "ktco2e", "kt co2e", "kilotonnes co2e", "thousand tco2e",
        "thousand tonnes co2e", "milliers de tco2e", "kt éq co2"
    ],
    "MtCO2e": [
        "mtco2e", "mt co2e", "million tonnes co2e",
        "millions of tonnes co2e", "millions de tonnes co2e",
        "million tco2e", "mt éq co2"
    ],
    "%": [
        "%", "percent", "percentage", "pourcentage", "pct"
    ],
    "kWh": ["kwh"],
    "MWh": ["mwh"],
    "GWh": ["gwh"],
    "TWh": ["twh"],
    "TJ": ["tj", "terajoules", "térajoules"],
    "m3": ["m3", "m³", "cubic meters", "cubic metres", "mètres cubes"],
}


UNIT_CONVERSION_FACTORS = {
    # Carbone vers tCO2e
    ("tCO2e", "tCO2e"): 1,
    ("ktCO2e", "tCO2e"): 1_000,
    ("MtCO2e", "tCO2e"): 1_000_000,

    # Énergie vers MWh
    ("kWh", "MWh"): 1 / 1_000,
    ("MWh", "MWh"): 1,
    ("GWh", "MWh"): 1_000,
    ("TWh", "MWh"): 1_000_000,
    ("TJ", "MWh"): 277.7777778,

    # Eau vers m3
    ("m3", "m3"): 1,

    # Pourcentage
    ("%", "%"): 1,
}


def normalize_raw_number(value):
    """
    Convertit une valeur brute en nombre flottant.

    Gère les formats :
    - français : 1 234,56
    - anglais : 1,234.56
    - espaces insécables
    - signes +/-
    - pourcentages
    """

    if value is None or pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    value = value.replace("\xa0", " ")
    value = value.replace("\u202f", " ")
    value = value.replace("%", "")
    value = value.replace("+", "")
    value = value.strip()

    # Garder uniquement chiffres, séparateurs, signe négatif
    value = re.sub(r"[^0-9,\.\-\s]", "", value)
    value = value.strip()

    if value == "":
        return None

    # Suppression des espaces de milliers
    value = value.replace(" ", "")

    # Cas français : 1.234,56 ou 1234,56
    if "," in value and "." in value:
        if value.rfind(",") > value.rfind("."):
            value = value.replace(".", "")
            value = value.replace(",", ".")
        else:
            value = value.replace(",", "")

    elif "," in value and "." not in value:
        # Si une seule virgule, on la traite comme séparateur décimal
        value = value.replace(",", ".")

    try:
        return float(value)
    except ValueError:
        return None


def normalize_unit(raw_unit):
    """
    Normalise une unité brute vers une unité canonique.
    """

    if raw_unit is None or pd.isna(raw_unit):
        return None

    unit = normalize_text_for_matching(str(raw_unit))

    if unit == "":
        return None

    for canonical_unit, aliases in UNIT_ALIASES.items():
        normalized_aliases = [
            normalize_text_for_matching(alias)
            for alias in aliases
        ]

        if unit in normalized_aliases:
            return canonical_unit

    return raw_unit


def normalize_metric_value(raw_value, raw_unit, target_unit=None):
    """
    Normalise une valeur et son unité.

    Si target_unit est fourni, la fonction tente une conversion.
    """

    value = normalize_raw_number(raw_value)
    unit = normalize_unit(raw_unit)

    if value is None:
        return None, unit

    if target_unit is None or unit is None:
        return value, unit

    conversion_key = (unit, target_unit)

    if conversion_key in UNIT_CONVERSION_FACTORS:
        value = value * UNIT_CONVERSION_FACTORS[conversion_key]
        return value, target_unit

    return value, unit


def extract_year_from_text(text):
    """
    Détecte les années présentes dans un texte.
    """

    if text is None or pd.isna(text):
        return []

    text = str(text)

    years = re.findall(r"\b(20[0-9]{2}|19[0-9]{2})\b", text)

    years = sorted(list(set(int(year) for year in years)))

    return years


def detect_target_year(text):
    """
    Détecte une année cible dans un texte.

    Exemple :
    - by 2030
    - target 2050
    - objectif 2030
    - horizon 2050
    """

    if text is None or pd.isna(text):
        return None

    normalized_text = normalize_text_for_matching(text)

    patterns = [
        r"\bby\s+(20[0-9]{2})\b",
        r"\btarget\s+(20[0-9]{2})\b",
        r"\bhorizon\s+(20[0-9]{2})\b",
        r"\bobjectif\s+(20[0-9]{2})\b",
        r"\bd'ici\s+(20[0-9]{2})\b",
        r"\bà\s+horizon\s+(20[0-9]{2})\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, normalized_text)
        if match:
            return int(match.group(1))

    return None


def infer_value_type(raw_value, raw_unit=None):
    """
    Infère un type simple de valeur.
    """

    unit = normalize_unit(raw_unit)

    if unit == "%":
        return "percentage"

    value = normalize_raw_number(raw_value)

    if value is None:
        return "text"

    if float(value).is_integer():
        return "integer"

    return "numeric"


def validate_normalized_value(value, min_value=None, max_value=None):
    """
    Contrôle si une valeur normalisée respecte des bornes simples.
    """

    if value is None or pd.isna(value):
        return False

    if min_value is not None and not pd.isna(min_value):
        if value < min_value:
            return False

    if max_value is not None and not pd.isna(max_value):
        if value > max_value:
            return False

    return True


def normalize_extracted_metric_candidate(
    raw_value,
    raw_unit,
    metric_registry_row=None
):
    """
    Normalise une valeur candidate en utilisant éventuellement le registre ESG.
    """

    target_unit = None
    min_value = None
    max_value = None

    if metric_registry_row is not None:
        target_unit = metric_registry_row.get("normalized_unit")
        min_value = metric_registry_row.get("min_value")
        max_value = metric_registry_row.get("max_value")

    normalized_value, normalized_unit = normalize_metric_value(
        raw_value=raw_value,
        raw_unit=raw_unit,
        target_unit=target_unit
    )

    is_valid_range = validate_normalized_value(
        normalized_value,
        min_value=min_value,
        max_value=max_value
    )

    return {
        "raw_value": raw_value,
        "raw_unit": raw_unit,
        "normalized_value": normalized_value,
        "normalized_unit": normalized_unit,
        "value_type": infer_value_type(raw_value, raw_unit),
        "is_valid_range": is_valid_range,
    }


print("Fonctions de normalisation des valeurs et unités chargées.")

Fonctions de normalisation des valeurs et unités chargées.


## 11. Extraction des métriques depuis les tables

Les tableaux sont privilégiés car ils contiennent souvent les indicateurs ESG dans un format plus structuré que le texte libre.

Cette cellule parcourt les tables extraites en phase de parsing, détecte les lignes associées aux métriques du registre ESG, identifie les colonnes correspondant aux années, puis extrait prioritairement la valeur liée à l’exercice fiscal du document.

Chaque valeur extraite est transformée en candidat métrique avec :
- le nom standardisé de la métrique ;
- la valeur brute ;
- l’unité détectée ;
- la valeur normalisée ;
- la page source ;
- un extrait de preuve ;
- une méthode d’extraction ;
- un score de confiance.

In [22]:
# ============================================================
# 11. EXTRACTION ROBUSTE DES MÉTRIQUES DEPUIS LES TABLES
# ============================================================

GENERIC_TABLE_ALIASES_TO_EXCLUDE = {
    "emissions", "total emissions", "analysis", "changes",
    "variation", "increase", "decrease"
}


def table_json_to_rows(table_json):
    table = safe_json_loads(table_json, default=[])

    if not isinstance(table, list):
        return []

    rows = []

    for row in table:
        if row is None:
            continue

        cleaned_row = [
            clean_extracted_text(cell) if cell is not None else ""
            for cell in row
        ]

        if any(cell.strip() for cell in cleaned_row):
            rows.append(cleaned_row)

    return rows


def row_to_text(row):
    return clean_extracted_text(
        " | ".join(str(cell) for cell in row if cell is not None)
    )


def safe_int_year(value):
    try:
        return int(float(value))
    except Exception:
        return None


def remove_non_metric_numbers(text):
    """
    Supprime les nombres appartenant à des libellés non métriques :
    Scope 1/2/3, ESRS E1, Tier 1, numéros de page, horizons temporels, etc.
    """
    if text is None or pd.isna(text):
        return ""

    text = str(text)

    patterns = [
        r"\bscope[s]?\s*1\s*(and|&|,|-|to)?\s*2?\s*(and|&|,|-|to)?\s*3?\b",
        r"\bscope[s]?\s*[123]\b",
        r"\besrs\s*[esg]?\s*\d+\b",
        r"\btier\s*\d+\b",
        r"\bp\.?\s*\d+\b",
        r"\bpage\s*\d+\b",
        r"\bbook\s*\d+\b",
        r"\bnext\s+\d+\s+years?\b",
        r"\b\d+\s+years?\b",
    ]

    cleaned = text

    for pattern in patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    return cleaned


def is_year_like_number(value):
    try:
        value = float(value)
        return value.is_integer() and 1900 <= int(value) <= 2100
    except Exception:
        return False


def is_footnote_like_value(raw_value):
    """
    Détecte les valeurs susceptibles d'être des notes de bas de page :
    (6), 6), note 6, etc.
    """
    if raw_value is None or pd.isna(raw_value):
        return False

    value = str(raw_value).strip()

    patterns = [
        r"^\(?\d+\)?$",
        r"^\(?\d+\)?\s*$",
        r"^note\s*\d+$",
        r"^\[\d+\]$",
    ]

    for pattern in patterns:
        if re.search(pattern, value, flags=re.IGNORECASE):
            return True

    return False


def has_ambiguous_unit_context(row_text):
    """
    Détecte les contextes où l'unité est ambiguë.
    Exemple : m3 (thous L), thous t CO2e, thousand tonnes, etc.
    """
    if row_text is None or pd.isna(row_text):
        return False

    text = normalize_text_for_matching(row_text)

    ambiguous_patterns = [
        r"\bthous\b",
        r"\bthousand\b",
        r"\bthousands\b",
        r"\bmillion\b",
        r"\bmillions\b",
        r"\bthous\s*t\b",
        r"\bthousand\s*t\b",
        r"\bthous\s*l\b",
        r"\bthousand\s*l\b",
    ]

    for pattern in ambiguous_patterns:
        if re.search(pattern, text):
            return True

    return False


def is_plausible_metric_value(cell, raw_unit=None):
    """
    Vérifie si une cellule contient une vraie valeur métrique plausible.
    """
    if cell is None or pd.isna(cell):
        return False

    cell_text = normalize_text_for_matching(cell)

    noise_patterns = [
        r"\bp\.?\s*\d+\b",
        r"\bpage\s*\d+\b",
        r"\bbook\s*\d+\b",
        r"\bnext\s+\d+\s+years?\b",
        r"\b\d+\s+years?\b",
    ]

    for pattern in noise_patterns:
        if re.search(pattern, cell_text):
            return False

    cleaned_cell = remove_non_metric_numbers(cell)
    value = normalize_raw_number(cleaned_cell)

    if value is None:
        return False

    if is_year_like_number(value):
        return False

    if raw_unit is None and abs(value) <= 5:
        return False

    return True


def metric_requires_explicit_unit(metric):
    """
    Indique si une métrique exige une unité explicite dans la ligne.
    """
    value_type = str(metric.get("value_type", "")).lower()
    normalized_unit = metric.get("normalized_unit")

    normalized_unit_missing = (
        normalized_unit is None
        or pd.isna(normalized_unit)
        or str(normalized_unit).strip().lower() in ["", "nan", "none"]
    )

    if value_type in ["numeric", "percentage"] and not normalized_unit_missing:
        return True

    return False


def alias_matches_row(alias, normalized_row_text):
    """
    Matching strict d'un alias dans une ligne.
    """
    alias = normalize_text_for_matching(alias)

    if not alias:
        return False

    if alias in GENERIC_TABLE_ALIASES_TO_EXCLUDE:
        return False

    pattern = r"\b" + re.escape(alias) + r"\b"

    return re.search(pattern, normalized_row_text) is not None


def detect_metric_in_row(row_text, metric_registry):
    """
    Détecte les métriques ESG présentes dans une ligne de table.
    """
    normalized_row_text = normalize_text_for_matching(row_text)
    matches = []

    for _, metric in metric_registry.iterrows():
        aliases = metric.get("normalized_aliases", [])

        if not isinstance(aliases, list):
            aliases = safe_json_loads(aliases, default=[])

        for alias in aliases:
            if alias_matches_row(alias, normalized_row_text):
                matches.append(metric)
                break

    return matches


def detect_year_columns(rows, max_header_rows=8):
    """
    Détecte les colonnes associées aux années dans les premières lignes du tableau.
    """
    year_columns = {}

    for row in rows[:max_header_rows]:
        for col_idx, cell in enumerate(row):
            years = extract_year_from_text(cell)

            for year in years:
                if 2000 <= year <= 2100:
                    year_columns[year] = col_idx

    return year_columns


def infer_unit_from_row(row_text, expected_units=None):
    """
    Détecte l'unité probable d'une ligne.
    """
    normalized_text = normalize_text_for_matching(row_text)

    if expected_units is None:
        expected_units = []

    for unit in expected_units:
        unit_norm = normalize_text_for_matching(unit)

        if unit_norm and re.search(r"\b" + re.escape(unit_norm) + r"\b", normalized_text):
            return unit

    for canonical_unit, aliases in UNIT_ALIASES.items():
        for alias in aliases:
            alias_norm = normalize_text_for_matching(alias)

            if alias_norm and re.search(r"\b" + re.escape(alias_norm) + r"\b", normalized_text):
                return canonical_unit

    return None


def extract_value_from_metric_row(row, fiscal_year, year_columns=None, raw_unit=None):
    """
    Extrait la valeur prioritairement depuis la colonne de l'exercice fiscal.
    """
    fiscal_year = safe_int_year(fiscal_year)

    if year_columns and fiscal_year in year_columns:
        col_idx = year_columns[fiscal_year]

        if col_idx < len(row):
            cleaned_cell = remove_non_metric_numbers(row[col_idx])

            if is_plausible_metric_value(cleaned_cell, raw_unit=raw_unit):
                return cleaned_cell, col_idx, fiscal_year

    candidate_values = []

    for col_idx, cell in enumerate(row):
        cleaned_cell = remove_non_metric_numbers(cell)

        if is_plausible_metric_value(cleaned_cell, raw_unit=raw_unit):
            candidate_values.append((cleaned_cell, col_idx))

    if candidate_values:
        value, col_idx = candidate_values[-1]
        return value, col_idx, fiscal_year

    return None, None, fiscal_year


def build_table_evidence(row, max_chars=800):
    return truncate_text(row_to_text(row), max_chars=max_chars)


def compute_table_candidate_confidence(
    raw_unit,
    normalized,
    source_col_idx,
    reference_year,
    fiscal_year
):
    """
    Score de confiance dynamique et conservateur.
    """
    score = 0.45

    if reference_year is not None and fiscal_year is not None and reference_year == fiscal_year:
        score += 0.15

    if source_col_idx is not None:
        score += 0.10

    if raw_unit is not None:
        score += 0.20

    if normalized.get("is_valid_range"):
        score += 0.10

    return min(round(score, 4), 1.0)


def build_quality_flags(
    raw_value,
    raw_unit,
    normalized,
    source_col_idx,
    reference_year,
    row_text
):
    flags = []

    if raw_unit is None:
        flags.append("UNIT_NOT_DETECTED")

    if source_col_idx is None:
        flags.append("YEAR_COLUMN_NOT_DETECTED")

    if reference_year is None:
        flags.append("REFERENCE_YEAR_NOT_DETECTED")

    if normalize_raw_number(raw_value) is None:
        flags.append("VALUE_NOT_NUMERIC")

    if not normalized.get("is_valid_range"):
        flags.append("VALUE_OUT_OF_EXPECTED_RANGE")

    if is_footnote_like_value(raw_value):
        flags.append("FOOTNOTE_LIKE_VALUE")

    if has_ambiguous_unit_context(row_text):
        flags.append("UNIT_AMBIGUOUS")

    return flags


def extract_metric_candidates_from_table_row(
    table_row,
    table_metadata,
    metric_registry,
    year_columns,
    doc_meta=None,
    row_idx=None
):
    """
    Extrait les candidats métriques ESG depuis une ligne de table.
    """
    candidates = []

    row_text = row_to_text(table_row)
    matched_metrics = detect_metric_in_row(row_text, metric_registry)

    if not matched_metrics:
        return candidates

    fiscal_year = safe_int_year(table_metadata.get("fiscal_year"))

    for metric in matched_metrics:
        expected_units = metric.get("expected_units_list", [])

        raw_unit = infer_unit_from_row(
            row_text=row_text,
            expected_units=expected_units
        )

        raw_value, source_col_idx, reference_year = extract_value_from_metric_row(
            row=table_row,
            fiscal_year=fiscal_year,
            year_columns=year_columns,
            raw_unit=raw_unit
        )

        if raw_value is None:
            continue

        if metric_requires_explicit_unit(metric) and raw_unit is None:
            continue

        normalized = normalize_extracted_metric_candidate(
            raw_value=raw_value,
            raw_unit=raw_unit,
            metric_registry_row=metric
        )

        confidence_score = compute_table_candidate_confidence(
            raw_unit=raw_unit,
            normalized=normalized,
            source_col_idx=source_col_idx,
            reference_year=reference_year,
            fiscal_year=fiscal_year
        )

        quality_flags = build_quality_flags(
            raw_value=raw_value,
            raw_unit=raw_unit,
            normalized=normalized,
            source_col_idx=source_col_idx,
            reference_year=reference_year,
            row_text=row_text
        )

        validation_status = (
            "AUTO_VALIDATED"
            if confidence_score >= 0.90
            and raw_unit is not None
            and normalized.get("is_valid_range")
            and "UNIT_AMBIGUOUS" not in quality_flags
            and "FOOTNOTE_LIKE_VALUE" not in quality_flags
            else "REVIEW_REQUIRED"
        )

        metric_id = make_stable_id(
            table_metadata.get("document_id"),
            table_metadata.get("table_id"),
            row_idx,
            metric.get("metric_name"),
            raw_value,
            prefix="metric"
        )

        candidates.append({
            "run_id": table_metadata.get("run_id"),
            "metric_id": metric_id,
            "document_id": table_metadata.get("document_id"),
            "company": table_metadata.get("company"),
            "ticker": doc_meta.get("ticker") if doc_meta else None,
            "isin": doc_meta.get("isin") if doc_meta else None,
            "jurisdiction": doc_meta.get("jurisdiction") if doc_meta else None,
            "fiscal_year": table_metadata.get("fiscal_year"),
            "doc_subtype": table_metadata.get("doc_subtype"),
            "metric_name": metric.get("metric_name"),
            "metric_category": metric.get("metric_category"),
            "raw_value": raw_value,
            "raw_unit": raw_unit,
            "normalized_value": normalized.get("normalized_value"),
            "normalized_unit": normalized.get("normalized_unit"),
            "source_page": table_metadata.get("page_number"),
            "source_block_id": table_metadata.get("table_id"),
            "source_text_excerpt": build_table_evidence(table_row),
            "extraction_method": "table_registry_year_matching",
            "confidence_score": confidence_score,
            "validation_status": validation_status,
            "quality_flags": safe_json_dumps(quality_flags),
            "created_at": now_utc_iso()
        })

    return candidates


def extract_metric_candidates_from_tables(
    extracted_tables_df,
    metric_registry,
    phase2_documents=None
):
    """
    Extrait les candidats métriques ESG depuis toutes les tables parsées.
    """
    all_candidates = []

    if extracted_tables_df is None or len(extracted_tables_df) == 0:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)

    doc_lookup = {}

    if phase2_documents is not None and len(phase2_documents) > 0:
        doc_lookup = (
            phase2_documents
            .drop_duplicates(subset=["document_id"])
            .set_index("document_id")
            .to_dict("index")
        )

    for _, table_row in extracted_tables_df.iterrows():
        rows = table_json_to_rows(table_row.get("table_json"))

        if not rows:
            continue

        year_columns = detect_year_columns(rows)
        table_metadata = table_row.to_dict()
        doc_meta = doc_lookup.get(table_metadata.get("document_id"), {})

        for row_idx, row in enumerate(rows):
            row_candidates = extract_metric_candidates_from_table_row(
                table_row=row,
                table_metadata=table_metadata,
                metric_registry=metric_registry,
                year_columns=year_columns,
                doc_meta=doc_meta,
                row_idx=row_idx
            )

            all_candidates.extend(row_candidates)

    candidates_df = pd.DataFrame(all_candidates)

    if len(candidates_df) == 0:
        candidates_df = pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    else:
        candidates_df = ensure_dataframe_schema(
            candidates_df,
            METRIC_CANDIDATE_COLUMNS
        )

    return candidates_df


table_metric_candidates_df = extract_metric_candidates_from_tables(
    extracted_tables_df=extracted_tables_df,
    metric_registry=esg_metric_registry,
    phase2_documents=phase2_documents
)

write_dataframe(
    table_metric_candidates_df,
    CANDIDATES_DIR / "table_metric_candidates.parquet"
)

print("Extraction robuste des métriques depuis les tables terminée.")
print(f"Nombre de candidats métriques extraits depuis les tables : {len(table_metric_candidates_df)}")

display(table_metric_candidates_df.head())

/tmp/ipykernel_139103/965948883.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


Extraction robuste des métriques depuis les tables terminée.
Nombre de candidats métriques extraits depuis les tables : 68


,run_id,metric_id,document_id,company,ticker,isin,jurisdiction,fiscal_year,doc_subtype,metric_name,...,normalized_value,normalized_unit,source_page,source_block_id,source_text_excerpt,extraction_method,confidence_score,validation_status,quality_flags,created_at
0,phase2_20260510_130745,metric_6dee86320a74dbf7,doc_4caf34e4800c704d,AXA,CS,FR0000120628,France,2023,sustainability_statement_csrd_esrs,energy_consumption,...,6.0,MWh,91,table_99319ad6285113d2,(6) Total fossil energy consumption (MWh) | |,table_registry_year_matching,1.0,AUTO_VALIDATED,[],2026-05-10T18:03:12Z
1,phase2_20260510_130745,metric_d27d2bfbe1162516,doc_c272d1a4039c4cb5,AXA,CS,FR0000120628,France,2024,assurance_report,scope_1_ghg_emissions,...,24.0,tCO2e,23,table_70dc3f7622ca2821,| Scope 1\n(tCO2e) | Scope 2\n(market-based)\n...,table_registry_year_matching,1.0,AUTO_VALIDATED,[],2026-05-10T18:03:16Z
2,phase2_20260510_130745,metric_23da49cd7b1a3627,doc_c272d1a4039c4cb5,AXA,CS,FR0000120628,France,2024,assurance_report,scope_2_ghg_emissions,...,24.0,tCO2e,23,table_70dc3f7622ca2821,| Scope 1\n(tCO2e) | Scope 2\n(market-based)\n...,table_registry_year_matching,1.0,AUTO_VALIDATED,[],2026-05-10T18:03:16Z
3,phase2_20260510_130745,metric_891c6f6a7f7f0322,doc_650a8b107d86f6ce,Airbus,AIR,NL0000235190,Netherlands,2023,sustainability_statement_csrd_esrs,scope_1_ghg_emissions,...,39.9,tCO2e,80,table_391620509c7ffacb,Total emissions (thous t CO2e) (Scope 1 & 2) |...,table_registry_year_matching,1.0,REVIEW_REQUIRED,"[""UNIT_AMBIGUOUS""]",2026-05-10T18:03:21Z
4,phase2_20260510_130745,metric_79ae108ed15094a4,doc_650a8b107d86f6ce,Airbus,AIR,NL0000235190,Netherlands,2023,sustainability_statement_csrd_esrs,water_withdrawal,...,44412.0,m3,91,table_2dc414ea01fea411,"| | Water consumption m3 (thous L) | 56,886 | ...",table_registry_year_matching,1.0,REVIEW_REQUIRED,"[""FOOTNOTE_LIKE_VALUE"", ""UNIT_AMBIGUOUS""]",2026-05-10T18:03:21Z


## 12. Extraction depuis texte et OCR

Certaines informations ESG importantes ne sont pas toujours contenues dans des tableaux. C’est notamment le cas des objectifs de transition, engagements net zero, politiques de gouvernance, validations externes, cibles climatiques ou commentaires qualitatifs associés aux indicateurs.

Cette cellule complète donc l’extraction tabulaire en exploitant deux sources :
- les blocs de texte extraits directement des PDFs ;
- les textes OCR issus des images, lorsque l’OCR a été activé lors du parsing.

La logique est volontairement prudente :
- recherche des alias du registre ESG dans les blocs ;
- extraction d’une fenêtre de contexte autour de l’alias ;
- détection d’une valeur proche et d’une unité éventuelle ;
- normalisation de la valeur ;
- attribution d’un score de confiance conservateur ;
- envoi en revue des cas ambigus.

Si l’OCR n’a pas été activé, les images restent disponibles mais ne produisent pas de candidats. Le pipeline signale alors clairement que l’extraction OCR est vide car les images ont été marquées `OCR_SKIPPED`.

In [27]:
# ============================================================
# 12. EXTRACTION ROBUSTE DEPUIS TEXTE ET OCR
# ============================================================

CONTEXT_WINDOW_CHARS = 450

TEXT_EXTRACTION_ALIASES_TO_EXCLUDE = {
    "emissions",
    "total emissions",
    "energy",
    "water",
    "governance",
}

TEXT_PERCENT_CONTEXT_KEYWORDS = {
    "share", "rate", "ratio", "percentage", "proportion", "coverage",
    "target", "objective", "reduction", "increase", "decrease",
    "part", "taux", "pourcentage", "proportion", "objectif",
    "réduction", "augmentation", "couverture"
}


def build_context_window(text, start, end, window=CONTEXT_WINDOW_CHARS):
    if text is None or pd.isna(text):
        return ""

    text = clean_extracted_text(text)

    left = max(0, start - window)
    right = min(len(text), end + window)

    return truncate_text(text[left:right], max_chars=2 * window)


def text_alias_matches(alias, normalized_text):
    alias = normalize_text_for_matching(alias)

    if not alias:
        return None

    if alias in TEXT_EXTRACTION_ALIASES_TO_EXCLUDE:
        return None

    pattern = r"\b" + re.escape(alias) + r"\b"

    return re.search(pattern, normalized_text)


def find_metric_matches_in_text(text, metric_registry):
    matches = []

    if text is None or pd.isna(text):
        return matches

    normalized_text = normalize_text_for_matching(text)

    for _, metric in metric_registry.iterrows():
        aliases = metric.get("normalized_aliases", [])

        if not isinstance(aliases, list):
            aliases = safe_json_loads(aliases, default=[])

        for alias in aliases:
            match = text_alias_matches(alias, normalized_text)

            if match:
                matches.append({
                    "metric": metric,
                    "alias": alias,
                    "start": match.start(),
                    "end": match.end()
                })
                break

    return matches


def strict_text_footnote_like_value(raw_value):
    """
    Détecte uniquement les vraies notes :
    (3), [4], note 5.
    """
    if raw_value is None or pd.isna(raw_value):
        return False

    value = str(raw_value).strip()

    patterns = [
        r"^\(\d+\)$",
        r"^\[\d+\]$",
        r"^note\s*\d+$",
    ]

    return any(
        re.search(pattern, value, flags=re.IGNORECASE)
        for pattern in patterns
    )


def context_has_percent_semantics(context):
    normalized_context = normalize_text_for_matching(context)

    return any(
        re.search(r"\b" + re.escape(keyword) + r"\b", normalized_context)
        for keyword in TEXT_PERCENT_CONTEXT_KEYWORDS
    )


def extract_candidate_numbers_from_context(context):
    if context is None or pd.isna(context):
        return []

    cleaned_context = remove_non_metric_numbers(context)

    number_pattern = r"[-+]?\d+(?:[\s\u202f\xa0]?\d{3})*(?:[,.]\d+)?|[-+]?\d+"

    raw_numbers = re.findall(number_pattern, cleaned_context)

    candidates = []

    for raw_number in raw_numbers:
        value = normalize_raw_number(raw_number)

        if value is None:
            continue

        if is_year_like_number(value):
            continue

        if strict_text_footnote_like_value(raw_number):
            continue

        candidates.append((raw_number, value))

    return candidates


def infer_unit_from_context(context, expected_units=None):
    return infer_unit_from_row(
        row_text=context,
        expected_units=expected_units
    )


def extract_value_from_text_context(context, metric):
    value_type = str(metric.get("value_type", "")).lower()
    expected_units = metric.get("expected_units_list", [])

    raw_unit = infer_unit_from_context(
        context=context,
        expected_units=expected_units
    )

    if value_type == "year":
        years = extract_year_from_text(context)
        future_years = [year for year in years if 2025 <= year <= 2100]

        if future_years:
            return str(future_years[0]), "year"

        return None, "year"

    if value_type == "percentage":
        if not context_has_percent_semantics(context):
            return None, "%"

        percent_matches = re.findall(
            r"[-+]?\d+(?:[,.]\d+)?\s*%",
            str(context)
        )

        if percent_matches:
            raw_value = percent_matches[0].replace("%", "").strip()

            if strict_text_footnote_like_value(raw_value):
                return None, "%"

            return raw_value, "%"

    candidates = extract_candidate_numbers_from_context(context)

    if not candidates:
        return None, raw_unit

    raw_value, _ = candidates[0]

    return raw_value, raw_unit


def compute_text_candidate_confidence(method, raw_unit, normalized, context):
    score = 0.45 if method == "text_block" else 0.35

    if raw_unit is not None:
        score += 0.15

    if normalized.get("is_valid_range"):
        score += 0.10

    if has_ambiguous_unit_context(context):
        score -= 0.10

    if raw_unit == "%" and not context_has_percent_semantics(context):
        score -= 0.15

    return max(0.0, min(round(score, 4), 1.0))


def build_text_quality_flags(raw_value, raw_unit, normalized, context, method):
    flags = [f"FROM_{method.upper()}"]

    if raw_unit is None:
        flags.append("UNIT_NOT_DETECTED")

    if normalize_raw_number(raw_value) is None and raw_unit != "year":
        flags.append("VALUE_NOT_NUMERIC")

    if not normalized.get("is_valid_range"):
        flags.append("VALUE_OUT_OF_EXPECTED_RANGE")

    if has_ambiguous_unit_context(context):
        flags.append("UNIT_AMBIGUOUS")

    if strict_text_footnote_like_value(raw_value):
        flags.append("FOOTNOTE_LIKE_VALUE")

    if raw_unit == "%" and not context_has_percent_semantics(context):
        flags.append("WEAK_PERCENT_CONTEXT")

    return flags


def summarize_ocr_availability(extracted_images_df):
    """
    Résume l'état OCR des images extraites.
    """
    if extracted_images_df is None or len(extracted_images_df) == 0:
        print("Aucune image extraite.")
        return 0

    if "ocr_status" not in extracted_images_df.columns:
        print("Colonne ocr_status absente : OCR non disponible dans cette sortie.")
        return 0

    print("Statut OCR des images :")
    display(extracted_images_df["ocr_status"].value_counts(dropna=False))

    n_with_text = (
        extracted_images_df["ocr_text"].notna()
        & extracted_images_df["ocr_text"].astype(str).str.len().gt(20)
    ).sum() if "ocr_text" in extracted_images_df.columns else 0

    print(f"Images avec texte OCR exploitable : {n_with_text}")

    return n_with_text


def has_existing_ocr_text(extracted_images_df, min_chars=20):
    """
    Vérifie si des textes OCR exploitables existent déjà.
    """
    if extracted_images_df is None or len(extracted_images_df) == 0:
        return False

    if "ocr_text" not in extracted_images_df.columns:
        return False

    return (
        extracted_images_df["ocr_text"].notna()
        & extracted_images_df["ocr_text"].astype(str).str.len().gt(min_chars)
    ).any()


def apply_ocr_to_extracted_images(
    extracted_images_df,
    max_images=None,
    overwrite_skipped=True
):
    """
    Applique l'OCR aux images déjà extraites si nécessaire.

    Parameters
    ----------
    extracted_images_df : DataFrame
        Table des images produite lors du parsing multimodal.
    max_images : int ou None
        Limite optionnelle pour tester l'OCR sur un sous-échantillon.
    overwrite_skipped : bool
        Si True, applique l'OCR aux images marquées OCR_SKIPPED.
    """
    if extracted_images_df is None or len(extracted_images_df) == 0:
        return pd.DataFrame(columns=EXTRACTED_IMAGE_COLUMNS)

    images = extracted_images_df.copy()

    if "ocr_text" not in images.columns:
        images["ocr_text"] = ""

    if "ocr_status" not in images.columns:
        images["ocr_status"] = "OCR_NOT_RUN"

    if not (PYTESSERACT_AVAILABLE and PIL_AVAILABLE):
        print("OCR indisponible : PIL ou pytesseract non disponible.")
        images["ocr_status"] = images["ocr_status"].fillna("OCR_NOT_AVAILABLE")
        return images

    if overwrite_skipped:
        mask = (
            images["ocr_status"].isna()
            | images["ocr_status"].astype(str).isin(["", "OCR_SKIPPED", "OCR_NOT_RUN"])
        )
    else:
        mask = (
            images["ocr_text"].isna()
            | images["ocr_text"].astype(str).str.len().le(20)
        )

    images_to_process = images[mask].copy()

    if max_images is not None:
        images_to_process = images_to_process.head(max_images).copy()

    print(f"Images à OCRiser : {len(images_to_process)}")

    for i, (idx, row) in enumerate(images_to_process.iterrows(), start=1):
        image_path = row.get("image_path")

        if i % 250 == 0:
            print(f"OCR progression : {i}/{len(images_to_process)}")

        if not image_path or not Path(image_path).exists():
            images.loc[idx, "ocr_text"] = ""
            images.loc[idx, "ocr_status"] = "IMAGE_FILE_NOT_FOUND"
            continue

        ocr_text, ocr_status = extract_ocr_from_image(image_path)

        images.loc[idx, "ocr_text"] = ocr_text
        images.loc[idx, "ocr_status"] = ocr_status

    return ensure_dataframe_schema(
        images,
        EXTRACTED_IMAGE_COLUMNS
    )


def ensure_ocr_ready_images(
    extracted_images_df,
    run_ocr_if_missing=True,
    max_ocr_images=None,
    save_output=True
):
    """
    Garantit que la table des images contient du texte OCR exploitable si possible.

    Si l'OCR existe déjà, la fonction ne refait rien.
    Si l'OCR n'existe pas et run_ocr_if_missing=True, elle applique l'OCR
    aux images déjà extraites.
    """
    print("Diagnostic OCR avant traitement :")
    n_existing_ocr = summarize_ocr_availability(extracted_images_df)

    if n_existing_ocr > 0:
        print("OCR déjà disponible. Passage direct à l'extraction OCR.")
        return extracted_images_df

    if not run_ocr_if_missing:
        print("OCR absent et run_ocr_if_missing=False. Extraction OCR ignorée.")
        return extracted_images_df

    print("OCR absent ou inexploitable. Application OCR sur les images déjà extraites...")

    images_with_ocr = apply_ocr_to_extracted_images(
        extracted_images_df=extracted_images_df,
        max_images=max_ocr_images,
        overwrite_skipped=True
    )

    print("Diagnostic OCR après traitement :")
    summarize_ocr_availability(images_with_ocr)

    if save_output:
        write_dataframe(
            images_with_ocr,
            IMAGES_DIR / "extracted_images.parquet"
        )

    return images_with_ocr


def build_ocr_blocks(
    extracted_images_df,
    run_id=RUN_ID,
    routing_dict=None
):
    if extracted_images_df is None or len(extracted_images_df) == 0:
        return pd.DataFrame(columns=DOCUMENT_BLOCK_COLUMNS)

    if "ocr_text" not in extracted_images_df.columns:
        return pd.DataFrame(columns=DOCUMENT_BLOCK_COLUMNS)

    rows = []

    ocr_images = extracted_images_df[
        extracted_images_df["ocr_text"].notna()
        & extracted_images_df["ocr_text"].astype(str).str.len().gt(20)
    ].copy()

    for _, image in ocr_images.iterrows():
        text = clean_extracted_text(image.get("ocr_text"))

        if not text:
            continue

        if routing_dict is not None:
            routing_output = route_text_to_esg_sections(
                text=text,
                routing_dict=routing_dict,
                min_score=1,
                max_labels=4
            )
            section_labels = routing_output["section_labels"]
        else:
            section_labels = []

        block_id = make_stable_id(
            image.get("image_id"),
            "ocr",
            text[:100],
            prefix="block"
        )

        rows.append({
            "run_id": run_id,
            "block_id": block_id,
            "document_id": image.get("document_id"),
            "company": image.get("company"),
            "fiscal_year": image.get("fiscal_year"),
            "doc_subtype": image.get("doc_subtype"),
            "source_pdf_path": image.get("source_pdf_path"),
            "page_number": image.get("page_number"),
            "block_type": "ocr_image",
            "block_order": image.get("image_order"),
            "content": text,
            "bbox": None,
            "section_labels": safe_json_dumps(section_labels),
            "confidence": 0.55,
            "created_at": now_utc_iso()
        })

    return ensure_dataframe_schema(
        pd.DataFrame(rows),
        DOCUMENT_BLOCK_COLUMNS
    )


def extract_metric_candidates_from_text_blocks(
    blocks_df,
    metric_registry,
    phase2_documents=None,
    run_id=RUN_ID,
    method="text_block",
    use_only_esg_relevant=True
):
    all_candidates = []

    if blocks_df is None or len(blocks_df) == 0:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)

    doc_lookup = {}

    if phase2_documents is not None and len(phase2_documents) > 0:
        doc_lookup = (
            phase2_documents
            .drop_duplicates(subset=["document_id"])
            .set_index("document_id")
            .to_dict("index")
        )

    blocks = blocks_df.copy()

    if use_only_esg_relevant and "is_esg_relevant" in blocks.columns:
        blocks = blocks[blocks["is_esg_relevant"].astype(bool)].copy()

    for _, block in blocks.iterrows():
        text = clean_extracted_text(block.get("content"))

        if len(text) < 30:
            continue

        matches = find_metric_matches_in_text(
            text=text,
            metric_registry=metric_registry
        )

        if not matches:
            continue

        doc_meta = doc_lookup.get(block.get("document_id"), {})

        for match_info in matches:
            metric = match_info["metric"]

            context = build_context_window(
                text=text,
                start=match_info["start"],
                end=match_info["end"],
                window=CONTEXT_WINDOW_CHARS
            )

            raw_value, raw_unit = extract_value_from_text_context(
                context=context,
                metric=metric
            )

            if raw_value is None:
                continue

            if metric_requires_explicit_unit(metric) and raw_unit is None:
                continue

            normalized = normalize_extracted_metric_candidate(
                raw_value=raw_value,
                raw_unit=raw_unit,
                metric_registry_row=metric
            )

            confidence_score = compute_text_candidate_confidence(
                method=method,
                raw_unit=raw_unit,
                normalized=normalized,
                context=context
            )

            quality_flags = build_text_quality_flags(
                raw_value=raw_value,
                raw_unit=raw_unit,
                normalized=normalized,
                context=context,
                method=method
            )

            validation_status = (
                "AUTO_VALIDATED"
                if confidence_score >= 0.90
                and raw_unit is not None
                and normalized.get("is_valid_range")
                and "UNIT_AMBIGUOUS" not in quality_flags
                and "FOOTNOTE_LIKE_VALUE" not in quality_flags
                and "WEAK_PERCENT_CONTEXT" not in quality_flags
                else "REVIEW_REQUIRED"
            )

            metric_id = make_stable_id(
                block.get("block_id"),
                metric.get("metric_name"),
                raw_value,
                method,
                prefix="metric"
            )

            all_candidates.append({
                "run_id": run_id,
                "metric_id": metric_id,
                "document_id": block.get("document_id"),
                "company": block.get("company"),
                "ticker": doc_meta.get("ticker"),
                "isin": doc_meta.get("isin"),
                "jurisdiction": doc_meta.get("jurisdiction"),
                "fiscal_year": block.get("fiscal_year") or doc_meta.get("fiscal_year"),
                "doc_subtype": block.get("doc_subtype"),
                "metric_name": metric.get("metric_name"),
                "metric_category": metric.get("metric_category"),
                "raw_value": raw_value,
                "raw_unit": raw_unit,
                "normalized_value": normalized.get("normalized_value"),
                "normalized_unit": normalized.get("normalized_unit"),
                "source_page": block.get("page_number"),
                "source_block_id": block.get("block_id"),
                "source_text_excerpt": context,
                "extraction_method": method,
                "confidence_score": confidence_score,
                "validation_status": validation_status,
                "quality_flags": safe_json_dumps(quality_flags),
                "created_at": now_utc_iso()
            })

    candidates_df = pd.DataFrame(all_candidates)

    if len(candidates_df) == 0:
        candidates_df = pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)
    else:
        candidates_df = ensure_dataframe_schema(
            candidates_df,
            METRIC_CANDIDATE_COLUMNS
        )

    return candidates_df


# ============================================================
# ORCHESTRATION TEXTE + OCR
# ============================================================

# 1. S'assurer que l'OCR est disponible si nécessaire
extracted_images_df = ensure_ocr_ready_images(
    extracted_images_df=extracted_images_df,
    run_ocr_if_missing=True,
    max_ocr_images=None,   # mettre un entier pour tester, ex. 200
    save_output=True
)

# 2. Construction des blocs OCR
ocr_blocks_df = build_ocr_blocks(
    extracted_images_df=extracted_images_df,
    run_id=RUN_ID,
    routing_dict=routing_dict
)

# 3. Extraction depuis les blocs texte
text_metric_candidates_df = extract_metric_candidates_from_text_blocks(
    blocks_df=document_blocks_df,
    metric_registry=esg_metric_registry,
    phase2_documents=phase2_documents,
    run_id=RUN_ID,
    method="text_block",
    use_only_esg_relevant=False
)

# 4. Extraction depuis les blocs OCR
ocr_metric_candidates_df = extract_metric_candidates_from_text_blocks(
    blocks_df=ocr_blocks_df,
    metric_registry=esg_metric_registry,
    phase2_documents=phase2_documents,
    run_id=RUN_ID,
    method="ocr_image",
    use_only_esg_relevant=False
)

# 5. Sauvegardes
write_dataframe(
    text_metric_candidates_df,
    CANDIDATES_DIR / "text_metric_candidates.parquet"
)

write_dataframe(
    ocr_metric_candidates_df,
    CANDIDATES_DIR / "ocr_metric_candidates.parquet"
)

write_dataframe(
    ocr_blocks_df,
    BLOCKS_DIR / "ocr_blocks.parquet"
)

print("Extraction depuis texte et OCR terminée.")
print(f"Blocs OCR construits : {len(ocr_blocks_df)}")
print(f"Candidats depuis texte : {len(text_metric_candidates_df)}")
print(f"Candidats depuis OCR : {len(ocr_metric_candidates_df)}")

display(text_metric_candidates_df.head())

Diagnostic OCR avant traitement :
Statut OCR des images :


ocr_status
OCR_SKIPPED    10435
Name: count, dtype: int64

Images avec texte OCR exploitable : 0
OCR absent ou inexploitable. Application OCR sur les images déjà extraites...
Images à OCRiser : 10435
OCR progression : 250/10435
OCR progression : 500/10435
OCR progression : 750/10435
OCR progression : 1000/10435
OCR progression : 1250/10435
OCR progression : 1500/10435
OCR progression : 1750/10435
OCR progression : 2000/10435
OCR progression : 2250/10435
OCR progression : 2500/10435
OCR progression : 2750/10435
OCR progression : 3000/10435
OCR progression : 3250/10435
OCR progression : 3500/10435
OCR progression : 3750/10435
OCR progression : 4000/10435
OCR progression : 4250/10435
OCR progression : 4500/10435
OCR progression : 4750/10435
OCR progression : 5000/10435
OCR progression : 5250/10435
OCR progression : 5500/10435
OCR progression : 5750/10435
OCR progression : 6000/10435
OCR progression : 6250/10435
OCR progression : 6500/10435
OCR progression : 6750/10435
OCR progression : 7000/10435
OCR progression : 7250/10435
OCR progression : 750

ocr_status
OCR_ERROR_TesseractNotFoundError    9822
OCR_ERROR_OSError                    613
Name: count, dtype: int64

Images avec texte OCR exploitable : 0


/tmp/ipykernel_139103/965948883.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


Extraction depuis texte et OCR terminée.
Blocs OCR construits : 0
Candidats depuis texte : 715
Candidats depuis OCR : 0


,run_id,metric_id,document_id,company,ticker,isin,jurisdiction,fiscal_year,doc_subtype,metric_name,...,normalized_value,normalized_unit,source_page,source_block_id,source_text_excerpt,extraction_method,confidence_score,validation_status,quality_flags,created_at
0,phase2_20260510_130745,metric_e8d29dd917fe7fb3,doc_44af52056b84b450,AXA,CS,FR0000120628,France,2023,annual_report_urd,renewable_energy_share,...,3.0,%,7,block_936ce2d5f6b72ac9,"(1) Compounded annual growth rate, 2020-2023 ....",text_block,0.7,REVIEW_REQUIRED,"[""FROM_TEXT_BLOCK""]",2026-05-10T19:48:38Z
1,phase2_20260510_130745,metric_2a485e9de151869d,doc_44af52056b84b450,AXA,CS,FR0000120628,France,2023,annual_report_urd,board_independence_share,...,50.0,%,78,block_c885d5cb36b684e3,At least 50% independent directors (Target met),text_block,0.7,REVIEW_REQUIRED,"[""FROM_TEXT_BLOCK""]",2026-05-10T19:48:40Z
2,phase2_20260510_130745,metric_4c634ed1152a2953,doc_44af52056b84b450,AXA,CS,FR0000120628,France,2023,annual_report_urd,renewable_energy_share,...,90.0,%,148,block_cd6332747975ca43,To support the development of renewable energy...,text_block,0.7,REVIEW_REQUIRED,"[""FROM_TEXT_BLOCK""]",2026-05-10T19:48:41Z
3,phase2_20260510_130745,metric_fc45bddbea95f390,doc_44af52056b84b450,AXA,CS,FR0000120628,France,2023,annual_report_urd,employee_turnover_rate,...,14.5,%,162,block_bfc8c098f7681a7d,Turnover rate of salaried workforce 14.5 % -0....,text_block,0.7,REVIEW_REQUIRED,"[""FROM_TEXT_BLOCK""]",2026-05-10T19:48:42Z
4,phase2_20260510_130745,metric_20f3f00b8e9e2553,doc_44af52056b84b450,AXA,CS,FR0000120628,France,2023,annual_report_urd,scope_2_ghg_emissions,...,2.0,kWh,182,block_4505b4d06cc561c4,Scope 2 GHG emissions The GHG emissions relate...,text_block,0.7,REVIEW_REQUIRED,"[""FROM_TEXT_BLOCK""]",2026-05-10T19:48:42Z


## 13. Validation métier, scoring et revue manuelle

Cette cellule applique une couche de validation métier aux candidats extraits depuis les tables, le texte et l’OCR.

L’objectif n’est pas de supprimer brutalement les candidats imparfaits, mais de les classer selon leur fiabilité :
- candidats validables automatiquement ;
- candidats nécessitant une revue manuelle ;
- candidats rejetés si l’information est inexploitable.

Les contrôles portent notamment sur :
- la présence d’une valeur numérique ;
- la cohérence de l’unité ;
- le respect des bornes définies dans le registre ESG ;
- la présence d’une preuve source ;
- la méthode d’extraction utilisée ;
- les flags qualité déjà produits aux étapes précédentes.

Cette étape rend le pipeline plus défendable : chaque métrique conserve une trace de sa source, de son score et des raisons éventuelles de revue.

In [28]:
# ============================================================
# 13. VALIDATION MÉTIER, SCORING ET REVUE MANUELLE
# ============================================================

VALIDATION_AUTO_THRESHOLD = 0.85
VALIDATION_REVIEW_THRESHOLD = 0.45


def combine_metric_candidate_sources(
    table_candidates_df=None,
    text_candidates_df=None,
    ocr_candidates_df=None
):
    """
    Combine les candidats issus des tables, du texte et de l'OCR.
    """

    candidate_frames = []

    for df in [table_candidates_df, text_candidates_df, ocr_candidates_df]:
        if df is not None and len(df) > 0:
            candidate_frames.append(
                ensure_dataframe_schema(df, METRIC_CANDIDATE_COLUMNS)
            )

    if not candidate_frames:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)

    combined = pd.concat(candidate_frames, ignore_index=True)
    combined = ensure_dataframe_schema(combined, METRIC_CANDIDATE_COLUMNS)

    return combined


def build_metric_registry_lookup(metric_registry):
    """
    Construit un dictionnaire metric_name -> ligne du registre.
    """

    lookup = {}

    for _, row in metric_registry.iterrows():
        metric_name = row.get("metric_name")

        if metric_name is not None and not pd.isna(metric_name):
            lookup[str(metric_name)] = row.to_dict()

    return lookup


def parse_quality_flags(flags):
    """
    Recharge les quality_flags stockés en JSON.
    """

    if flags is None or pd.isna(flags):
        return []

    if isinstance(flags, list):
        return flags

    return safe_json_loads(flags, default=[])


def is_missing_value(value):
    """
    Vérifie si une valeur est absente ou non exploitable.
    """

    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except Exception:
        pass

    if str(value).strip().lower() in ["", "nan", "none"]:
        return True

    return False


def expected_unit_for_metric(metric_spec):
    """
    Récupère l'unité normalisée attendue pour une métrique.
    """

    if metric_spec is None:
        return None

    unit = metric_spec.get("normalized_unit")

    if unit is None or pd.isna(unit):
        return None

    unit = str(unit).strip()

    if unit.lower() in ["", "nan", "none"]:
        return None

    return unit


def validate_unit_consistency(candidate_unit, metric_spec):
    """
    Vérifie la cohérence de l'unité normalisée.
    """

    expected_unit = expected_unit_for_metric(metric_spec)

    if expected_unit is None:
        return True

    if candidate_unit is None or pd.isna(candidate_unit):
        return False

    candidate_unit = str(candidate_unit).strip()

    return candidate_unit == expected_unit


def validate_range_consistency(value, metric_spec):
    """
    Vérifie le respect des bornes définies dans le registre ESG.
    """

    if is_missing_value(value):
        return False

    try:
        value = float(value)
    except Exception:
        return False

    min_value = metric_spec.get("min_value")
    max_value = metric_spec.get("max_value")

    if min_value is not None and not pd.isna(min_value):
        if value < float(min_value):
            return False

    if max_value is not None and not pd.isna(max_value):
        if value > float(max_value):
            return False

    return True


def has_source_evidence(candidate):
    """
    Vérifie la présence d'une preuve source exploitable.
    """

    source_page = candidate.get("source_page")
    excerpt = candidate.get("source_text_excerpt")

    if is_missing_value(source_page):
        return False

    if excerpt is None or pd.isna(excerpt):
        return False

    if len(str(excerpt).strip()) < 10:
        return False

    return True


def score_method_adjustment(extraction_method):
    """
    Ajuste le score selon la méthode d'extraction.
    """

    method = str(extraction_method).lower()

    if "table" in method:
        return 0.05

    if "text" in method:
        return 0.00

    if "ocr" in method:
        return -0.15

    return 0.00


def validate_single_metric_candidate(candidate, registry_lookup):
    """
    Valide un candidat métrique unique.
    """

    candidate = candidate.copy()

    metric_name = candidate.get("metric_name")
    metric_spec = registry_lookup.get(str(metric_name))

    flags = parse_quality_flags(candidate.get("quality_flags"))
    flags = list(flags)

    try:
        score = float(candidate.get("confidence_score") or 0.0)
    except Exception:
        score = 0.0

    status = "AUTO_VALIDATED"

    if metric_spec is None:
        flags.append("UNKNOWN_METRIC")
        score -= 0.25
        status = "REVIEW_REQUIRED"

    value = candidate.get("normalized_value")
    unit = candidate.get("normalized_unit")

    if is_missing_value(value):
        flags.append("MISSING_VALUE")
        score -= 0.35
        status = "REJECTED"

    if metric_spec is not None and not is_missing_value(value):
        if not validate_range_consistency(value, metric_spec):
            flags.append("VALUE_OUT_OF_EXPECTED_RANGE")
            score -= 0.20
            status = "REVIEW_REQUIRED"

        expected_unit = expected_unit_for_metric(metric_spec)

        if expected_unit is not None:
            if not validate_unit_consistency(unit, metric_spec):
                flags.append("UNEXPECTED_OR_MISSING_UNIT")
                score -= 0.20
                status = "REVIEW_REQUIRED"

    if not has_source_evidence(candidate):
        flags.append("MISSING_SOURCE_EVIDENCE")
        score -= 0.20

        if status != "REJECTED":
            status = "REVIEW_REQUIRED"

    method_adjustment = score_method_adjustment(candidate.get("extraction_method"))
    score += method_adjustment

    extraction_method = str(candidate.get("extraction_method")).lower()

    if "ocr" in extraction_method:
        flags.append("OCR_SOURCE")

    critical_review_flags = {
        "UNIT_NOT_DETECTED",
        "UNEXPECTED_OR_MISSING_UNIT",
        "UNIT_AMBIGUOUS",
        "FOOTNOTE_LIKE_VALUE",
        "WEAK_PERCENT_CONTEXT",
        "VALUE_OUT_OF_EXPECTED_RANGE",
        "MISSING_SOURCE_EVIDENCE",
    }

    if any(flag in flags for flag in critical_review_flags):
        if status != "REJECTED":
            status = "REVIEW_REQUIRED"

    score = round(max(0.0, min(1.0, score)), 4)

    if status == "AUTO_VALIDATED" and score < VALIDATION_AUTO_THRESHOLD:
        status = "REVIEW_REQUIRED"
        flags.append("LOW_CONFIDENCE")

    if score < VALIDATION_REVIEW_THRESHOLD and status != "REJECTED":
        status = "REVIEW_REQUIRED"
        flags.append("VERY_LOW_CONFIDENCE")

    candidate["confidence_score"] = score
    candidate["validation_status"] = status
    candidate["quality_flags"] = safe_json_dumps(sorted(set(flags)))

    return candidate


def validate_metric_candidates(candidates_df, metric_registry):
    """
    Applique la validation métier à tous les candidats métriques.
    """

    if candidates_df is None or len(candidates_df) == 0:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)

    registry_lookup = build_metric_registry_lookup(metric_registry)

    validated_rows = []

    candidates_df = ensure_dataframe_schema(
        candidates_df,
        METRIC_CANDIDATE_COLUMNS
    )

    for _, row in candidates_df.iterrows():
        validated_candidate = validate_single_metric_candidate(
            candidate=row.to_dict(),
            registry_lookup=registry_lookup
        )
        validated_rows.append(validated_candidate)

    validated_df = pd.DataFrame(validated_rows)
    validated_df = ensure_dataframe_schema(
        validated_df,
        METRIC_CANDIDATE_COLUMNS
    )

    return validated_df


def build_review_queue(validated_candidates_df):
    """
    Construit la file de revue manuelle.
    """

    if validated_candidates_df is None or len(validated_candidates_df) == 0:
        return pd.DataFrame(columns=REVIEW_QUEUE_COLUMNS)

    review_df = validated_candidates_df[
        validated_candidates_df["validation_status"].isin([
            "REVIEW_REQUIRED",
            "CONFLICT_REVIEW_REQUIRED"
        ])
    ].copy()

    if len(review_df) == 0:
        return pd.DataFrame(columns=REVIEW_QUEUE_COLUMNS)

    review_rows = []

    for _, row in review_df.iterrows():
        flags = parse_quality_flags(row.get("quality_flags"))

        if "VALUE_OUT_OF_EXPECTED_RANGE" in flags:
            priority = "HIGH"
        elif "UNEXPECTED_OR_MISSING_UNIT" in flags or "UNIT_AMBIGUOUS" in flags:
            priority = "HIGH"
        elif "LOW_CONFIDENCE" in flags or "VERY_LOW_CONFIDENCE" in flags:
            priority = "MEDIUM"
        else:
            priority = "MEDIUM"

        review_rows.append({
            "run_id": row.get("run_id"),
            "review_id": make_stable_id(
                row.get("metric_id"),
                "review",
                prefix="review"
            ),
            "metric_id": row.get("metric_id"),
            "document_id": row.get("document_id"),
            "company": row.get("company"),
            "fiscal_year": row.get("fiscal_year"),
            "doc_subtype": row.get("doc_subtype"),
            "metric_name": row.get("metric_name"),
            "metric_category": row.get("metric_category"),
            "raw_value": row.get("raw_value"),
            "raw_unit": row.get("raw_unit"),
            "normalized_value": row.get("normalized_value"),
            "normalized_unit": row.get("normalized_unit"),
            "source_page": row.get("source_page"),
            "source_block_id": row.get("source_block_id"),
            "source_text_excerpt": row.get("source_text_excerpt"),
            "confidence_score": row.get("confidence_score"),
            "review_reason": safe_json_dumps(flags),
            "review_priority": priority,
            "review_status": "TO_REVIEW",
            "created_at": now_utc_iso()
        })

    review_queue_df = pd.DataFrame(review_rows)
    review_queue_df = ensure_dataframe_schema(
        review_queue_df,
        REVIEW_QUEUE_COLUMNS
    )

    return review_queue_df


def summarize_validation_results(validated_candidates_df):
    """
    Résume les résultats de validation.
    """

    print("Résumé validation métier :")

    if validated_candidates_df is None or len(validated_candidates_df) == 0:
        print("Aucun candidat métrique à valider.")
        return

    print(f"Nombre total de candidats : {len(validated_candidates_df)}")

    print("\nStatuts de validation :")
    display(validated_candidates_df["validation_status"].value_counts(dropna=False))

    print("\nMéthodes d'extraction :")
    display(validated_candidates_df["extraction_method"].value_counts(dropna=False))

    print("\nScore de confiance moyen par statut :")
    display(
        validated_candidates_df
        .groupby("validation_status")["confidence_score"]
        .mean()
        .round(4)
    )


# ============================================================
# ORCHESTRATION VALIDATION MÉTIER
# ============================================================

all_metric_candidates_df = combine_metric_candidate_sources(
    table_candidates_df=table_metric_candidates_df,
    text_candidates_df=text_metric_candidates_df,
    ocr_candidates_df=ocr_metric_candidates_df
)

validated_metric_candidates_df = validate_metric_candidates(
    candidates_df=all_metric_candidates_df,
    metric_registry=esg_metric_registry
)

review_queue_df = build_review_queue(
    validated_candidates_df=validated_metric_candidates_df
)

write_dataframe(
    all_metric_candidates_df,
    CANDIDATES_DIR / "all_metric_candidates.parquet"
)

write_dataframe(
    validated_metric_candidates_df,
    VALIDATED_DIR / "validated_metric_candidates.parquet"
)

write_dataframe(
    review_queue_df,
    REVIEW_DIR / "review_queue.parquet"
)

summarize_validation_results(validated_metric_candidates_df)

print(f"File de revue manuelle : {len(review_queue_df)} candidats")
display(review_queue_df.head())

Résumé validation métier :
Nombre total de candidats : 783

Statuts de validation :


/tmp/ipykernel_139103/965948883.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


validation_status
REVIEW_REQUIRED    769
AUTO_VALIDATED      14
Name: count, dtype: int64


Méthodes d'extraction :


extraction_method
text_block                      715
table_registry_year_matching     68
Name: count, dtype: int64


Score de confiance moyen par statut :


validation_status
AUTO_VALIDATED     1.0000
REVIEW_REQUIRED    0.6556
Name: confidence_score, dtype: float64

File de revue manuelle : 769 candidats


,run_id,review_id,metric_id,document_id,company,fiscal_year,doc_subtype,metric_name,metric_category,raw_value,...,normalized_value,normalized_unit,source_page,source_block_id,source_text_excerpt,confidence_score,review_reason,review_priority,review_status,created_at
0,phase2_20260510_130745,review_fd1b18c96434a10e,metric_891c6f6a7f7f0322,doc_650a8b107d86f6ce,Airbus,2023,sustainability_statement_csrd_esrs,scope_1_ghg_emissions,climate,39.9,...,39.9,tCO2e,80,table_391620509c7ffacb,Total emissions (thous t CO2e) (Scope 1 & 2) |...,1.00,"[""UNIT_AMBIGUOUS""]",HIGH,TO_REVIEW,2026-05-10T20:27:47Z
1,phase2_20260510_130745,review_4a97494a2ef47a2c,metric_79ae108ed15094a4,doc_650a8b107d86f6ce,Airbus,2023,sustainability_statement_csrd_esrs,water_withdrawal,water,44412,...,44412.0,m3,91,table_2dc414ea01fea411,"| | Water consumption m3 (thous L) | 56,886 | ...",1.00,"[""FOOTNOTE_LIKE_VALUE"", ""UNIT_AMBIGUOUS""]",HIGH,TO_REVIEW,2026-05-10T20:27:47Z
2,phase2_20260510_130745,review_b181cd21c3f98fa8,metric_3cc043c60a453956,doc_650a8b107d86f6ce,Airbus,2023,sustainability_statement_csrd_esrs,scope_1_ghg_emissions,climate,Internal Diesel B0 (kWh),...,0.0,kWh,93,table_f7fb28b7d7eb68fc,"Internal Diesel B0 (kWh) Scope 1 | 18,571,126 ...",0.85,"[""UNEXPECTED_OR_MISSING_UNIT""]",HIGH,TO_REVIEW,2026-05-10T20:27:47Z
3,phase2_20260510_130745,review_b1d74fa3154b5175,metric_91f6a84871865116,doc_650a8b107d86f6ce,Airbus,2023,sustainability_statement_csrd_esrs,scope_1_ghg_emissions,climate,Internal Diesel B7 (kWh),...,7.0,kWh,93,table_f7fb28b7d7eb68fc,"Internal Diesel B7 (kWh) Scope 1 | 36,607,088 ...",0.85,"[""UNEXPECTED_OR_MISSING_UNIT""]",HIGH,TO_REVIEW,2026-05-10T20:27:47Z
4,phase2_20260510_130745,review_84581ee11f1b1ea9,metric_0f1227f5b8e306ae,doc_650a8b107d86f6ce,Airbus,2023,sustainability_statement_csrd_esrs,scope_1_ghg_emissions,climate,External Diesel B7 (kWh),...,7.0,kWh,93,table_f7fb28b7d7eb68fc,"External Diesel B7 (kWh) Scope 1 | 11,156,759 ...",0.85,"[""UNEXPECTED_OR_MISSING_UNIT""]",HIGH,TO_REVIEW,2026-05-10T20:27:47Z


## 14. Construction du panel entreprise-année-indicateur

Cette cellule transforme les candidats validés en panel analytique final.

Pour chaque triplet entreprise-année-indicateur, le pipeline conserve le meilleur candidat disponible selon :
- le statut de validation ;
- le score de confiance ;
- la méthode d’extraction ;
- la présence d’une preuve source ;
- les flags qualité.

Le panel conserve les informations essentielles :
- entreprise, ticker, ISIN, juridiction ;
- exercice fiscal ;
- métrique ESG ;
- valeur normalisée ;
- unité ;
- score de confiance ;
- statut de validation ;
- document source ;
- page et extrait de preuve.

Cette table constitue la sortie principale de la phase 2. Elle alimentera ensuite le scoring ESG, les comparaisons sectorielles, l’analyse climat et les signaux quantitatifs.

In [29]:
# ============================================================
# 14. CONSTRUCTION DU PANEL ENTREPRISE-ANNÉE-INDICATEUR
# ============================================================

PANEL_STATUS_RANK = {
    "AUTO_VALIDATED": 0,
    "REVIEW_REQUIRED": 1,
    "CONFLICT_REVIEW_REQUIRED": 2,
    "REJECTED": 9,
}

PANEL_METHOD_RANK = {
    "table_registry_year_matching": 0,
    "text_block": 1,
    "ocr_image": 2,
}


def parse_panel_flags(flags):
    """
    Recharge les quality_flags pour le scoring panel.
    """
    return parse_quality_flags(flags)


def panel_candidate_has_value(row):
    """
    Vérifie qu'un candidat possède une valeur exploitable.
    """
    value = row.get("normalized_value")

    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except Exception:
        pass

    return str(value).strip().lower() not in ["", "nan", "none"]


def panel_candidate_has_evidence(row):
    """
    Vérifie la présence d'une preuve source exploitable.
    """
    excerpt = row.get("source_text_excerpt")
    page = row.get("source_page")

    if excerpt is None or pd.isna(excerpt) or len(str(excerpt).strip()) < 10:
        return False

    if page is None or pd.isna(page):
        return False

    return True


def compute_panel_selection_score(row):
    """
    Calcule un score de sélection pour choisir le meilleur candidat.
    """
    try:
        score = float(row.get("confidence_score") or 0.0)
    except Exception:
        score = 0.0

    method = str(row.get("extraction_method", ""))
    flags = parse_panel_flags(row.get("quality_flags"))

    # Bonus preuve source
    if panel_candidate_has_evidence(row):
        score += 0.03

    # Bonus table, car les tableaux sont plus structurés
    if method == "table_registry_year_matching":
        score += 0.05

    # Pénalité OCR
    if method == "ocr_image":
        score -= 0.10

    # Pénalités qualité
    penalty_flags = {
        "UNIT_AMBIGUOUS": 0.08,
        "UNEXPECTED_OR_MISSING_UNIT": 0.08,
        "UNIT_NOT_DETECTED": 0.06,
        "FOOTNOTE_LIKE_VALUE": 0.08,
        "WEAK_PERCENT_CONTEXT": 0.08,
        "VALUE_OUT_OF_EXPECTED_RANGE": 0.10,
        "LOW_CONFIDENCE": 0.08,
        "VERY_LOW_CONFIDENCE": 0.12,
    }

    for flag, penalty in penalty_flags.items():
        if flag in flags:
            score -= penalty

    return round(max(0.0, min(1.0, score)), 4)


def prepare_candidates_for_panel(validated_candidates_df):
    """
    Prépare les candidats validés pour la construction du panel.
    """
    if validated_candidates_df is None or len(validated_candidates_df) == 0:
        return pd.DataFrame(columns=METRIC_CANDIDATE_COLUMNS)

    candidates = ensure_dataframe_schema(
        validated_candidates_df,
        METRIC_CANDIDATE_COLUMNS
    ).copy()

    # On exclut seulement les candidats rejetés ou sans valeur
    candidates = candidates[
        candidates["validation_status"].isin([
            "AUTO_VALIDATED",
            "REVIEW_REQUIRED",
            "CONFLICT_REVIEW_REQUIRED"
        ])
    ].copy()

    candidates = candidates[
        candidates.apply(panel_candidate_has_value, axis=1)
    ].copy()

    if len(candidates) == 0:
        return candidates

    candidates["status_rank"] = (
        candidates["validation_status"]
        .map(PANEL_STATUS_RANK)
        .fillna(9)
    )

    candidates["method_rank"] = (
        candidates["extraction_method"]
        .map(PANEL_METHOD_RANK)
        .fillna(9)
    )

    candidates["panel_selection_score"] = candidates.apply(
        compute_panel_selection_score,
        axis=1
    )

    return candidates


def build_company_year_esg_panel(
    validated_candidates_df,
    run_id=RUN_ID
):
    """
    Construit le panel entreprise-année-indicateur final.
    """
    usable = prepare_candidates_for_panel(validated_candidates_df)

    if usable is None or len(usable) == 0:
        panel = pd.DataFrame(columns=COMPANY_YEAR_PANEL_COLUMNS)
        write_dataframe(panel, FINAL_PANEL_PATH)
        return panel

    sort_columns = [
        "company",
        "fiscal_year",
        "metric_name",
        "status_rank",
        "panel_selection_score",
        "confidence_score",
        "method_rank",
    ]

    usable = usable.sort_values(
        sort_columns,
        ascending=[True, True, True, True, False, False, True]
    )

    best = (
        usable
        .groupby(["company", "fiscal_year", "metric_name"], dropna=False)
        .head(1)
        .copy()
    )

    panel = pd.DataFrame({
        "run_id": run_id,
        "company": best["company"],
        "ticker": best["ticker"],
        "isin": best["isin"],
        "jurisdiction": best["jurisdiction"],
        "fiscal_year": best["fiscal_year"],
        "metric_name": best["metric_name"],
        "metric_category": best["metric_category"],
        "metric_value": best["normalized_value"],
        "metric_unit": best["normalized_unit"],
        "best_confidence_score": best["confidence_score"],
        "validation_status": best["validation_status"],
        "source_document_id": best["document_id"],
        "source_doc_subtype": best["doc_subtype"],
        "source_page": best["source_page"],
        "source_text_excerpt": best["source_text_excerpt"],
        "quality_flags": best["quality_flags"],
        "created_at": now_utc_iso(),
    })

    panel = ensure_dataframe_schema(
        panel,
        COMPANY_YEAR_PANEL_COLUMNS
    )

    panel = panel.sort_values(
        ["company", "fiscal_year", "metric_name"]
    ).reset_index(drop=True)

    write_dataframe(panel, FINAL_PANEL_PATH)

    return panel


def summarize_company_year_panel(panel_df):
    """
    Résume le panel final.
    """
    print("Résumé du panel entreprise-année-indicateur :")

    if panel_df is None or len(panel_df) == 0:
        print("Panel vide.")
        return

    print(f"Nombre de lignes panel : {len(panel_df)}")
    print(f"Nombre d'entreprises : {panel_df['company'].nunique()}")
    print(f"Nombre d'années fiscales : {panel_df['fiscal_year'].nunique()}")
    print(f"Nombre de métriques distinctes : {panel_df['metric_name'].nunique()}")

    print("\nStatuts dans le panel :")
    display(panel_df["validation_status"].value_counts(dropna=False))

    print("\nNombre de métriques par catégorie :")
    display(panel_df["metric_category"].value_counts(dropna=False))

    print("\nCouverture par entreprise :")
    display(
        panel_df
        .groupby("company")["metric_name"]
        .nunique()
        .sort_values(ascending=False)
        .head(20)
    )


company_year_esg_panel_df = build_company_year_esg_panel(
    validated_candidates_df=validated_metric_candidates_df,
    run_id=RUN_ID
)

summarize_company_year_panel(company_year_esg_panel_df)

display(company_year_esg_panel_df.head())

Résumé du panel entreprise-année-indicateur :
Nombre de lignes panel : 259
Nombre d'entreprises : 34
Nombre d'années fiscales : 2
Nombre de métriques distinctes : 11

Statuts dans le panel :


/tmp/ipykernel_139103/965948883.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


validation_status
REVIEW_REQUIRED    248
AUTO_VALIDATED      11
Name: count, dtype: int64


Nombre de métriques par catégorie :


metric_category
climate          87
energy           78
water            27
governance       26
human_capital    25
diversity        10
health_safety     6
Name: count, dtype: int64


Couverture par entreprise :


company
Engie                  11
AXA                    10
Stellantis             10
Societe Generale        9
Dassault Systemes       9
ArcelorMittal           9
Eurofins Scientific     9
TotalEnergies           9
Publicis Groupe         8
Euronext                8
Capgemini               7
LVMH                    7
Legrand                 7
Veolia                  7
Airbus                  7
Air Liquide             7
BNP Paribas             7
Danone                  6
L'Oreal                 6
Bureau Veritas          6
Name: metric_name, dtype: int64

,run_id,company,ticker,isin,jurisdiction,fiscal_year,metric_name,metric_category,metric_value,metric_unit,best_confidence_score,validation_status,source_document_id,source_doc_subtype,source_page,source_text_excerpt,quality_flags,created_at
0,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,board_independence_share,governance,50.0,%,0.7,REVIEW_REQUIRED,doc_44af52056b84b450,annual_report_urd,78,At least 50% independent directors (Target met),"[""FROM_TEXT_BLOCK"", ""LOW_CONFIDENCE""]",2026-05-10T20:46:00Z
1,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,employee_turnover_rate,human_capital,14.5,%,0.7,REVIEW_REQUIRED,doc_44af52056b84b450,annual_report_urd,162,Turnover rate of salaried workforce 14.5 % -0....,"[""FROM_TEXT_BLOCK"", ""LOW_CONFIDENCE""]",2026-05-10T20:46:00Z
2,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,energy_consumption,energy,6.0,MWh,1.0,AUTO_VALIDATED,doc_4caf34e4800c704d,sustainability_statement_csrd_esrs,91,(6) Total fossil energy consumption (MWh) | |,[],2026-05-10T20:46:00Z
3,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,renewable_energy_share,energy,3.0,%,0.7,REVIEW_REQUIRED,doc_44af52056b84b450,annual_report_urd,7,"(1) Compounded annual growth rate, 2020-2023 ....","[""FROM_TEXT_BLOCK"", ""LOW_CONFIDENCE""]",2026-05-10T20:46:00Z
4,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,scope_1_ghg_emissions,climate,2000000.0,tCO2e,0.7,REVIEW_REQUIRED,doc_409d340cd4f6b397,climate_report_tcfd_transition_plan,42,(MtCO2e) FY2020 FY2021 Scope 1 0.4 0.2 Scope 2...,"[""FROM_TEXT_BLOCK"", ""LOW_CONFIDENCE""]",2026-05-10T20:46:00Z


## 15. Base finale des métriques retenues

Cette cellule construit la base finale exploitable à partir du panel entreprise-année-indicateur.

Contrairement au panel complet, qui conserve aussi les métriques à revoir, cette base ne garde que les métriques automatiquement validées. Elle correspond donc au sous-ensemble le plus fiable du pipeline, directement mobilisable pour les étapes suivantes : scoring ESG, analyse descriptive, comparaison sectorielle ou premiers signaux quantitatifs.

Les candidats en revue manuelle ne sont pas supprimés du projet : ils restent disponibles dans `review_queue`. Ici, on produit simplement une base finale prudente, fondée uniquement sur les métriques retenues.

In [30]:
# ============================================================
# 15. BASE FINALE DES MÉTRIQUES RETENUES
# ============================================================

FINAL_RETAINED_PANEL_PATH = PANEL_DIR / "esg_panel_retained_metrics.csv"


def build_retained_metrics_panel(
    company_year_panel_df,
    retained_statuses=None,
    output_path=FINAL_RETAINED_PANEL_PATH
):
    """
    Construit la base finale des métriques retenues.

    Par défaut, seules les métriques AUTO_VALIDATED sont conservées.
    """

    if retained_statuses is None:
        retained_statuses = ["AUTO_VALIDATED"]

    if company_year_panel_df is None or len(company_year_panel_df) == 0:
        retained_panel = pd.DataFrame(columns=COMPANY_YEAR_PANEL_COLUMNS)
        write_dataframe(retained_panel, output_path)
        return retained_panel

    retained_panel = company_year_panel_df[
        company_year_panel_df["validation_status"].isin(retained_statuses)
    ].copy()

    retained_panel = retained_panel[
        retained_panel["metric_value"].notna()
    ].copy()

    retained_panel = ensure_dataframe_schema(
        retained_panel,
        COMPANY_YEAR_PANEL_COLUMNS
    )

    retained_panel = retained_panel.sort_values(
        ["company", "fiscal_year", "metric_name"]
    ).reset_index(drop=True)

    write_dataframe(retained_panel, output_path)

    return retained_panel


def summarize_retained_metrics_panel(retained_panel_df):
    """
    Résume la base finale retenue.
    """

    print("Résumé de la base finale retenue :")

    if retained_panel_df is None or len(retained_panel_df) == 0:
        print("Aucune métrique automatiquement validée retenue.")
        return

    print(f"Nombre de lignes retenues : {len(retained_panel_df)}")
    print(f"Nombre d'entreprises couvertes : {retained_panel_df['company'].nunique()}")
    print(f"Nombre d'années fiscales : {retained_panel_df['fiscal_year'].nunique()}")
    print(f"Nombre de métriques distinctes : {retained_panel_df['metric_name'].nunique()}")

    print("\nMétriques retenues :")
    display(retained_panel_df["metric_name"].value_counts())

    print("\nCouverture par entreprise :")
    display(
        retained_panel_df
        .groupby("company")["metric_name"]
        .nunique()
        .sort_values(ascending=False)
    )


retained_esg_panel_df = build_retained_metrics_panel(
    company_year_panel_df=company_year_esg_panel_df,
    retained_statuses=["AUTO_VALIDATED"],
    output_path=FINAL_RETAINED_PANEL_PATH
)

summarize_retained_metrics_panel(retained_esg_panel_df)

display(retained_esg_panel_df.head())

Résumé de la base finale retenue :
Nombre de lignes retenues : 11
Nombre d'entreprises couvertes : 6
Nombre d'années fiscales : 2
Nombre de métriques distinctes : 6

Métriques retenues :


metric_name
energy_consumption        4
scope_1_ghg_emissions     2
water_withdrawal          2
scope_2_ghg_emissions     1
renewable_energy_share    1
total_ghg_emissions       1
Name: count, dtype: int64


Couverture par entreprise :


company
AXA           3
L'Oreal       2
Thales        2
Airbus        1
Danone        1
Stellantis    1
Name: metric_name, dtype: int64

,run_id,company,ticker,isin,jurisdiction,fiscal_year,metric_name,metric_category,metric_value,metric_unit,best_confidence_score,validation_status,source_document_id,source_doc_subtype,source_page,source_text_excerpt,quality_flags,created_at
0,phase2_20260510_130745,AXA,CS,FR0000120628,France,2023,energy_consumption,energy,6.0,MWh,1.0,AUTO_VALIDATED,doc_4caf34e4800c704d,sustainability_statement_csrd_esrs,91,(6) Total fossil energy consumption (MWh) | |,[],2026-05-10T20:46:00Z
1,phase2_20260510_130745,AXA,CS,FR0000120628,France,2024,scope_1_ghg_emissions,climate,24.0,tCO2e,1.0,AUTO_VALIDATED,doc_c272d1a4039c4cb5,assurance_report,23,| Scope 1\n(tCO2e) | Scope 2\n(market-based)\n...,[],2026-05-10T20:46:00Z
2,phase2_20260510_130745,AXA,CS,FR0000120628,France,2024,scope_2_ghg_emissions,climate,24.0,tCO2e,1.0,AUTO_VALIDATED,doc_c272d1a4039c4cb5,assurance_report,23,| Scope 1\n(tCO2e) | Scope 2\n(market-based)\n...,[],2026-05-10T20:46:00Z
3,phase2_20260510_130745,Airbus,AIR,NL0000235190,Netherlands,2023,water_withdrawal,water,3.0,m3,1.0,AUTO_VALIDATED,doc_650a8b107d86f6ce,sustainability_statement_csrd_esrs,103,Year | Water Consumption (m3),[],2026-05-10T20:46:00Z
4,phase2_20260510_130745,Danone,BN,FR0000120644,France,2024,renewable_energy_share,energy,85.7,%,1.0,AUTO_VALIDATED,doc_5a03b815bc592d0e,assurance_report,350,Total percentage of renewable electricity | 71...,[],2026-05-10T20:46:00Z
